# ATLAS-FN 04 — Extended Evidence + TIES TRUE FINAL v2: optional 03, fresh FP/UCL/TIES run

This final notebook restores the completed Notebook 01 data-prep workspace and Notebook 02 checkpoint package from Google Drive, then performs a fresh FP/UCL external-validation and TIES run. Notebook 03 Master Evidence outputs are treated as **optional contextual evidence**: they are used when an archive is available, but their absence must not stop FP/UCL external validation or TIES because those experiments depend on 01 + 02, not on the 03 visual-audit package.


In [1]:
# ============================================================
# 0. Colab restore controls: master evidence + optional data/checkpoint archives
#    v3 BULLETPROOF restore: master-first, retryable Drive staging, resumable copy
# ============================================================
from __future__ import annotations

import os, sys, json, time, shutil, tarfile, zipfile, hashlib, platform, subprocess, warnings, contextlib, re, math, random
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple
from datetime import datetime, timezone
from collections import defaultdict

# File suffix constants used across restore/dataset-adapter cells.
IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
ARCHIVE_SUFFIXES = ('.zip', '.tar', '.tar.gz', '.tgz')

warnings.filterwarnings("ignore", category=UserWarning)

try:
    import numpy as np
    import pandas as pd
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas"])
    import numpy as np
    import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tqdm"])
    from tqdm.auto import tqdm

# -----------------------------
# User controls
# -----------------------------
WORKSPACE = Path("/content/atlas_fn_workspace")
OUTPUT_ROOT_BOOT = WORKSPACE / "outputs" / "extended_evidence_colab_bootstrap"
OUTPUT_ROOT_BOOT.mkdir(parents=True, exist_ok=True)

# Optional contextual input from the final Master Evidence notebook.
# Notebook 04 does not require this archive for FP/UCL external validation or TIES.
# If present, it is restored and used for cross-notebook traceability; if absent, the notebook continues.
MASTER_EVIDENCE_ARCHIVE = None  # optional; auto-select latest ATLAS_FN_master_evidence_inference_*.tar.gz from GDrive

# Optional fallback archives from the previous two notebooks.
# They may be large. By default, restore them only if the master-evidence package does not contain required roots.
DATAPREP_ARCHIVE = Path("/content/drive/MyDrive/ATLAS_FN_exports/ATLAS_FN_dataprep_prepared_plus_audit_20260707_003119.tar.gz")
CHECKPOINT_TRAINING_ARCHIVE = Path("/content/drive/MyDrive/ATLAS_FN_exports/ATLAS_FN_training_checkpoints_DockerRootReplica_20260707_133539.tar.gz")

# Notebook 04 input locks from the completed 01/02/03 sequence.
# Master Evidence has a timestamped filename, so auto-select the latest matching archive.
MASTER_EVIDENCE_PATTERNS = [
    "ATLAS_FN_master_evidence_inference_*.tar.gz",
    "ATLAS_FN_master_evidence_inference_*.tar",
    "ATLAS_FN_MasterEvidence*.tar.gz",
]
EXPECTED_ARCHIVE_SHA256 = {
    "data_prep": "95d7884ad787682ce70eede35650b010e5e0f107816b9299dd7cee0eecef5902",
    "checkpoint_training": "f1f499315918e6ace904fd24a217ecf160b952e06241465395b1e4ef7f3e9538",
    # Master Evidence archive is timestamped from Notebook 03 and may be regenerated;
    # leave blank unless you want to lock a specific 03 export.
    "master_evidence": "",
}
MIN_DATA_PREP_ARCHIVE_GB = 6.0
REJECT_SMALL_DATAPREP_ARCHIVES = True
FINAL04_FRESH_RUN = True
NOTEBOOK03_MASTER_EVIDENCE_REQUIRED_FOR_04 = False
DOMAIN_SPECIALIST_CHECKPOINTS_EXPECTED_AFTER_SECTION5 = True

GDRIVE_EXPORT_DIR = Path("/content/drive/MyDrive/ATLAS_FN_exports")
MOUNT_GDRIVE = True
STAGE_ARCHIVES_TO_LOCAL = True
LOCAL_ARCHIVE_CACHE_DIR = Path("/content/atlas_fn_archive_cache")
RESTORE_STAGE_DIR = Path("/content/atlas_fn_restore_stage_extended")
ARCHIVE_COPY_CHUNK_MB = 32
SHOW_PROGRESS = True

# Bulletproof Drive staging controls.
# Google Drive FUSE can intermittently disconnect with OSError 107.
# The staged .part copy is resumable across retries and cells within the same runtime.
ARCHIVE_COPY_MAX_RETRIES = 8
ARCHIVE_COPY_RETRY_SLEEP_SEC = 5
ARCHIVE_COPY_RESUME_PARTIAL = True
ARCHIVE_COPY_FORCE_REMOUNT_ON_TRANSPORT_ERROR = True
ARCHIVE_COPY_VERIFY_SIZE = True
ARCHIVE_COPY_VERIFY_SHA256 = True
REUSE_STAGED_ARCHIVE_IF_SIZE_MATCHES = True

# Avoid optional archive transfers unless they are genuinely required.
RESTORE_MASTER_FIRST = True  # optional/contextual; absence must not block Notebook 04
RESTORE_DATAPREP_IF_NEEDED = True
RESTORE_CHECKPOINTS_IF_NEEDED = True
OPTIONAL_ARCHIVE_FAILURE_IS_NONFATAL = False
SKIP_OPTIONAL_RESTORE_IF_REQUIREMENTS_ALREADY_MET = False

# If True, missing optional archives will be found by searching only the export folder, not all of Drive.
AUTO_SEARCH_OPTIONAL_ARCHIVES = True
OPTIONAL_CHECKPOINT_PATTERNS = [
    # Export name produced by ATLAS_FN_CheckpointTraining_DockerRootReplica_Colab_v5_RESTORE_REPAIR.ipynb
    "ATLAS_FN_training_checkpoints_DockerRootReplica*.tar.gz",
    "ATLAS_FN_training_checkpoints*.tar.gz",
    "*training*checkpoint*.tar.gz",
    "*training*checkpoints*.tar.gz",
    # Backward-compatible variants
    "ATLAS_FN_checkpoint*.tar.gz", "ATLAS_FN_Checkpoint*.tar.gz", "*checkpoint*training*.tar.gz",
    "ATLAS_FN_training_checkpoints_DockerRootReplica*.zip",
    "ATLAS_FN_training_checkpoints*.zip",
    "*training*checkpoint*.zip", "*training*checkpoints*.zip",
    "ATLAS_FN_checkpoint*.zip", "ATLAS_FN_Checkpoint*.zip", "*checkpoint*training*.zip",
]
OPTIONAL_DATAPREP_PATTERNS = [
    "ATLAS_FN_dataprep_prepared_plus_audit*.tar.gz", "ATLAS_FN_dataprep*.tar.gz", "ATLAS_FN_dataprep*.zip",
]

# Extended-evidence behaviour controls.
CONTINUE_ON_AUDIT_FAILURE = True
RUN_EXTERNAL_FIND_INFERENCE = True
RUN_EXTERNAL_FIND_FINETUNING = True
RUN_TIES = True
REQUIRE_COMPLETE_TIES_FOR_EXPORT = False  # audit missing TIES rather than hard-stopping.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# -----------------------------
# Logging and utilities
# -----------------------------
def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def log(msg: str):
    print(f"[atlas-extended] {msg}", flush=True)

def json_safe(v):
    if isinstance(v, Path):
        return str(v)
    if isinstance(v, (np.integer, np.floating)):
        return v.item()
    if isinstance(v, dict):
        return {str(k): json_safe(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [json_safe(x) for x in v]
    return v

BOOT_LOG = OUTPUT_ROOT_BOOT / "extended_restore_progress.jsonl"
def record_boot(stage: str, message: str, **kwargs):
    row = {"ts": utc_now(), "stage": stage, "message": message, **{k: json_safe(v) for k, v in kwargs.items()}}
    try:
        BOOT_LOG.parent.mkdir(parents=True, exist_ok=True)
        with BOOT_LOG.open("a", encoding="utf-8") as f:
            f.write(json.dumps(row) + "\n")
    except Exception:
        pass
    extra = " | " + ", ".join(f"{k}={v}" for k, v in kwargs.items()) if kwargs else ""
    log(f"{stage}: {message}{extra}")

def file_size_gb(path: Path) -> float:
    return path.stat().st_size / (1024**3) if path and path.exists() else float("nan")

def sha256_file_progress(path: Path, chunk_size: int = 32 * 1024 * 1024) -> str:
    total = path.stat().st_size
    h = hashlib.sha256()
    with path.open("rb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=f"SHA256 {path.name}", disable=not SHOW_PROGRESS) as pbar:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
            pbar.update(len(b))
    return h.hexdigest()

def _is_transport_disconnect_error(e: BaseException) -> bool:
    s = repr(e).lower()
    return (
        "transport endpoint is not connected" in s
        or "errno 107" in s
        or "input/output error" in s
        or "stale file handle" in s
        or "bad file descriptor" in s
    )

def _safe_stat_size(path: Path) -> Optional[int]:
    try:
        return path.stat().st_size if path.exists() else None
    except Exception:
        return None

def remount_gdrive_if_possible(force: bool = True) -> bool:
    """Best-effort Google Drive remount after intermittent FUSE disconnects."""
    if not MOUNT_GDRIVE:
        return False
    try:
        from google.colab import drive
        record_boot("gdrive", "remount Google Drive after transfer error", force=force)
        try:
            drive.flush_and_unmount()
            time.sleep(2)
        except Exception as e:
            record_boot("gdrive", "flush/unmount warning", error=repr(e))
        drive.mount("/content/drive", force_remount=force)
        ok = Path("/content/drive/MyDrive").exists()
        record_boot("gdrive", "remount status", ok=ok)
        return ok
    except Exception as e:
        record_boot("gdrive", "remount failed", error=repr(e))
        return False

def stage_archive_to_local(src: Path) -> Path:
    """Stage Drive/local archive into /content with resumable copy and remount-aware retries.

    This protects long Colab runs from transient Google Drive FUSE errors such as:
    OSError: [Errno 107] Transport endpoint is not connected.
    """
    src = Path(src)
    if not src.exists():
        raise FileNotFoundError(src)
    if not STAGE_ARCHIVES_TO_LOCAL:
        return src

    LOCAL_ARCHIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    dst = LOCAL_ARCHIVE_CACHE_DIR / src.name
    part = LOCAL_ARCHIVE_CACHE_DIR / f"{src.name}.part"

    total = src.stat().st_size
    chunk = max(1, int(ARCHIVE_COPY_CHUNK_MB)) * 1024 * 1024

    if REUSE_STAGED_ARCHIVE_IF_SIZE_MATCHES and dst.exists() and dst.stat().st_size == total:
        record_boot("restore", "reuse staged archive", path=dst, size_gb=round(file_size_gb(dst), 4))
        return dst

    # If a previously completed copy is wrong-sized, keep it as diagnostic but do not reuse.
    if dst.exists() and dst.stat().st_size != total:
        bad = dst.with_suffix(dst.suffix + f".bad_size_{int(time.time())}")
        try:
            dst.rename(bad)
            record_boot("restore", "renamed wrong-sized staged archive", old=dst, new=bad, observed_bytes=bad.stat().st_size, expected_bytes=total)
        except Exception as e:
            record_boot("restore", "could not rename wrong-sized staged archive", path=dst, error=repr(e))
            dst.unlink(missing_ok=True)

    record_boot("restore", "stage archive to local /content", src=src, dst=dst, size_gb=round(file_size_gb(src), 4), chunk_mb=ARCHIVE_COPY_CHUNK_MB, max_retries=ARCHIVE_COPY_MAX_RETRIES)

    attempt = 0
    t0 = time.time()
    last_error = None

    while attempt < ARCHIVE_COPY_MAX_RETRIES:
        attempt += 1
        try:
            if not src.exists():
                raise FileNotFoundError(f"Source archive disappeared: {src}")

            current = part.stat().st_size if (ARCHIVE_COPY_RESUME_PARTIAL and part.exists()) else 0
            if current > total:
                record_boot("restore", "partial archive larger than source; restarting", partial_bytes=current, total_bytes=total)
                part.unlink(missing_ok=True)
                current = 0

            mode = "ab" if current else "wb"
            record_boot("restore", "copy attempt", attempt=attempt, resume_bytes=current, remaining_bytes=max(0, total-current))

            with src.open("rb") as fsrc, part.open(mode) as fdst, tqdm(total=total, initial=current, unit="B", unit_scale=True, desc=f"stage {src.name}", disable=not SHOW_PROGRESS) as pbar:
                if current:
                    fsrc.seek(current)
                while current < total:
                    b = fsrc.read(chunk)
                    if not b:
                        break
                    fdst.write(b)
                    current += len(b)
                    pbar.update(len(b))
                fdst.flush()
                os.fsync(fdst.fileno())

            observed = part.stat().st_size
            if observed != total:
                raise IOError(f"Incomplete staged archive after attempt {attempt}: observed={observed}, expected={total}")

            if dst.exists():
                dst.unlink()
            part.rename(dst)
            elapsed = time.time() - t0
            record_boot("restore", "archive staged", elapsed_sec=round(elapsed, 2), mbps=round((total/(1024**2))/max(elapsed, 1e-9), 2), dst=dst)
            return dst

        except Exception as e:
            last_error = e
            record_boot("restore", "archive staging attempt failed", attempt=attempt, error=repr(e), partial_bytes=_safe_stat_size(part), total_bytes=total)
            if _is_transport_disconnect_error(e) and ARCHIVE_COPY_FORCE_REMOUNT_ON_TRANSPORT_ERROR:
                remount_gdrive_if_possible(force=True)
            time.sleep(ARCHIVE_COPY_RETRY_SLEEP_SEC * min(attempt, 4))

    raise RuntimeError(f"Failed to stage archive after {ARCHIVE_COPY_MAX_RETRIES} attempts: {src}\nLast error: {repr(last_error)}")

def safe_extract_tar(archive: Path, target_dir: Path) -> Path:
    target_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, "r:*") as tar:
        members = tar.getmembers()
        total_size = sum(max(0, getattr(m, "size", 0)) for m in members)
        record_boot("restore", "extract tar archive", archive=archive.name, members=len(members), declared_gb=round(total_size/(1024**3), 4))
        with tqdm(total=total_size if total_size else None, unit="B", unit_scale=True, desc=f"extract {archive.name}", disable=not SHOW_PROGRESS) as pbar:
            for m in members:
                # tarfile path safety
                dest = (target_dir / m.name).resolve()
                if not str(dest).startswith(str(target_dir.resolve())):
                    record_boot("restore", "skip unsafe tar member", member=m.name)
                    continue
                # Python <3.14 supports this signature. Python 3.14 may require explicit filter.
                try:
                    tar.extract(m, target_dir, filter="data")
                except TypeError:
                    tar.extract(m, target_dir)
                pbar.update(max(0, getattr(m, "size", 0)))
    roots = [p for p in target_dir.iterdir()]
    if len(roots) == 1 and roots[0].is_dir():
        return roots[0]
    return target_dir

def safe_extract_zip(archive: Path, target_dir: Path) -> Path:
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive, "r") as z:
        infos = z.infolist()
        total_size = sum(max(0, i.file_size) for i in infos)
        record_boot("restore", "extract zip archive", archive=archive.name, members=len(infos), declared_gb=round(total_size/(1024**3), 4))
        with tqdm(total=total_size if total_size else None, unit="B", unit_scale=True, desc=f"extract {archive.name}", disable=not SHOW_PROGRESS) as pbar:
            for info in infos:
                dest = (target_dir / info.filename).resolve()
                if not str(dest).startswith(str(target_dir.resolve())):
                    record_boot("restore", "skip unsafe zip member", member=info.filename)
                    continue
                z.extract(info, target_dir)
                pbar.update(max(0, info.file_size))
    roots = [p for p in target_dir.iterdir()]
    if len(roots) == 1 and roots[0].is_dir():
        return roots[0]
    return target_dir

def merge_tree(src_root: Path, workspace: Path = WORKSPACE):
    """Merge a restored artifact into /content/atlas_fn_workspace while preserving nested trees."""
    workspace.mkdir(parents=True, exist_ok=True)
    top_items = list(src_root.iterdir()) if src_root.exists() and src_root.is_dir() else []
    copied = 0
    # Many exported archives have roots like data/, outputs/, Training_Runs/, dataset_structures/.
    # Some full-workspace exports contain atlas_fn_workspace/ as a child.
    if (src_root / "atlas_fn_workspace").exists():
        src_root = src_root / "atlas_fn_workspace"
        top_items = list(src_root.iterdir())
    for item in top_items:
        dst = workspace / item.name
        if item.is_dir():
            for p in item.rglob("*"):
                rel = p.relative_to(item)
                out = dst / rel
                if p.is_dir():
                    out.mkdir(parents=True, exist_ok=True)
                elif p.is_file():
                    out.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(p, out)
                    copied += 1
        elif item.is_file():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, dst)
            copied += 1
    record_boot("restore", "merged restored archive root into workspace", src_root=src_root, workspace=workspace, files=copied)
    return copied

def restore_archive(src: Optional[Path], label: str, required: bool = False):
    if src is None:
        record_boot("restore", f"{label} archive not configured")
        if required:
            raise FileNotFoundError(f"Required archive not configured: {label}")
        return None
    src = Path(src)
    if not src.exists():
        record_boot("restore", f"{label} archive missing", path=src)
        if required:
            raise FileNotFoundError(src)
        return None
    try:
        staged = stage_archive_to_local(src)
        # Guard against accidentally restoring older partial Notebook 01 data-prep packages.
        if label == "data_prep" and globals().get("REJECT_SMALL_DATAPREP_ARCHIVES", True):
            min_bytes = float(globals().get("MIN_DATA_PREP_ARCHIVE_GB", 0)) * (1024**3)
            if staged.stat().st_size < min_bytes:
                raise RuntimeError(f"Rejected data-prep archive because it is smaller than the expected complete workspace export: {staged} ({staged.stat().st_size/(1024**3):.3f} GB)")
        digest = sha256_file_progress(staged) if ARCHIVE_COPY_VERIFY_SHA256 else None
        if digest:
            record_boot("restore", f"{label} archive sha256", sha256=digest)
            expected_digest = str(globals().get("EXPECTED_ARCHIVE_SHA256", {}).get(label, "")).strip().lower()
            if expected_digest and digest.lower() != expected_digest:
                raise RuntimeError(f"SHA256 mismatch for {label}: observed={digest} expected={expected_digest}")
        target = RESTORE_STAGE_DIR / label
        if target.exists():
            shutil.rmtree(target)
        if staged.suffix.lower() == ".zip":
            root = safe_extract_zip(staged, target)
        else:
            root = safe_extract_tar(staged, target)
        merge_tree(root, WORKSPACE)
        return {"label": label, "source": str(src), "staged": str(staged), "sha256": digest, "extracted_root": str(root)}
    except Exception as e:
        record_boot("restore", f"{label} archive restore failed", error=repr(e), required=required)
        if required or not OPTIONAL_ARCHIVE_FAILURE_IS_NONFATAL:
            raise
        return None

def search_latest_archive(patterns: Sequence[str]) -> Optional[Path]:
    if not AUTO_SEARCH_OPTIONAL_ARCHIVES or not GDRIVE_EXPORT_DIR.exists():
        return None
    hits = []
    for pat in patterns:
        hits.extend(GDRIVE_EXPORT_DIR.glob(pat))
    hits = [p for p in hits if p.is_file()]
    if not hits:
        return None
    hits = sorted(hits, key=lambda p: p.stat().st_mtime, reverse=True)
    return hits[0]

def _count_images_in_dataset_root(root: Path) -> int:
    root = Path(root)
    if not root.exists():
        return 0
    n = 0
    for split in ("train", "val", "test"):
        for folder in [root / "images" / split, root / split / "images"]:
            if folder.exists():
                n += sum(1 for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)
    return n

def workspace_has_prepared_data() -> bool:
    candidates = [
        WORKSPACE / "dataset_structures",
        WORKSPACE / "data" / "prepared" / "dataset_structures",
    ]
    return any(_count_images_in_dataset_root(p) >= 3000 for p in candidates)

def workspace_has_checkpoints() -> bool:
    """Return True only when the source/base Find checkpoint is present.

    Earlier notebook versions returned True if *any* best.pt existed. That could
    skip the checkpoint-training package even when the base BrainLocator checkpoint
    was absent, causing FP/UCL fine-tuning and TIES to fail later.
    """
    required_base_patterns = [
        "Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt",
        "Training_Runs/BrainLocator_YOLO26n_MuSGD/best.pt",
        "checkpoints/find/best.pt",
        "checkpoints/find.pt",
        "outputs/**/checkpoints/find.pt",
        "outputs/**/checkpoints/find/best.pt",
    ]
    for pat in required_base_patterns:
        hits = [p for p in WORKSPACE.glob(pat) if p.is_file() and p.stat().st_size > 0]
        if hits:
            return True
    # Conservative recursive fallback: BrainLocator/source Find only, not FP/UCL specialists.
    for p in WORKSPACE.rglob("*.pt"):
        sp = str(p).lower()
        if not ("brainlocator" in sp or "yolo26n_musgd" in sp or "/checkpoints/find/" in sp or sp.endswith("/checkpoints/find.pt")):
            continue
        if any(x in sp for x in ["/fp/", "_fp", "/ucl/", "_ucl", "all_external", "ties", "nano", "measure", "confirm", "s1_", "s3_"]):
            continue
        if p.is_file() and p.stat().st_size > 0:
            return True
    return False

def workspace_has_master_evidence() -> bool:
    patterns = [
        "outputs/master_evidence/**/source_fcm_frame_ledger.csv",
        "outputs/master_evidence/**/manuscript_result_register.csv",
        "outputs/master_evidence_inference/**/source_fcm_frame_ledger.csv",
        "outputs/**/source_fcm_frame_ledger.csv",
    ]
    return any(list(WORKSPACE.glob(pat)) for pat in patterns)

# -----------------------------
# Mount and restore archives
# -----------------------------
if MOUNT_GDRIVE:
    try:
        from google.colab import drive
        record_boot("gdrive", "mount Google Drive")
        drive.mount("/content/drive", force_remount=False)
        record_boot("gdrive", "mount status", ok=Path("/content/drive/MyDrive").exists())
    except Exception as e:
        record_boot("gdrive", "mount failed or not in Colab", error=repr(e))

if MASTER_EVIDENCE_ARCHIVE is None or (isinstance(MASTER_EVIDENCE_ARCHIVE, Path) and not MASTER_EVIDENCE_ARCHIVE.exists()):
    MASTER_EVIDENCE_ARCHIVE = search_latest_archive(MASTER_EVIDENCE_PATTERNS)
    record_boot("restore", "auto-selected latest Master Evidence archive", archive=MASTER_EVIDENCE_ARCHIVE)
if DATAPREP_ARCHIVE is None or (isinstance(DATAPREP_ARCHIVE, Path) and not DATAPREP_ARCHIVE.exists()):
    DATAPREP_ARCHIVE = search_latest_archive(OPTIONAL_DATAPREP_PATTERNS)
    record_boot("restore", "auto-selected fallback data-prep archive", archive=DATAPREP_ARCHIVE)
if CHECKPOINT_TRAINING_ARCHIVE is None or (isinstance(CHECKPOINT_TRAINING_ARCHIVE, Path) and not CHECKPOINT_TRAINING_ARCHIVE.exists()):
    CHECKPOINT_TRAINING_ARCHIVE = search_latest_archive(OPTIONAL_CHECKPOINT_PATTERNS)
    record_boot("restore", "auto-selected fallback checkpoint-training archive", archive=CHECKPOINT_TRAINING_ARCHIVE)

RESTORE_MANIFEST: List[Dict[str, Any]] = []

# Restore the latest master evidence package first. This is the most important dependency.
res = restore_archive(MASTER_EVIDENCE_ARCHIVE, "master_evidence", required=True)
if res:
    RESTORE_MANIFEST.append(res)

# Restore optional archives only if the master package does not already provide the needed contents.
if RESTORE_DATAPREP_IF_NEEDED and (not SKIP_OPTIONAL_RESTORE_IF_REQUIREMENTS_ALREADY_MET or not workspace_has_prepared_data()):
    res = restore_archive(DATAPREP_ARCHIVE, "data_prep", required=False)
    if res:
        RESTORE_MANIFEST.append(res)
else:
    record_boot("restore", "skip data_prep archive; prepared data already present or optional restore disabled")

if RESTORE_CHECKPOINTS_IF_NEEDED and (not SKIP_OPTIONAL_RESTORE_IF_REQUIREMENTS_ALREADY_MET or not workspace_has_checkpoints()):
    res = restore_archive(CHECKPOINT_TRAINING_ARCHIVE, "checkpoint_training", required=False)
    if res:
        RESTORE_MANIFEST.append(res)
else:
    record_boot("restore", "skip checkpoint archive; checkpoints already present or optional restore disabled")

(WORKSPACE / "outputs" / "extended_evidence_colab_bootstrap" / "restore_manifest.json").parent.mkdir(parents=True, exist_ok=True)
(WORKSPACE / "outputs" / "extended_evidence_colab_bootstrap" / "restore_manifest.json").write_text(json.dumps(RESTORE_MANIFEST, indent=2))
record_boot("restore", "restore phase complete", restored=[r["label"] for r in RESTORE_MANIFEST], workspace=WORKSPACE, has_data=workspace_has_prepared_data(), has_checkpoints=workspace_has_checkpoints(), has_master=workspace_has_master_evidence())

# Notebook 04 restore contract: document which completed 01/02/03 artefacts were used.
restore_contract = {
    "notebook01_data_prep_archive": str(DATAPREP_ARCHIVE) if DATAPREP_ARCHIVE else None,
    "notebook02_checkpoint_archive": str(CHECKPOINT_TRAINING_ARCHIVE) if CHECKPOINT_TRAINING_ARCHIVE else None,
    "notebook03_master_evidence_archive": str(MASTER_EVIDENCE_ARCHIVE) if MASTER_EVIDENCE_ARCHIVE else None,
    "fresh_run": bool(globals().get("FINAL04_FRESH_RUN", True)),
    "has_data": workspace_has_prepared_data(),
    "has_checkpoints": workspace_has_checkpoints(),
    "has_master": workspace_has_master_evidence(),
    "restore_manifest": RESTORE_MANIFEST,
}
(WORKSPACE / "outputs" / "extended_evidence_colab_bootstrap" / "restore_contract_from_01_02_03.json").write_text(json.dumps(json_safe(restore_contract), indent=2))

# Set environment hints used by later cells.
os.environ["ATLAS_FN_WORKSPACE"] = str(WORKSPACE)
os.environ["ATLAS_EXTENDED_OUTPUT_ROOT"] = str(WORKSPACE / "outputs" / "extended_evidence_ties_colab")
os.environ["ATLAS_DOCKERROOT"] = str(WORKSPACE)


[atlas-extended] gdrive: mount Google Drive
Mounted at /content/drive
[atlas-extended] gdrive: mount status | ok=True
[atlas-extended] restore: auto-selected latest Master Evidence archive | archive=/content/drive/MyDrive/ATLAS_FN_exports/ATLAS_FN_master_evidence_inference_20260708T115550Z.tar.gz
[atlas-extended] restore: stage archive to local /content | src=/content/drive/MyDrive/ATLAS_FN_exports/ATLAS_FN_master_evidence_inference_20260708T115550Z.tar.gz, dst=/content/atlas_fn_archive_cache/ATLAS_FN_master_evidence_inference_20260708T115550Z.tar.gz, size_gb=0.0128, chunk_mb=32, max_retries=8
[atlas-extended] restore: copy attempt | attempt=1, resume_bytes=0, remaining_bytes=13772750


stage ATLAS_FN_master_evidence_inference_20260708T115550Z.tar.gz:   0%|          | 0.00/13.8M [00:00<?, ?B/s]

[atlas-extended] restore: archive staged | elapsed_sec=0.94, mbps=13.99, dst=/content/atlas_fn_archive_cache/ATLAS_FN_master_evidence_inference_20260708T115550Z.tar.gz


SHA256 ATLAS_FN_master_evidence_inference_20260708T115550Z.tar.gz:   0%|          | 0.00/13.8M [00:00<?, ?B/s]

[atlas-extended] restore: master_evidence archive sha256 | sha256=e8259f8c8054c3ae9ddfc2410f3788693ebb751c3b13a7514c3c44eb8ed3b576
[atlas-extended] restore: extract tar archive | archive=ATLAS_FN_master_evidence_inference_20260708T115550Z.tar.gz, members=55, declared_gb=0.014


...

[atlas-extended] restore: archive staged | elapsed_sec=11.85, mbps=3.67, dst=/content/atlas_fn_archive_cache/ATLAS_FN_training_checkpoints_DockerRootReplica_20260707_133539.tar.gz


SHA256 ATLAS_FN_training_checkpoints_DockerRootReplica_20260707_133539.tar.gz:   0%|          | 0.00/45.6M [00…

[atlas-extended] restore: checkpoint_training archive sha256 | sha256=f1f499315918e6ace904fd24a217ecf160b952e06241465395b1e4ef7f3e9538
[atlas-extended] restore: extract tar archive | archive=ATLAS_FN_training_checkpoints_DockerRootReplica_20260707_133539.tar.gz, members=58, declared_gb=0.0542


extract ATLAS_FN_training_checkpoints_DockerRootReplica_20260707_133539.tar.gz:   0%|          | 0.00/58.2M [0…

[atlas-extended] restore: merged restored archive root into workspace | src_root=/content/atlas_fn_restore_stage_extended/checkpoint_training/ATLAS_FN_training_checkpoints_DockerRootReplica_20260707_133539, workspace=/content/atlas_fn_workspace, files=43
[atlas-extended] restore: restore phase complete | restored=['master_evidence', 'data_prep', 'checkpoint_training'], workspace=/content/atlas_fn_workspace, has_data=True, has_checkpoints=True, has_master=False


In [2]:
# ============================================================
# 0. Runtime and path configuration
# ============================================================
from __future__ import annotations

import os
import sys
import json
import math
import re
import time
import random
import shutil
import hashlib
import subprocess
import warnings
import contextlib
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

try:
    import yaml
except Exception:
    yaml = None

try:
    from PIL import Image
except Exception as e:
    raise RuntimeError("Pillow is required for image-size and inference utilities. Install with: pip install pillow") from e

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

@dataclass
class ExtendedEvidenceConfig:
    # Existing DockerRoot / ATLAS-FN workspace used by the DockerRoot and Master Evidence notebooks.
    # The notebook will robustly auto-discover this, so these are only fallback defaults.
    dockerroot_candidates: List[Path] = field(default_factory=lambda: [
        Path(os.environ.get("ATLAS_DOCKERROOT", "")) if os.environ.get("ATLAS_DOCKERROOT") else None,
        Path(os.environ.get("DOCKERROOT", "")) if os.environ.get("DOCKERROOT") else None,
        Path("/content/atlas_fn_workspace"),
        Path("/content/drive/MyDrive/DockerRoot"),
        Path("/content/drive/MyDrive"),
        Path("/workspace"),
        Path("/content/atlas_fn_workspace"),
        Path("/mnt/data"),
        Path.cwd(),
    ])

    # Root of the existing ATLAS-FN reproducibility package.
    # Auto-discovery seeks atlas_fn/, configs/config.yaml, MasterEvidence notebooks and checkpoints under DockerRoot.
    atlas_repro_root: Path = Path("/content/atlas_fn_workspace")

    # Multicentre landmark benchmark root and prepared head-annotation ledger.
    # If missing, the notebook clones the public repo and attempts to download/extract data archives.
    multicentre_root: Path = Path("/content/atlas_fn_workspace/data/raw/Multicentre-Fetal-Biometry")
    multicentre_annotations: Path = Path("/content/atlas_fn_workspace/data/raw/Multicentre-Fetal-Biometry/prepared_head_landmarks.csv")
    multicentre_repo_url: str = "https://github.com/surgical-vision/Multicentre-Fetal-Biometry.git"
    multicentre_repo_zip_url: str = "https://github.com/surgical-vision/Multicentre-Fetal-Biometry/archive/refs/heads/main.zip"
    multicentre_data_doi: str = "10.5522/04/30819911"
    auto_download_multicentre: bool = True
    auto_extract_archives: bool = True
    download_data_archives: bool = True
    download_timeout_sec: int = 120

    # Idempotent rerun/cache controls.  These make Runtime -> Run all safe after
    # the dataset has already been downloaded/prepared once.
    reuse_existing_multicentre_data: bool = True   # raw dataset may be reused; ledgers are rebuilt fresh
    reuse_prepared_external_ledger: bool = False
    force_redownload_multicentre: bool = False
    force_rebuild_external_ledger: bool = True
    allow_network_downloads: bool = True
    max_cache_validation_rows: int = 25

    # v8 progress/telemetry controls. These keep long cells visibly active.
    progress_enabled: bool = True
    progress_mininterval_sec: float = 1.0
    progress_log_every_rows: int = 250
    scan_progress_every_entries: int = 10_000
    print_dataframe_head_rows: int = 10
    status_jsonl_name: str = "extended_evidence_progress_log.jsonl"

    # v8 checkpoint compatibility controls. DockerRoot/Master-Evidence YOLO26 checkpoints
    # may have been pickled with a custom class named SPPF_YOLO26_Compatible under
    # __main__. Registering this shim before torch/Ultralytics loading prevents
    # AttributeError: Can't get attribute 'SPPF_YOLO26_Compatible' on <module '__main__'>.
    register_yolo26_compat_shim: bool = True
    preflight_yolo_checkpoint_load: bool = False

    # Existing HC18 root. HC18 remains the ATLAS-FN source/development benchmark and is
    # excluded from the *primary external* cohort by default to avoid circular evidence.
    hc18_root: Path = Path("/content/atlas_fn_workspace/data/HC18")

    # v11 domain policy. The multicentre benchmark may contain HC18 rows because HC18 is one
    # component of that public benchmark, but primary external evidence must use non-HC18
    # domains unless this is intentionally changed. HC18 can be included only as an explicitly
    # labelled internal/source-domain anchor, not as cross-dataset external validation.
    training_source_domain: str = "HC18"
    primary_external_domains: Tuple[str, ...] = ("FP", "UCL")
    include_training_source_domain_as_anchor: bool = False
    exclude_training_source_from_primary: bool = True
    ties_domains: Tuple[str, ...] = ("FP", "UCL")
    reuse_postsplit_working_ledger: bool = False

    # v11 ledger-policy controls. These prevent stale v8/v9 post-split caches from
    # being used as evidence after the domain-policy correction.
    ledger_schema_version: str = "v12_true_source_shm_safe_complete_ties"
    use_true_image_source_as_domain: bool = True
    exclude_multicentre_aggregate_as_domain: bool = True
    deduplicate_true_source_image_rows: bool = True
    primary_eval_split: str = "test"
    force_subject_split_when_no_test: bool = True

    # Existing ATLAS-FN Find checkpoint. Auto-discovery searches DockerRoot for BrainLocator/YOLO26n checkpoints.
    base_find_checkpoint: Path = Path("/content/atlas_fn_workspace/Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt")

    # Optional domain-specific checkpoints for TIES. If missing and RUN_TRAINING is False,
    # TIES will run in analysis-only mode if synthetic/demo vectors are enabled, otherwise it will skip.
    domain_checkpoints: Dict[str, Path] = field(default_factory=lambda: {
        "FP": Path("/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/find_FP/best.pt"),
        "UCL": Path("/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/find_UCL/best.pt"),
        # "HC18": Path("/workspace/checkpoints/find_HC18.pt"),  # source-domain anchor only; disabled by default
        # M-C is deliberately absent: it is an aggregate annotation container, not an image-source domain.
    })

    output_root: Path = Path(os.environ.get("ATLAS_EXTENDED_OUTPUT_ROOT", "/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab"))

    # Inference/training controls
    run_find_inference: bool = True
    run_yolo_training: bool = True       # v11 default: run Find fine-tuning so Experiment 2 is complete.
    # Fine-tune Find only. Measure is deterministic geometry + pixel spacing and is never trained.
    # Options: "domain_specific", "pooled", "both". Domain-specific enables TIES.
    yolo_training_strategy: str = "both"  # v11: domain specialists for TIES + pooled model for Exp1 sensitivity
    yolo_train_domains: Tuple[str, ...] = ("FP", "UCL")  # HC18 excluded by default; M-C is an aggregate container, not a trainable domain
    train_pooled_external_find: bool = True
    min_train_images_for_finetune: int = 50  # UCL is small; keep valid subject-exclusive train/val/test rather than skipping it.
    run_ties_merge: bool = True
    allow_demo_vectors_if_checkpoints_missing: bool = False  # Must be False for submission-grade evidence.
    require_complete_exp2: bool = False        # v11 fail-fast guard: do not export if TIES did not actually run.
    create_executable_ties_checkpoint: bool = True
    evaluate_ties_merged_checkpoint: bool = True
    reuse_existing_finetuned_checkpoints: bool = False
    ties_density_primary: float = 0.20
    ties_density_grid: Tuple[float, ...] = (0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40)
    exp2_required_model_names: Tuple[str, ...] = ("base", "specialist_FP", "specialist_UCL", "ties_merged_density_020")

    # YOLO training defaults for optional domain fine-tuning.
    # v12 deliberately uses SHM-safe defaults. In Docker/Colab, PyTorch dataloader
    # workers can fail with bus errors or OSError [Errno 28] because /dev/shm or
    # semaphore quotas are small. workers=0 avoids multiprocessing queues entirely.
    yolo_imgsz: int = 640
    yolo_epochs: int = 50
    yolo_patience: int = 10
    yolo_batch: int = 8
    yolo_safe_retry_batch: int = 4
    yolo_conf: float = 0.25
    yolo_lr0: float = 0.001
    yolo_lrf: float = 0.01
    yolo_weight_decay: float = 0.0005
    yolo_workers: int = 0
    yolo_safe_retry_workers: int = 0
    yolo_device: Optional[str] = "0"  # e.g., "0" on Colab/Docker GPU; None lets Ultralytics decide.
    yolo_cache: bool = False
    yolo_amp: bool = False             # AMP checks add overhead and can destabilise small Docker SHM environments.
    yolo_plots: bool = False           # Reduce disk writes during repeated runs.
    yolo_save_period: int = -1         # Do not save per-epoch checkpoints.
    yolo_freeze: Optional[int] = None  # optionally freeze early layers; None keeps all trainable.
    yolo_retry_on_shm_error: bool = True
    yolo_cleanup_before_train: bool = True
    yolo_delete_dataset_caches_before_train: bool = True
    yolo_clean_incomplete_runs: bool = True

    # Pixel spacing rules inherited from ATLAS-FN.
    spacing_min_mm: float = 0.04
    spacing_max_mm: float = 0.40
    spacing_fallback_mm: float = 0.125

    # Geometry export controls
    ref_box_margin_px: float = 12.0      # Margin around BPD/OFD landmark endpoints when deriving axis-aligned head box.
    bootstrap_n: int = 10_000
    bootstrap_seed: int = 42

CFG = ExtendedEvidenceConfig()

# Final 04 default: fresh run of Extended Evidence/TIES outputs.
# This purges only the Extended Evidence output root, not restored data/checkpoints/master-evidence artefacts.
FINAL04_FRESH_RUN = bool(globals().get("FINAL04_FRESH_RUN", True))
if FINAL04_FRESH_RUN and CFG.output_root.exists():
    shutil.rmtree(CFG.output_root)
CFG.output_root.mkdir(parents=True, exist_ok=True)
(CFG.output_root / "tables").mkdir(exist_ok=True)
(CFG.output_root / "figures").mkdir(exist_ok=True)
(CFG.output_root / "yolo_exports").mkdir(exist_ok=True)
(CFG.output_root / "logs").mkdir(exist_ok=True)

# ============================================================
# v6 telemetry utilities: timestamped prints, JSONL progress log and tqdm wrappers
# ============================================================
try:
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover - fallback for very minimal runtimes
    tqdm = None

NOTEBOOK_START_TS = time.time()
PROGRESS_LOG_PATH = CFG.output_root / "logs" / CFG.status_jsonl_name

def _json_safe(v):
    if isinstance(v, Path):
        return str(v)
    if isinstance(v, (np.integer, np.floating)):
        return v.item()
    if isinstance(v, dict):
        return {str(k): _json_safe(val) for k, val in v.items()}
    if isinstance(v, (list, tuple)):
        return [_json_safe(x) for x in v]
    return v

def log_event(stage: str, message: str, **kwargs):
    """Timestamped screen print + JSONL telemetry. Safe in Colab/Jupyter/Docker."""
    elapsed = time.time() - NOTEBOOK_START_TS
    stamp = time.strftime("%H:%M:%S")
    line = f"[{stamp} +{elapsed:7.1f}s] [{stage}] {message}"
    if kwargs:
        compact = ", ".join(f"{k}={v}" for k, v in kwargs.items() if k not in {"large"})
        if compact:
            line += " | " + compact
    print(line, flush=True)
    try:
        PROGRESS_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
        with open(PROGRESS_LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps({"time": stamp, "elapsed_sec": elapsed, "stage": stage, "message": message, **_json_safe(kwargs)}, default=str) + "\n")
    except Exception:
        pass

@contextlib.contextmanager
def stage_timer(stage: str, message: str):
    t0 = time.time()
    log_event(stage, "START: " + message)
    try:
        yield
    except Exception as e:
        log_event(stage, "ERROR: " + message, duration_sec=round(time.time() - t0, 2), error=repr(e))
        raise
    else:
        log_event(stage, "DONE: " + message, duration_sec=round(time.time() - t0, 2))

def progress_iter(iterable, desc: str = "", total=None, stage: str = "progress", **tqdm_kwargs):
    """Rerun-safe progress iterator.

    This wrapper intentionally removes duplicate tqdm keyword arguments before
    calling tqdm. It prevents the recurrent Colab failure:
        TypeError: tqdm() got multiple values for keyword argument 'leave'
    when downstream TIES loops pass leave=False or custom mininterval values.
    """
    desc = tqdm_kwargs.pop("desc", desc)
    total = tqdm_kwargs.pop("total", total)
    if total is None:
        try:
            total = len(iterable)
        except Exception:
            total = None
    tqdm_kwargs.setdefault("leave", True)
    tqdm_kwargs.setdefault("mininterval", getattr(CFG, "progress_mininterval_sec", 1.0))
    if getattr(CFG, "progress_enabled", True) and tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, **tqdm_kwargs)
    try:
        log_event(stage, f"{desc}: progress bar unavailable", total=total)
    except Exception:
        pass
    return iterable

def show_df(df: pd.DataFrame, name: str, max_rows: Optional[int] = None):
    """Consistent screen output for dataframe checkpoints."""
    max_rows = CFG.print_dataframe_head_rows if max_rows is None else max_rows
    if df is None:
        log_event("DATA", f"{name}: None")
        return
    log_event("DATA", f"{name}", rows=len(df), columns=len(df.columns) if hasattr(df, 'columns') else 'NA')
    try:
        display(df.head(max_rows))
    except Exception:
        print(df.head(max_rows).to_string(), flush=True)

def file_count_fast(root: Path, suffixes: Optional[set] = None, limit: Optional[int] = None) -> int:
    root = Path(root)
    if not root.exists():
        return 0
    n = 0
    for p in root.rglob('*'):
        if p.is_file() and (suffixes is None or p.suffix.lower() in suffixes):
            n += 1
            if limit and n >= limit:
                return n
    return n

log_event("BOOT", "Progress telemetry initialised", log=str(PROGRESS_LOG_PATH))

print("Extended Evidence output root:", CFG.output_root)
print(json.dumps({
    k: (str(v) if isinstance(v, Path) else [str(x) for x in v if x is not None] if isinstance(v, list) else {kk: str(vv) for kk, vv in v.items()} if isinstance(v, dict) else v)
    for k, v in asdict(CFG).items()
}, indent=2))


# ============================================================
# v8 checkpoint compatibility shim for DockerRoot/MasterEvidence YOLO26 checkpoints
# ============================================================
def register_yolo26_compatibility_shims(verbose: bool = True) -> bool:
    """Register compatibility classes required by ATLAS-FN DockerRoot YOLO26 checkpoints.

    Some ATLAS-FN Find checkpoints were saved from notebooks where the custom
    `SPPF_YOLO26_Compatible` module lived in `__main__`. PyTorch pickle therefore
    expects `__main__.SPPF_YOLO26_Compatible` to exist at load time. This function
    defines that class before any Ultralytics `YOLO(...)` or raw `torch.load(...)`
    call, and also exposes it through common Ultralytics namespaces as a safeguard.
    """
    if not getattr(CFG, "register_yolo26_compat_shim", True):
        return False
    try:
        import __main__
        import torch
        import torch.nn as nn
    except Exception as e:
        if verbose:
            try:
                log_event("COMPAT", "shim registration skipped: torch unavailable", error=repr(e))
            except Exception:
                print("[COMPAT] shim registration skipped: torch unavailable", repr(e), flush=True)
        return False

    if not hasattr(__main__, "SPPF_YOLO26_Compatible"):
        class SPPF_YOLO26_Compatible(nn.Module):
            """YOLO26-compatible SPPF block for unpickling historical ATLAS-FN checkpoints.

            The checkpoint supplies the learned submodules (`cv1`, `cv2`, `m`) via
            pickle state. The constructor is intentionally permissive; the forward
            pass mirrors Ultralytics SPPF: cv1 -> three max-pool applications -> cv2.
            """
            def __init__(self, *args, **kwargs):
                super().__init__()
                # Fallback modules are only used if an object is instantiated from scratch.
                self.cv1 = kwargs.get("cv1", nn.Identity())
                self.cv2 = kwargs.get("cv2", nn.Identity())
                self.m = kwargs.get("m", nn.MaxPool2d(kernel_size=5, stride=1, padding=2))

            def forward(self, x):
                y0 = self.cv1(x)
                y1 = self.m(y0)
                y2 = self.m(y1)
                y3 = self.m(y2)
                return self.cv2(torch.cat((y0, y1, y2, y3), dim=1))

        __main__.SPPF_YOLO26_Compatible = SPPF_YOLO26_Compatible
        globals()["SPPF_YOLO26_Compatible"] = SPPF_YOLO26_Compatible
    else:
        globals()["SPPF_YOLO26_Compatible"] = getattr(__main__, "SPPF_YOLO26_Compatible")

    cls = getattr(__main__, "SPPF_YOLO26_Compatible")
    # Expose through likely Ultralytics namespaces too. This is harmless when modules are absent.
    for mod_name in ["ultralytics.nn.modules.block", "ultralytics.nn.tasks", "ultralytics.nn.modules"]:
        try:
            mod = __import__(mod_name, fromlist=["dummy"])
            setattr(mod, "SPPF_YOLO26_Compatible", cls)
        except Exception:
            pass
    # PyTorch >=2.6 safe-global registration. Harmless if unavailable.
    try:
        import torch.serialization as serialization
        if hasattr(serialization, "add_safe_globals"):
            serialization.add_safe_globals([cls])
    except Exception:
        pass
    if verbose:
        try:
            log_event("COMPAT", "YOLO26 compatibility shim registered", class_name="SPPF_YOLO26_Compatible")
        except Exception:
            print("[COMPAT] YOLO26 compatibility shim registered", flush=True)
    return True

# Register once during boot so all later training/inference/TIES cells inherit the class in __main__.
register_yolo26_compatibility_shims(verbose=True)


[12:11:28 +    0.0s] [BOOT] Progress telemetry initialised | log=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/logs/extended_evidence_progress_log.jsonl
Extended Evidence output root: /content/atlas_fn_workspace/outputs/extended_evidence_ties_colab
{
  "dockerroot_candidates": [
    "/content/atlas_fn_workspace",
    "/content/atlas_fn_workspace",
    "/content/drive/MyDrive/DockerRoot",
    "/content/drive/MyDrive",
    "/workspace",
    "/content/atlas_fn_workspace",
    "/mnt/data",
    "/content"
  ],
  "atlas_repro_root": "/content/atlas_fn_workspace",
  "multicentre_root": "/content/atlas_fn_workspace/data/raw/Multicentre-Fetal-Biometry",
  "multicentre_annotations": "/content/atlas_fn_workspace/data/raw/Multicentre-Fetal-Biometry/prepared_head_landmarks.csv",
  "multicentre_repo_url": "https://github.com/surgical-vision/Multicentre-Fetal-Biometry.git",
  "multicentre_repo_zip_url": "https://github.com/surgical-vision/Multicentre-Fetal-Biometry/archive/refs/hea

True

In [3]:
# ============================================================
# 0C. Resolve restored workspace paths and checkpoint anchors
#     v4: bulletproof base-Find checkpoint resolution
# ============================================================
from pathlib import Path
import json, os, shutil, re

WORKSPACE = Path(os.environ.get("ATLAS_FN_WORKSPACE", "/content/atlas_fn_workspace"))
CFG.atlas_repro_root = WORKSPACE
CFG.multicentre_root = WORKSPACE / "data" / "raw" / "Multicentre-Fetal-Biometry"
CFG.multicentre_annotations = CFG.multicentre_root / "prepared_head_landmarks.csv"
CFG.output_root = Path(os.environ.get("ATLAS_EXTENDED_OUTPUT_ROOT", str(WORKSPACE / "outputs" / "extended_evidence_ties_colab")))
for sub in ["tables", "figures", "logs", "checkpoints", "yolo_exports", "packages"]:
    (CFG.output_root / sub).mkdir(parents=True, exist_ok=True)

CHECKPOINT_RESOLUTION_ROWS = []

def _record_ckpt_resolution(role: str, path: Path, reason: str, selected: bool = False):
    p = Path(path) if path is not None else None
    CHECKPOINT_RESOLUTION_ROWS.append({
        "role": role,
        "path": str(p) if p is not None else None,
        "exists": bool(p is not None and p.exists()),
        "bytes": int(p.stat().st_size) if p is not None and p.exists() and p.is_file() else 0,
        "reason": reason,
        "selected": bool(selected),
    })

def first_existing(paths, role: str = "checkpoint"):
    for p in paths:
        if p is None:
            continue
        p = Path(p)
        _record_ckpt_resolution(role, p, "explicit_candidate", selected=False)
        if p.exists() and p.is_file() and p.stat().st_size > 0:
            _record_ckpt_resolution(role, p, "explicit_candidate_selected", selected=True)
            return p
    return None

def _score_base_find_candidate(p: Path) -> int:
    sp = str(p).lower()
    name = p.name.lower()
    score = 0
    if name in {"best.pt", "find.pt"}: score += 5
    if "brainlocator" in sp: score += 40
    if "yolo26n_musgd" in sp or "musgd" in sp: score += 25
    if "/checkpoints/find/" in sp or sp.endswith("/checkpoints/find.pt"): score += 30
    if "master_evidence" in sp: score += 10
    # Strong exclusions: these are not the HC18 source Find checkpoint.
    bad_terms = ["/fp/", "_fp", "find_fp", "/ucl/", "_ucl", "find_ucl", "all_external", "pooled", "ties", "nano", "measure", "confirm", "sam2", "s1_", "s3_"]
    if any(t in sp for t in bad_terms): score -= 1000
    return score

def discover_base_find_checkpoint(workspace: Path = WORKSPACE, restore_checkpoint_package_if_missing: bool = True) -> Optional[Path]:
    """Locate the source HC18 Find/BrainLocator checkpoint and standardise its path.

    Supported layouts:
      - DockerRoot: Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt
      - checkpoint package: checkpoints/find/best.pt
      - master-evidence package: outputs/**/checkpoints/find.pt or find/best.pt

    If the checkpoint package has not yet been restored and an export archive is available,
    this function performs a one-time optional restore, then repeats discovery.
    """
    conventional = workspace / "Training_Runs" / "BrainLocator_YOLO26n_MuSGD" / "weights" / "best.pt"
    explicit = [
        conventional,
        workspace / "Training_Runs" / "BrainLocator_YOLO26n_MuSGD" / "best.pt",
        workspace / "checkpoints" / "find" / "best.pt",
        workspace / "checkpoints" / "find.pt",
        workspace / "outputs" / "master_evidence_inference" / "checkpoints" / "find.pt",
        workspace / "outputs" / "master_evidence_inference" / "checkpoints" / "find" / "best.pt",
    ]
    hit = first_existing(explicit, role="base_find")
    if hit is None:
        globbed = []
        for pat in ["**/BrainLocator_YOLO26n_MuSGD/**/best.pt", "**/checkpoints/find/best.pt", "**/checkpoints/find.pt", "**/*BrainLocator*.pt", "**/*find*.pt"]:
            try:
                globbed.extend(list(workspace.glob(pat)))
            except Exception:
                pass
        scored = []
        for p in set(globbed):
            if p.exists() and p.is_file() and p.stat().st_size > 0:
                sc = _score_base_find_candidate(p)
                _record_ckpt_resolution("base_find", p, f"recursive_candidate_score={sc}", selected=False)
                if sc > 0:
                    scored.append((sc, p.stat().st_size, p))
        if scored:
            scored.sort(reverse=True)
            hit = scored[0][2]
            _record_ckpt_resolution("base_find", hit, f"recursive_candidate_selected_score={scored[0][0]}", selected=True)

    # If not found, try restoring the checkpoint-training package now. Earlier cells may
    # have skipped it if any unrelated checkpoint existed.
    if hit is None and restore_checkpoint_package_if_missing:
        try:
            ck_archive = CHECKPOINT_TRAINING_ARCHIVE
        except NameError:
            ck_archive = None
        try:
            if ck_archive is None or (isinstance(ck_archive, Path) and not ck_archive.exists()):
                ck_archive = search_latest_archive(OPTIONAL_CHECKPOINT_PATTERNS)
            if ck_archive is not None and Path(ck_archive).exists():
                record_boot("restore", "base Find missing; restoring checkpoint-training archive", archive=ck_archive)
                res = restore_archive(Path(ck_archive), "checkpoint_training_late_base_find_repair", required=False)
                if res:
                    hit = discover_base_find_checkpoint(workspace, restore_checkpoint_package_if_missing=False)
        except Exception as e:
            try:
                record_boot("restore", "late checkpoint-training restore failed; continuing without fine-tuning", error=repr(e))
            except Exception:
                print("late checkpoint-training restore failed", repr(e))

    if hit is not None:
        # Standardise into DockerRoot path so later cells and exported audit records agree.
        conventional.parent.mkdir(parents=True, exist_ok=True)
        if hit.resolve() != conventional.resolve():
            try:
                shutil.copy2(hit, conventional)
                _record_ckpt_resolution("base_find", conventional, f"standardised_copy_from={hit}", selected=True)
                hit = conventional
            except Exception as e:
                _record_ckpt_resolution("base_find", conventional, f"standardise_copy_failed={repr(e)}", selected=False)
        return hit
    return None

base_ckpt = discover_base_find_checkpoint(WORKSPACE, restore_checkpoint_package_if_missing=True)
if base_ckpt is not None:
    CFG.base_find_checkpoint = Path(base_ckpt)
else:
    # Keep the conventional path for audit only; training/inference cells will skip gracefully.
    CFG.base_find_checkpoint = WORKSPACE / "Training_Runs" / "BrainLocator_YOLO26n_MuSGD" / "weights" / "best.pt"
    try:
        record_boot("checkpoint", "base Find checkpoint unavailable after all discovery/restore attempts", expected=CFG.base_find_checkpoint)
    except Exception:
        print("[checkpoint] base Find checkpoint unavailable after all discovery/restore attempts")

# If the training notebook already exported FP/UCL specialists, reuse them. Otherwise later cells train them.
DOMAIN_CANDIDATES = {
    "FP": [
        CFG.output_root / "checkpoints" / "find_FP.pt",
        CFG.output_root / "checkpoints" / "find_FP" / "best.pt",
        CFG.output_root / "yolo_exports" / "find_FP" / "weights" / "best.pt",
        WORKSPACE / "Training_Runs" / "Find_FP" / "weights" / "best.pt",
        WORKSPACE / "outputs" / "extended_evidence_ties_colab" / "checkpoints" / "find_FP.pt",
        WORKSPACE / "outputs" / "extended_evidence_ties_colab" / "checkpoints" / "find_FP" / "best.pt",
    ],
    "UCL": [
        CFG.output_root / "checkpoints" / "find_UCL.pt",
        CFG.output_root / "checkpoints" / "find_UCL" / "best.pt",
        CFG.output_root / "yolo_exports" / "find_UCL" / "weights" / "best.pt",
        WORKSPACE / "Training_Runs" / "Find_UCL" / "weights" / "best.pt",
        WORKSPACE / "outputs" / "extended_evidence_ties_colab" / "checkpoints" / "find_UCL.pt",
        WORKSPACE / "outputs" / "extended_evidence_ties_colab" / "checkpoints" / "find_UCL" / "best.pt",
    ],
}
resolved_domain = {}
for d, paths in DOMAIN_CANDIDATES.items():
    hit = first_existing(paths, role=f"domain_find_{d}")
    if hit is not None:
        resolved_domain[d] = hit
CFG.domain_checkpoints.update(resolved_domain)

# Colab-safe defaults inherited from the legacy Extended Evidence v12 notebook.
CFG.run_find_inference = True
CFG.run_yolo_training = RUN_EXTERNAL_FIND_FINETUNING
CFG.run_ties_merge = RUN_TIES
CFG.require_complete_exp2 = REQUIRE_COMPLETE_TIES_FOR_EXPORT
CFG.yolo_training_strategy = "both"
CFG.yolo_train_domains = ("FP", "UCL")
CFG.train_pooled_external_find = True
CFG.yolo_workers = 0
CFG.yolo_safe_retry_workers = 0
CFG.yolo_amp = False
CFG.yolo_cache = False
CFG.yolo_plots = False
CFG.yolo_device = "0"

# If the base Find checkpoint is absent, do not hard-stop here. Zero-shot,
# fine-tuning and TIES cells will record skip rows rather than crashing.
path_audit = {
    "workspace": str(WORKSPACE),
    "output_root": str(CFG.output_root),
    "multicentre_root": str(CFG.multicentre_root),
    "base_find_checkpoint": str(CFG.base_find_checkpoint),
    "base_find_exists": Path(CFG.base_find_checkpoint).exists(),
    "domain_checkpoints": {k: str(v) for k,v in CFG.domain_checkpoints.items()},
    "domain_checkpoint_exists": {k: Path(v).exists() for k,v in CFG.domain_checkpoints.items()},
}
(CFG.output_root / "logs" / "restored_workspace_path_resolution.json").write_text(json.dumps(path_audit, indent=2))
pd.DataFrame(CHECKPOINT_RESOLUTION_ROWS).to_csv(CFG.output_root / "tables" / "checkpoint_resolution_audit.csv", index=False)
print(json.dumps(path_audit, indent=2))
try:
    show_df(pd.DataFrame(CHECKPOINT_RESOLUTION_ROWS), "checkpoint resolution audit", max_rows=80)
except Exception:
    pass


{
  "workspace": "/content/atlas_fn_workspace",
  "output_root": "/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab",
  "multicentre_root": "/content/atlas_fn_workspace/data/raw/Multicentre-Fetal-Biometry",
  "base_find_checkpoint": "/content/atlas_fn_workspace/Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt",
  "base_find_exists": true,
  "domain_checkpoints": {
    "FP": "/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/find_FP/best.pt",
    "UCL": "/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/find_UCL/best.pt"
  },
  "domain_checkpoint_exists": {
    "FP": false,
    "UCL": false
  }
}
[12:11:32 +    4.0s] [DATA] checkpoint resolution audit | rows=17, columns=6


,role,path,exists,bytes,reason,selected
0,base_find,/content/atlas_fn_workspace/Training_Runs/Brai...,False,0,explicit_candidate,False
1,base_find,/content/atlas_fn_workspace/Training_Runs/Brai...,False,0,explicit_candidate,False
2,base_find,/content/atlas_fn_workspace/checkpoints/find/b...,True,5370949,explicit_candidate,False
3,base_find,/content/atlas_fn_workspace/checkpoints/find/b...,True,5370949,explicit_candidate_selected,True
4,base_find,/content/atlas_fn_workspace/Training_Runs/Brai...,True,5370949,standardised_copy_from=/content/atlas_fn_works...,True
5,domain_find_FP,/content/atlas_fn_workspace/outputs/extended_e...,False,0,explicit_candidate,False
6,domain_find_FP,/content/atlas_fn_workspace/outputs/extended_e...,False,0,explicit_candidate,False
7,domain_find_FP,/content/atlas_fn_workspace/outputs/extended_e...,False,0,explicit_candidate,False
8,domain_find_FP,/content/atlas_fn_workspace/Training_Runs/Find...,False,0,explicit_candidate,False
9,domain_find_FP,/content/atlas_fn_workspace/outputs/extended_e...,False,0,explicit_candidate,False


In [4]:

# ============================================================
# 0D. Final 04 cross-notebook restore and model-label contract — v2
# ============================================================
# Notebook 04 requires Notebook 01 data and Notebook 02 source/checkpoint package.
# Notebook 03 master-evidence export is useful for traceability but is not a dependency
# for FP/UCL external Find/Measure validation or TIES. Therefore a missing 03 archive is
# recorded as an optional-context warning, not a hard stop.

restore_bootstrap_dir = WORKSPACE / "outputs" / "extended_evidence_colab_bootstrap"
restore_contract_path = restore_bootstrap_dir / "restore_contract_from_01_02_03.json"
try:
    restore_contract = json.loads(restore_contract_path.read_text()) if restore_contract_path.exists() else {}
except Exception:
    restore_contract = {}

# Re-resolve base Find after restore. Cell 3 may have standardised a copy under Training_Runs.
base_find = Path(getattr(CFG, "base_find_checkpoint", WORKSPACE / "Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt"))
if not base_find.exists():
    for cand in [
        WORKSPACE / "Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt",
        WORKSPACE / "checkpoints/find/best.pt",
        WORKSPACE / "checkpoints/find_bmc/best.pt",
        WORKSPACE / "checkpoints/find.pt",
    ]:
        if cand.exists() and cand.stat().st_size > 0:
            base_find = cand
            try:
                CFG.base_find_checkpoint = cand
            except Exception:
                pass
            break

optimised_candidates = [
    WORKSPACE / "Training_Runs/BrainLocator_YOLO26n_OPTIMISED/weights/best.pt",
    WORKSPACE / "checkpoints/find_optimized/best.pt",
    WORKSPACE / "checkpoints/find_optimised/best.pt",
]
optimised_find = next((p for p in optimised_candidates if p.exists() and p.stat().st_size > 0), None)

model_label_rows = [
    {
        "model_label": "BMC_LOCKED_FIND",
        "role": "manuscript source Find and external zero-shot anchor",
        "path": str(base_find),
        "exists": bool(base_find.exists() and base_find.stat().st_size > 0),
        "used_for_primary_external": True,
    },
    {
        "model_label": "OPTIMISED_FIND",
        "role": "new optimised checkpoint from Notebook 02; diagnostic/sensitivity only",
        "path": str(optimised_find) if optimised_find else "",
        "exists": bool(optimised_find and optimised_find.exists() and optimised_find.stat().st_size > 0),
        "used_for_primary_external": False,
    },
]
MODEL_LABEL_REGISTRY_04 = pd.DataFrame(model_label_rows)
MODEL_LABEL_REGISTRY_04.to_csv(CFG.output_root / "tables" / "model_label_registry_bmc_vs_optimised_04.csv", index=False)

# Domain specialist checkpoints are expected to be generated later by Section 5.
# Their absence at the start is not a dependency failure.
initial_domain_paths = {
    "FP": Path(getattr(CFG, "output_root", WORKSPACE / "outputs" / "extended_evidence_ties_colab")) / "checkpoints" / "find_FP" / "best.pt",
    "UCL": Path(getattr(CFG, "output_root", WORKSPACE / "outputs" / "extended_evidence_ties_colab")) / "checkpoints" / "find_UCL" / "best.pt",
}
domain_initial_rows = []
for dom, p in initial_domain_paths.items():
    domain_initial_rows.append({
        "domain": dom,
        "expected_stage": "generated_by_section_5_yolo_finetuning_before_TIES",
        "path_checked_at_notebook_start": str(p),
        "exists_at_notebook_start": bool(p.exists() and p.stat().st_size > 0),
        "blocking_before_section_5": False,
    })
DOMAIN_SPECIALIST_INITIAL_AUDIT = pd.DataFrame(domain_initial_rows)
DOMAIN_SPECIALIST_INITIAL_AUDIT.to_csv(CFG.output_root / "tables" / "domain_specialist_checkpoint_initial_state_audit.csv", index=False)

# Required dependency policy.
dependency_rows = [
    {
        "dependency": "01_data_prep",
        "archive": restore_contract.get("notebook01_data_prep_archive"),
        "required_for_04_execution": True,
        "available_after_restore": bool(restore_contract.get("has_data")),
        "blocking": True,
        "policy": "required: contains complete HC/source workspace and prepared data roots",
    },
    {
        "dependency": "02_checkpoint_training",
        "archive": restore_contract.get("notebook02_checkpoint_archive"),
        "required_for_04_execution": True,
        "available_after_restore": bool(restore_contract.get("has_checkpoints")) and bool(base_find.exists()),
        "blocking": True,
        "policy": "required: provides BMC/source Find checkpoint and optional optimised checkpoint",
    },
    {
        "dependency": "03_master_evidence",
        "archive": restore_contract.get("notebook03_master_evidence_archive"),
        "required_for_04_execution": bool(globals().get("NOTEBOOK03_MASTER_EVIDENCE_REQUIRED_FOR_04", False)),
        "available_after_restore": bool(restore_contract.get("has_master")),
        "blocking": bool(globals().get("NOTEBOOK03_MASTER_EVIDENCE_REQUIRED_FOR_04", False)),
        "policy": "optional/contextual: used for traceability if present; not needed for FP/UCL external validation or TIES",
    },
    {
        "dependency": "04_fresh_run",
        "archive": None,
        "required_for_04_execution": True,
        "available_after_restore": bool(globals().get("FINAL04_FRESH_RUN", True)),
        "blocking": True,
        "policy": "required: run fresh extended-evidence outputs rather than reuse stale V5/V7 artefacts",
    },
]
CROSS_NOTEBOOK_DEPENDENCY_AUDIT = pd.DataFrame(dependency_rows)
CROSS_NOTEBOOK_DEPENDENCY_AUDIT["status"] = np.where(
    CROSS_NOTEBOOK_DEPENDENCY_AUDIT["available_after_restore"],
    "available",
    np.where(CROSS_NOTEBOOK_DEPENDENCY_AUDIT["blocking"], "blocking_missing", "optional_missing_continue"),
)
CROSS_NOTEBOOK_DEPENDENCY_AUDIT.to_csv(CFG.output_root / "tables" / "cross_notebook_dependency_audit_01_02_03_to_04.csv", index=False)

sequence_contract = {
    "notebook": "04_ExtendedEvidence_TIES_TRUE_FINAL_v2",
    "dependency_archives": restore_contract,
    "primary_external_model_label": "BMC_LOCKED_FIND",
    "optimised_find_policy": "available only as diagnostic/sensitivity; not substituted into manuscript external-validation or TIES baseline unless explicitly selected",
    "measure_policy": "deterministic geometry plus pixel spacing; no trained Measure checkpoint is used for FP/UCL external evidence",
    "confirm_policy": "external FP/UCL claims are Find/Measure only; no external Confirm validation is claimed",
    "notebook03_policy": "optional contextual restore; missing 03 archive does not block fresh FP/UCL/TIES execution",
    "domain_specialist_checkpoint_policy": "FP/UCL specialist checkpoints are expected to be generated by Section 5 and are not required at notebook start",
    "fresh_run": bool(globals().get("FINAL04_FRESH_RUN", True)),
}
(CFG.output_root / "manifests").mkdir(parents=True, exist_ok=True)
(CFG.output_root / "manifests" / "final04_sequence_contract.json").write_text(json.dumps(_json_safe(sequence_contract) if '_json_safe' in globals() else sequence_contract, indent=2))

print("Cross-notebook dependency audit")
display(CROSS_NOTEBOOK_DEPENDENCY_AUDIT)
print("Initial domain specialist checkpoint audit")
display(DOMAIN_SPECIALIST_INITIAL_AUDIT)
print("Model label registry")
display(MODEL_LABEL_REGISTRY_04)

blocking_fail = CROSS_NOTEBOOK_DEPENDENCY_AUDIT[
    (CROSS_NOTEBOOK_DEPENDENCY_AUDIT["blocking"] == True) &
    (CROSS_NOTEBOOK_DEPENDENCY_AUDIT["available_after_restore"] == False)
]
if len(blocking_fail):
    raise RuntimeError(
        "Final 04 blocking dependency audit failed. Missing required dependencies: " +
        "; ".join(blocking_fail["dependency"].astype(str).tolist()) +
        ". Notebook 03 is optional for 04 unless NOTEBOOK03_MASTER_EVIDENCE_REQUIRED_FOR_04=True."
    )


Cross-notebook dependency audit


,dependency,archive,required_for_04_execution,available_after_restore,blocking,policy,status
0,01_data_prep,/content/drive/MyDrive/ATLAS_FN_exports/ATLAS_...,True,True,True,required: contains complete HC/source workspac...,available
1,02_checkpoint_training,/content/drive/MyDrive/ATLAS_FN_exports/ATLAS_...,True,True,True,required: provides BMC/source Find checkpoint ...,available
2,03_master_evidence,/content/drive/MyDrive/ATLAS_FN_exports/ATLAS_...,False,False,False,optional/contextual: used for traceability if ...,optional_missing_continue
3,04_fresh_run,None,True,True,True,required: run fresh extended-evidence outputs ...,available


Initial domain specialist checkpoint audit


,domain,expected_stage,path_checked_at_notebook_start,exists_at_notebook_start,blocking_before_section_5
0,FP,generated_by_section_5_yolo_finetuning_before_...,/content/atlas_fn_workspace/outputs/extended_e...,False,False
1,UCL,generated_by_section_5_yolo_finetuning_before_...,/content/atlas_fn_workspace/outputs/extended_e...,False,False


Model label registry


,model_label,role,path,exists,used_for_primary_external
0,BMC_LOCKED_FIND,manuscript source Find and external zero-shot ...,/content/atlas_fn_workspace/Training_Runs/Brai...,True,True
1,OPTIMISED_FIND,new optimised checkpoint from Notebook 02; dia...,/content/atlas_fn_workspace/checkpoints/find_o...,True,False


In [5]:

# ============================================================
# 1A. Runtime dependency guard — bulletproof Colab execution
# ============================================================
# This notebook is an executable extended-evidence replica. Colab runtimes are not
# guaranteed to include Ultralytics, OpenCV, PyYAML or other runtime packages.
# This cell installs/verifies the required stack before any YOLO training,
# zero-shot inference or TIES checkpoint loading is attempted.
#
# Replication policy:
#   - Prefer the pinned Ultralytics version used in the successful Colab/DockerRoot
#     training logs.
#   - Record the installed version and import path.
#   - Do not require a runtime restart.
#   - If installation fails, downstream cells record a skip audit rather than
#     producing silent partial evidence.

import importlib
import subprocess
import sys
from pathlib import Path
from typing import Any, Dict, Optional
import json, os, time

ULTRALYTICS_PIN = os.environ.get("ATLAS_ULTRALYTICS_PIN", "8.4.87")
DEPENDENCY_AUDIT_ROWS = []

def _dependency_audit(name: str, package: str, status: str, version: Optional[str] = None,
                      detail: str = "", install_cmd: Optional[str] = None) -> None:
    DEPENDENCY_AUDIT_ROWS.append({
        "name": name,
        "package": package,
        "status": status,
        "version": version,
        "detail": detail,
        "install_cmd": install_cmd,
        "ts": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    })

def _module_version(module_name: str) -> Optional[str]:
    try:
        mod = importlib.import_module(module_name)
        return getattr(mod, "__version__", None)
    except Exception:
        return None

def _pip_install(args, label: str, quiet: bool = False) -> bool:
    cmd = [sys.executable, "-m", "pip", "install"]
    if quiet:
        cmd.append("-q")
    cmd += list(args)
    print(f"[dependency] installing {label}: {' '.join(cmd)}", flush=True)
    try:
        subprocess.check_call(cmd)
        return True
    except Exception as e:
        print(f"[dependency:warning] install failed for {label}: {repr(e)}", flush=True)
        return False

def ensure_importable(module_name: str, pip_spec: str, label: Optional[str] = None,
                      quiet: bool = False, required: bool = True) -> bool:
    label = label or module_name
    try:
        mod = importlib.import_module(module_name)
        _dependency_audit(label, pip_spec, "already_importable", getattr(mod, "__version__", None))
        return True
    except Exception as first_error:
        _dependency_audit(label, pip_spec, "missing_before_install", detail=repr(first_error))
    ok = _pip_install([pip_spec], label=label, quiet=quiet)
    if not ok:
        _dependency_audit(label, pip_spec, "install_failed", detail="pip returned non-zero")
        if required:
            return False
        return False
    try:
        mod = importlib.import_module(module_name)
        _dependency_audit(label, pip_spec, "installed_importable", getattr(mod, "__version__", None))
        return True
    except Exception as second_error:
        _dependency_audit(label, pip_spec, "install_but_import_failed", detail=repr(second_error))
        return False

def ensure_ultralytics_available(required: bool = True) -> bool:
    """Ensure Ultralytics YOLO is importable, pinned for YOLO26 compatibility where possible."""
    try:
        import ultralytics
        ver = getattr(ultralytics, "__version__", None)
        _dependency_audit("ultralytics", f"ultralytics=={ULTRALYTICS_PIN}", "already_importable", ver)
        return True
    except Exception as first_error:
        _dependency_audit("ultralytics", f"ultralytics=={ULTRALYTICS_PIN}", "missing_before_install", detail=repr(first_error))
    # Install pinned version first.
    ok = _pip_install([f"ultralytics=={ULTRALYTICS_PIN}"], label=f"ultralytics=={ULTRALYTICS_PIN}", quiet=False)
    if not ok:
        # Controlled fallback: still prefer completing evidence generation to hard failure,
        # but make drift explicit in the audit.
        ok = _pip_install(["ultralytics"], label="ultralytics fallback latest", quiet=False)
        if not ok:
            _dependency_audit("ultralytics", f"ultralytics=={ULTRALYTICS_PIN}", "install_failed", detail="pinned and fallback install failed")
            return False
        _dependency_audit("ultralytics", "ultralytics", "fallback_latest_installed", detail="pinned install failed; latest installed")
    try:
        import ultralytics
        ver = getattr(ultralytics, "__version__", None)
        _dependency_audit("ultralytics", f"ultralytics=={ULTRALYTICS_PIN}", "installed_importable", ver)
        return True
    except Exception as e:
        _dependency_audit("ultralytics", f"ultralytics=={ULTRALYTICS_PIN}", "install_but_import_failed", detail=repr(e))
        return False

# Minimal runtime stack used by adapter, metrics and figures. Torch is usually preinstalled
# on Colab GPU runtimes and should not be force-reinstalled unless the user deliberately
# changes the runtime image.
ensure_importable("yaml", "pyyaml", label="pyyaml", quiet=True, required=False)
ensure_importable("PIL", "pillow", label="pillow", quiet=True, required=False)
ensure_importable("cv2", "opencv-python-headless", label="opencv-python-headless", quiet=True, required=False)
ensure_importable("matplotlib", "matplotlib", label="matplotlib", quiet=True, required=False)
ensure_importable("scipy", "scipy", label="scipy", quiet=True, required=False)
ensure_ultralytics_available(required=False)

# Re-register compatibility shims after package installation because installing/importing
# ultralytics can replace module objects in sys.modules.
try:
    register_yolo26_compatibility_shims(verbose=True)
except Exception as e:
    print(f"[dependency:warning] YOLO26 compatibility shim registration after install failed: {repr(e)}", flush=True)
    _dependency_audit("yolo26_compatibility_shim", "not_a_package", "registration_failed", detail=repr(e))

# Write dependency audit early and keep it appendable for later cells.
try:
    dep_df = pd.DataFrame(DEPENDENCY_AUDIT_ROWS)
    dep_path = CFG.output_root / "tables" / "runtime_dependency_audit.csv"
    dep_path.parent.mkdir(parents=True, exist_ok=True)
    dep_df.to_csv(dep_path, index=False)
    show_df(dep_df, "runtime dependency audit", max_rows=40)
except Exception:
    print(json.dumps(DEPENDENCY_AUDIT_ROWS, indent=2))


# ============================================================
# 1. Environment and package discovery
# ============================================================
def sha256_file(path: Path, block_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(block_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def path_status(path: Path) -> Dict[str, Any]:
    path = Path(path)
    return {
        "path": str(path),
        "exists": path.exists(),
        "is_file": path.is_file(),
        "is_dir": path.is_dir(),
        "sha256": sha256_file(path) if path.exists() and path.is_file() else None,
    }

if CFG.atlas_repro_root.exists():
    sys.path.insert(0, str(CFG.atlas_repro_root))

ATLAS_IMPORTS = {}
try:
    from atlas_fn.biometry import bpd_hc_from_xyxy, bpd_error_mm, hc_error_pct, broad_pair_flag
    ATLAS_IMPORTS["atlas_fn.biometry"] = True
except Exception as e:
    ATLAS_IMPORTS["atlas_fn.biometry"] = repr(e)
    # Fallback implementations are defined below.

try:
    from atlas_fn.preprocessing import preprocess_image
    ATLAS_IMPORTS["atlas_fn.preprocessing"] = True
except Exception as e:
    ATLAS_IMPORTS["atlas_fn.preprocessing"] = repr(e)

print("Import status:")
print(json.dumps(ATLAS_IMPORTS, indent=2))
print("Checkpoint status:", path_status(CFG.base_find_checkpoint))


# Optional early checkpoint-load preflight. Disabled by default because loading a YOLO model can take time.
def preflight_yolo_checkpoint_load(checkpoint: Path, label: str = "base") -> Dict[str, Any]:
    checkpoint = Path(checkpoint)
    status = path_status(checkpoint)
    status["label"] = label
    if not checkpoint.exists():
        log_event("PREFLIGHT", "checkpoint missing", label=label, checkpoint=checkpoint)
        return status
    register_yolo26_compatibility_shims(verbose=False)
    if not getattr(CFG, "preflight_yolo_checkpoint_load", False):
        log_event("PREFLIGHT", "checkpoint load preflight skipped", label=label, checkpoint=checkpoint, note="set CFG.preflight_yolo_checkpoint_load=True to force an early YOLO load check")
        return status
    try:
        from ultralytics import YOLO
        _ = YOLO(str(checkpoint))
        status["yolo_loadable"] = True
        log_event("PREFLIGHT", "checkpoint loadable", label=label, checkpoint=checkpoint)
    except Exception as e:
        status["yolo_loadable"] = False
        status["load_error"] = repr(e)
        log_event("PREFLIGHT", "checkpoint load failed", label=label, checkpoint=checkpoint, error=repr(e))
    return status

_ = preflight_yolo_checkpoint_load(CFG.base_find_checkpoint, label="base_find")


[dependency] installing ultralytics==8.4.87: /usr/bin/python3 -m pip install ultralytics==8.4.87
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[12:11:37 +    8.7s] [COMPAT] YOLO26 compatibility shim registered | class_name=SPPF_YOLO26_Compatible
[12:11:37 +    8.8s] [DATA] runtime dependency audit | rows=7, columns=7


,name,package,status,version,detail,install_cmd,ts
0,pyyaml,pyyaml,already_importable,6.0.3,,None,2026-07-09T12:11:32Z
1,pillow,pillow,already_importable,11.3.0,,None,2026-07-09T12:11:32Z
2,opencv-python-headless,opencv-python-headless,already_importable,4.13.0,,None,2026-07-09T12:11:32Z
3,matplotlib,matplotlib,already_importable,3.10.0,,None,2026-07-09T12:11:32Z
4,scipy,scipy,already_importable,1.16.3,,None,2026-07-09T12:11:32Z
5,ultralytics,ultralytics==8.4.87,missing_before_install,None,"ModuleNotFoundError(""No module named 'ultralyt...",None,2026-07-09T12:11:32Z
6,ultralytics,ultralytics==8.4.87,installed_importable,8.4.87,,None,2026-07-09T12:11:36Z


Import status:
{
  "atlas_fn.biometry": "ModuleNotFoundError(\"No module named 'atlas_fn'\")",
  "atlas_fn.preprocessing": "ModuleNotFoundError(\"No module named 'atlas_fn'\")"
}
Checkpoint status: {'path': '/content/atlas_fn_workspace/Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt', 'exists': True, 'is_file': True, 'is_dir': False, 'sha256': 'bcc55db31feaf617d1643185c89c743b1450ef5de4b9dff2e422702a66ff14cd'}
[12:11:37 +    8.8s] [PREFLIGHT] checkpoint load preflight skipped | label=base_find, checkpoint=/content/atlas_fn_workspace/Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt, note=set CFG.preflight_yolo_checkpoint_load=True to force an early YOLO load check


In [6]:
# ============================================================
# 2. Geometry and metric utilities — matches ATLAS-FN Measure boundary
# ============================================================
def finite(x: Any) -> bool:
    try:
        return bool(np.isfinite(float(x)))
    except Exception:
        return False

# Fallback only if atlas_fn.biometry was not importable.
if "bpd_hc_from_xyxy" not in globals():
    def bpd_hc_from_xyxy(xyxy: Sequence[float], pixel_spacing_mm: float) -> Tuple[float, float]:
        if xyxy is None or len(xyxy) < 4:
            return float("nan"), float("nan")
        x1, y1, x2, y2 = map(float, xyxy[:4])
        bw_px, bh_px = abs(x2 - x1), abs(y2 - y1)
        if bw_px <= 0 or bh_px <= 0 or not finite(pixel_spacing_mm) or pixel_spacing_mm <= 0:
            return float("nan"), float("nan")
        bpd = min(bw_px, bh_px) * pixel_spacing_mm
        a, b = max(bw_px, bh_px) / 2.0, min(bw_px, bh_px) / 2.0
        hc = math.pi * math.sqrt(2.0 * (a*a + b*b)) * pixel_spacing_mm
        return bpd, hc

    def bpd_error_mm(auto_bpd: float, ref_bpd: float) -> float:
        return abs(auto_bpd - ref_bpd) if finite(auto_bpd) and finite(ref_bpd) else float("nan")

    def hc_error_pct(auto_hc: float, ref_hc: float) -> float:
        return 100.0 * abs(auto_hc - ref_hc) / ref_hc if finite(auto_hc) and finite(ref_hc) and ref_hc > 0 else float("nan")

    def broad_pair_flag(bpd_err: float, hc_err_pct: float, bpd_tol_mm: float = 2.0, hc_tol_pct: float = 5.0):
        return int(bpd_err <= bpd_tol_mm and hc_err_pct <= hc_tol_pct) if finite(bpd_err) and finite(hc_err_pct) else float("nan")

def point_distance(p1: Tuple[float, float], p2: Tuple[float, float]) -> float:
    return float(math.hypot(float(p1[0]) - float(p2[0]), float(p1[1]) - float(p2[1])))

def ellipse_hc_from_diameters_px(bpd_px: float, ofd_px: float, spacing_mm: float) -> float:
    if not (finite(bpd_px) and finite(ofd_px) and finite(spacing_mm)) or bpd_px <= 0 or ofd_px <= 0 or spacing_mm <= 0:
        return float("nan")
    a, b = max(bpd_px, ofd_px) / 2.0, min(bpd_px, ofd_px) / 2.0
    return math.pi * math.sqrt(2.0 * (a*a + b*b)) * spacing_mm

def reference_box_from_landmarks(points: Sequence[Tuple[float, float]], margin_px: float = 12.0, image_size: Optional[Tuple[int, int]] = None) -> List[float]:
    xs = [float(x) for x, y in points if finite(x) and finite(y)]
    ys = [float(y) for x, y in points if finite(x) and finite(y)]
    if not xs or not ys:
        return [float("nan")] * 4
    x1, x2 = min(xs) - margin_px, max(xs) + margin_px
    y1, y2 = min(ys) - margin_px, max(ys) + margin_px
    if image_size is not None:
        w, h = image_size
        x1, x2 = max(0, x1), min(w - 1, x2)
        y1, y2 = max(0, y1), min(h - 1, y2)
    return [float(x1), float(y1), float(x2), float(y2)]

def xyxy_iou(a: Sequence[float], b: Sequence[float]) -> float:
    if a is None or b is None or len(a) < 4 or len(b) < 4:
        return float("nan")
    ax1, ay1, ax2, ay2 = map(float, a[:4])
    bx1, by1, bx2, by2 = map(float, b[:4])
    if not all(map(finite, [ax1, ay1, ax2, ay2, bx1, by1, bx2, by2])):
        return float("nan")
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    aa = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    ba = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    denom = aa + ba - inter
    return float(inter / denom) if denom > 0 else float("nan")

def normalise_xywh_from_xyxy(xyxy: Sequence[float], w: int, h: int) -> Tuple[float, float, float, float]:
    x1, y1, x2, y2 = map(float, xyxy[:4])
    cx = ((x1 + x2) / 2.0) / w
    cy = ((y1 + y2) / 2.0) / h
    bw = abs(x2 - x1) / w
    bh = abs(y2 - y1) / h
    return cx, cy, bw, bh

def safe_spacing(x: Any, fallback: float = CFG.spacing_fallback_mm) -> Tuple[float, str]:
    try:
        val = float(x)
    except Exception:
        return fallback, "fallback_missing"
    if not np.isfinite(val):
        return fallback, "fallback_nonfinite"
    if CFG.spacing_min_mm <= val <= CFG.spacing_max_mm:
        return val, "metadata"
    return fallback, "fallback_out_of_range"

print("Geometry utilities loaded.")


Geometry utilities loaded.


## 2. Dataset adapter for the multicentre landmark benchmark

This cell automatically prepares the multicentre fetal-biometry annotations when possible.
It first reuses an already-downloaded dataset and cached prepared ledgers in DockerRoot/Drive. Network download is attempted only when the dataset assets are genuinely missing and `CFG.allow_network_downloads=True`. It then builds or loads a single head-landmark ledger compatible with the ATLAS-FN Find/Measure boundary.

Minimum fields needed for Experiment 1:

- image path or filename
- subject/patient identifier
- domain/source/site label (e.g., FP, HC18, UCL, M-C)
- split label (train/val/test), or a reproducible split can be generated
- pixel spacing in mm/px
- BPD and OFD landmark endpoints

The notebook derives reference BPD, OFD, HC and a reference axis-aligned head box from the landmarks. This makes it compatible with the ATLAS-FN Find/Measure boundary while explicitly preserving the landmark-vs-box geometry limitation.


In [7]:
# ============================================================
# 3. Dataset adapter
# ============================================================
AUTO_CANDIDATES = {
    "image_path": ["image_path", "img_path", "path", "filename", "file", "image", "Image", "file_name", "image_name", "ImageName", "name"],
    "subject_id": ["subject_id", "patient_id", "case_id", "patient", "subject", "PatientID", "SubjectID", "ID"],
    "domain": ["domain", "dataset", "source", "site", "centre", "center", "hospital", "Dataset", "_domain_from_path"],
    "split": ["split", "Split", "partition", "set"],
    "spacing": ["pixel_spacing", "pixel_spacing_mm", "spacing", "mm_per_pixel", "px_to_mm_rate", "px_to_mm", "PixelSpacing"],
    "bpd_x1": ["bpd_x1", "BPD_x1", "bpd1_x", "BPD1_x", "bpd_1_x", "BPD_1_x", "bpd_lx", "BPD_L_x"],
    "bpd_y1": ["bpd_y1", "BPD_y1", "bpd1_y", "BPD1_y", "bpd_1_y", "BPD_1_y", "bpd_ly", "BPD_L_y"],
    "bpd_x2": ["bpd_x2", "BPD_x2", "bpd2_x", "BPD2_x", "bpd_2_x", "BPD_2_x", "bpd_rx", "BPD_R_x"],
    "bpd_y2": ["bpd_y2", "BPD_y2", "bpd2_y", "BPD2_y", "bpd_2_y", "BPD_2_y", "bpd_ry", "BPD_R_y"],
    "ofd_x1": ["ofd_x1", "OFD_x1", "ofd1_x", "OFD1_x", "ofd_1_x", "OFD_1_x", "ofd_lx", "OFD_L_x"],
    "ofd_y1": ["ofd_y1", "OFD_y1", "ofd1_y", "OFD1_y", "ofd_1_y", "OFD_1_y", "ofd_ly", "OFD_L_y"],
    "ofd_x2": ["ofd_x2", "OFD_x2", "ofd2_x", "OFD2_x", "ofd_2_x", "OFD_2_x", "ofd_rx", "OFD_R_x"],
    "ofd_y2": ["ofd_y2", "OFD_y2", "ofd2_y", "OFD2_y", "ofd_2_y", "OFD_2_y", "ofd_ry", "OFD_R_y"],
}

# Override this if the public dataset uses different columns.
COLUMN_MAP: Dict[str, str] = {}

def _first_existing(cols: Sequence[str], candidates: Sequence[str]) -> Optional[str]:
    lower = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand in cols:
            return cand
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None

def infer_column_map(df: pd.DataFrame, manual: Optional[Dict[str, str]] = None) -> Dict[str, str]:
    manual = manual or {}
    out = {}
    for key, candidates in AUTO_CANDIDATES.items():
        if key in manual and manual[key] in df.columns:
            out[key] = manual[key]
        else:
            found = _first_existing(df.columns, candidates)
            if found is not None:
                out[key] = found
    return out

def read_annotation_file(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Annotation file not found: {path}")
    if path.suffix.lower() in [".csv", ".tsv"]:
        return pd.read_csv(path, sep="\t" if path.suffix.lower() == ".tsv" else ",")
    if path.suffix.lower() in [".json", ".jsonl"]:
        if path.suffix.lower() == ".jsonl":
            return pd.read_json(path, lines=True)
        return pd.read_json(path)
    raise ValueError(f"Unsupported annotation file extension: {path.suffix}")

def canonical_domain_token(x: Any) -> str:
    """Canonicalise source-domain labels without treating M-C as an image-source domain."""
    v = str(x).strip()
    low = v.lower().replace("_", "-").replace(" ", "")
    if low in {"hc18", "hc-18"}:
        return "HC18"
    if low in {"fp", "fetalplanes", "fetal-planes", "fetal-planes-db"}:
        return "FP"
    if low == "ucl":
        return "UCL"
    if low in {"m-c", "mc", "multi-centre", "multicentre", "multi-center", "multicenter", "multi-centre-aggregate"}:
        return "M-C_AGGREGATE"
    if low in {"nan", "none", "", "unknown"}:
        return "unknown"
    return v

def infer_true_image_source_domain(image_path: Any, source_csv: Any = None, declared_domain: Any = None) -> str:
    """Infer the true image-source domain from the resolved image path.

    The public multicentre benchmark includes aggregate M-C CSVs. Those CSVs are
    containers combining FP/HC18/UCL images; therefore the domain used for external
    validation must come from the resolved image location, not the CSV container.
    """
    probes = [image_path, source_csv]
    for probe in probes:
        parts = [p.upper().replace("_", "-") for p in Path(str(probe)).parts]
        # Check true image-source directories first.
        if "FP" in parts:
            return "FP"
        if "HC18" in parts or "HC-18" in parts:
            return "HC18"
        if "UCL" in parts:
            return "UCL"
    dec = canonical_domain_token(declared_domain)
    if dec in {"FP", "HC18", "UCL"}:
        return dec
    if dec == "M-C_AGGREGATE":
        return "M-C_AGGREGATE"
    return "unknown"

def _is_missing_subject(v: Any) -> bool:
    if v is None:
        return True
    try:
        if pd.isna(v):
            return True
    except Exception:
        pass
    s = str(v).strip()
    return s == "" or s.lower() in {"nan", "none", "unknown", "unknown_subject", "missing"}

def derive_subject_id(raw_subject: Any, image_path: Any, true_source_domain: str) -> Tuple[str, str]:
    """Return subject_id and provenance. UCL subjects are derived from filename prefix when absent."""
    domain = canonical_domain_token(true_source_domain)
    if not _is_missing_subject(raw_subject):
        s = str(raw_subject).strip()
        s = re.sub(r"\.0$", "", s)
        # Prefix UCL only if not already prefixed, so 002 remains unambiguous across domains.
        if domain == "UCL" and not s.upper().startswith("UCL"):
            return f"UCL_{s.zfill(3) if s.isdigit() else s}", "annotation_subject_prefixed"
        return s, "annotation_subject"
    stem = Path(str(image_path)).stem
    # UCL filenames have patterns such as 002_HC.jpeg, 002_3HC.jpeg, 121_4HC.jpeg.
    m = re.match(r"^(\d+)", stem)
    if m:
        prefix = m.group(1).zfill(3)
        return f"{domain}_{prefix}" if domain != "unknown" else f"SUBJ_{prefix}", "filename_prefix"
    # Fallback remains deterministic but is visibly not an annotated subject.
    digest = hashlib.sha1(str(image_path).encode("utf-8")).hexdigest()[:10]
    return f"{domain}_UNKNOWN_{digest}", "path_hash_fallback"

def _row_get(row: pd.Series, col: Optional[str], default: Any = np.nan) -> Any:
    if col is None:
        return default
    try:
        return row[col]
    except Exception:
        return default

def standardise_multicentre_annotations(annotation_path: Path, root: Path, column_map: Optional[Dict[str, str]] = None) -> pd.DataFrame:
    raw = read_annotation_file(annotation_path)
    cmap = infer_column_map(raw, column_map or COLUMN_MAP)
    # subject_id is optional in v11 because UCL may omit it; it is repaired from filename.
    required = ["image_path", "domain", "spacing", "bpd_x1", "bpd_y1", "bpd_x2", "bpd_y2", "ofd_x1", "ofd_y1", "ofd_x2", "ofd_y2"]
    missing = [k for k in required if k not in cmap]
    if missing:
        raise ValueError(
            "Could not infer required columns: " + ", ".join(missing) +
            f"\nAvailable columns: {list(raw.columns)}\nCurrent inferred map: {cmap}\n"
            "Update COLUMN_MAP at the top of this cell."
        )

    records = []
    log_event("ANN", "standardising annotation file using v11 true-source policy", path=annotation_path, rows=len(raw))
    for idx, r in progress_iter(raw.iterrows(), desc=f"standardise {Path(annotation_path).name}", total=len(raw), stage="ANN"):
        source_csv = str(_row_get(r, "_source_csv", annotation_path))
        rel = str(r[cmap["image_path"]])
        declared_raw = str(r[cmap["domain"]]) if "domain" in cmap else infer_domain_from_path(Path(source_csv))
        declared_domain = canonical_domain_token(declared_raw)
        split = str(r[cmap["split"]]) if "split" in cmap else "unknown"
        spacing, spacing_source = safe_spacing(r[cmap["spacing"]])
        # Resolve using the declared domain as a hint only; true domain is inferred from the resolved path below.
        resolve_hint = "M-C" if declared_domain == "M-C_AGGREGATE" else declared_domain
        img_path = resolve_dataset_image(Path(root), rel, domain=resolve_hint, anatomy="Head")
        true_source_domain = infer_true_image_source_domain(img_path, source_csv=source_csv, declared_domain=declared_domain)
        subject_raw = _row_get(r, cmap.get("subject_id"), np.nan)
        subject, subject_source = derive_subject_id(subject_raw, img_path, true_source_domain)

        bpd_p1 = (float(r[cmap["bpd_x1"]]), float(r[cmap["bpd_y1"]]))
        bpd_p2 = (float(r[cmap["bpd_x2"]]), float(r[cmap["bpd_y2"]]))
        ofd_p1 = (float(r[cmap["ofd_x1"]]), float(r[cmap["ofd_y1"]]))
        ofd_p2 = (float(r[cmap["ofd_x2"]]), float(r[cmap["ofd_y2"]]))

        try:
            with Image.open(img_path) as im:
                img_w, img_h = im.size
        except Exception:
            img_w, img_h = np.nan, np.nan

        bpd_px = point_distance(bpd_p1, bpd_p2)
        ofd_px = point_distance(ofd_p1, ofd_p2)
        ref_bpd_mm = bpd_px * spacing if finite(spacing) else np.nan
        ref_ofd_mm = ofd_px * spacing if finite(spacing) else np.nan
        ref_hc_mm = ellipse_hc_from_diameters_px(bpd_px, ofd_px, spacing)
        ref_xyxy = reference_box_from_landmarks([bpd_p1, bpd_p2, ofd_p1, ofd_p2], margin_px=CFG.ref_box_margin_px, image_size=(int(img_w), int(img_h)) if finite(img_w) and finite(img_h) else None)

        records.append({
            "row_id": idx,
            "image_path": str(img_path),
            "image_exists": img_path.exists(),
            "img_w": img_w,
            "img_h": img_h,
            "subject_id": subject,
            "subject_id_source": subject_source,
            "declared_domain": declared_domain,
            "true_source_domain": true_source_domain,
            "domain": true_source_domain,
            "split": split.lower(),
            "pixel_spacing_mm": spacing,
            "spacing_source": spacing_source,
            "bpd_p1_x": bpd_p1[0], "bpd_p1_y": bpd_p1[1],
            "bpd_p2_x": bpd_p2[0], "bpd_p2_y": bpd_p2[1],
            "ofd_p1_x": ofd_p1[0], "ofd_p1_y": ofd_p1[1],
            "ofd_p2_x": ofd_p2[0], "ofd_p2_y": ofd_p2[1],
            "ref_bpd_px": bpd_px,
            "ref_ofd_px": ofd_px,
            "ref_bpd_mm": ref_bpd_mm,
            "ref_ofd_mm": ref_ofd_mm,
            "ref_hc_mm": ref_hc_mm,
            "ref_x1": ref_xyxy[0], "ref_y1": ref_xyxy[1], "ref_x2": ref_xyxy[2], "ref_y2": ref_xyxy[3],
            "source_csv": source_csv,
            "column_map_json": json.dumps(cmap, sort_keys=True),
            "ledger_schema_version": getattr(CFG, "ledger_schema_version", "v11_true_image_source_domain"),
        })
    df = pd.DataFrame(records)
    return df

# ------------------------------------------------------------
# Automatic download / clone / extraction and head-annotation preparation
# ------------------------------------------------------------
def run_cmd(cmd: Sequence[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    log_event("CMD", "running", cmd=" ".join(map(str, cmd)), cwd=cwd or Path.cwd())
    t0 = time.time()
    proc = subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, check=check, text=True, capture_output=False)
    log_event("CMD", "finished", returncode=proc.returncode, elapsed_sec=round(time.time()-t0, 2))
    return proc

def download_file(url: str, dest: Path, timeout: int = 120, chunk_size: int = 1 << 20, force: bool = False) -> Path:
    """Streaming downloader with idempotent skip and a visible byte-progress bar."""
    import urllib.request
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0 and not force:
        log_event("DOWNLOAD", "exists; skipping", dest=dest, size_mb=round(dest.stat().st_size/1e6, 2))
        return dest
    if not CFG.allow_network_downloads:
        raise RuntimeError(f"Network downloads disabled and required file missing/stale: {dest}")
    log_event("DOWNLOAD", "start", url=url, dest=dest)
    t0 = time.time()
    with urllib.request.urlopen(url, timeout=timeout) as r:
        total = int(r.headers.get("Content-Length", "0") or 0)
        pbar = tqdm(total=total if total else None, unit="B", unit_scale=True, desc=f"download {dest.name}", leave=True) if (CFG.progress_enabled and tqdm is not None) else None
        with open(dest, "wb") as f:
            while True:
                chunk = r.read(chunk_size)
                if not chunk:
                    break
                f.write(chunk)
                if pbar is not None:
                    pbar.update(len(chunk))
        if pbar is not None:
            pbar.close()
    log_event("DOWNLOAD", "done", dest=dest, size_mb=round(dest.stat().st_size/1e6, 2), elapsed_sec=round(time.time()-t0, 2))
    return dest
    if not CFG.allow_network_downloads:
        raise RuntimeError(f"Network downloads disabled and required file missing/stale: {dest}")
    print(f"[download] {url} -> {dest}")
    req = urllib.request.Request(url, headers={"User-Agent": "ATLAS-FN-ExtendedEvidence/5.0"})
    tmp = dest.with_suffix(dest.suffix + ".partial")
    with urllib.request.urlopen(req, timeout=timeout) as r, open(tmp, "wb") as f:
        while True:
            chunk = r.read(chunk_size)
            if not chunk:
                break
            f.write(chunk)
    tmp.replace(dest)
    return dest


def looks_like_multicentre_root(root: Path) -> bool:
    """Return True when a path appears to contain the Multicentre Fetal Biometry benchmark.

    This helper is deliberately independent of later notebook functions so it is safe to
    call during repository/download discovery.  It recognises either the GitHub repository
    root or an already extracted data root containing Head_Train/Head_Test CSV files.
    """
    root = Path(root)
    if not root.exists():
        return False

    # Direct repository/data-root signals.
    direct_signals = [
        root / "README.md",
        root / "data" / "README-FP.md",
        root / "data" / "README-UCL.md",
        root / "data" / "annotations",
        root / "data" / "images",
        root / "annotations",
        root / "images",
    ]
    if any(p.exists() for p in direct_signals):
        # Still require at least one annotation-like sign where possible.
        try:
            if any(root.glob("**/Head_*.csv")):
                return True
        except Exception:
            pass
        if (root / "data" / "annotations").exists() or (root / "annotations").exists():
            return True

    # Robust fallback: find a small number of expected landmark CSV names.
    try:
        head_csvs = []
        for pat in ("**/Head_Train.csv", "**/Head_Test.csv", "**/*Head*Train*.csv", "**/*Head*Test*.csv"):
            head_csvs.extend(list(root.glob(pat))[:5])
            if len(head_csvs) >= 2:
                return True
        return len(head_csvs) > 0
    except Exception:
        return False


def clone_or_download_repo(repo_url: str, zip_url: str, dest: Path) -> Path:
    dest = Path(dest)
    if looks_like_multicentre_root(dest) or (dest / "README.md").exists() or (dest / ".git").exists():
        print(f"[repo] already present; clone skipped: {dest}")
        return dest
    if not CFG.allow_network_downloads:
        print(f"[repo] network disabled and repo not present: {dest}")
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    # Prefer git clone; fallback to GitHub ZIP if git is unavailable.
    try:
        run_cmd(["git", "clone", "--depth", "1", repo_url, str(dest)], check=True)
        return dest
    except Exception as e:
        print("[repo] git clone failed, falling back to zip:", repr(e))
    zip_path = dest.parent / "Multicentre-Fetal-Biometry-main.zip"
    try:
        download_file(zip_url, zip_path, timeout=CFG.download_timeout_sec, force=CFG.force_redownload_multicentre)
        shutil.unpack_archive(str(zip_path), str(dest.parent))
        extracted = dest.parent / "Multicentre-Fetal-Biometry-main"
        if extracted.exists() and not dest.exists():
            extracted.rename(dest)
    except Exception as e:
        print("[repo] GitHub ZIP download also failed. Continuing with expected path and clear downstream audit:", repr(e))
    return dest

def figshare_file_manifest_from_doi(doi: str) -> List[Dict[str, Any]]:
    """Resolve a Figshare-style DOI such as 10.5522/04/30819911 to file URLs."""
    import urllib.request
    doi_tail = doi.rstrip('/').split('/')[-1]
    article_id = ''.join(ch for ch in doi_tail if ch.isdigit())
    if not article_id:
        return []
    api_url = f"https://api.figshare.com/v2/articles/{article_id}"
    try:
        with urllib.request.urlopen(api_url, timeout=CFG.download_timeout_sec) as r:
            meta = json.loads(r.read().decode('utf-8'))
        return meta.get("files", []) or []
    except Exception as e:
        print(f"[data] Figshare/UCL API lookup failed for {doi}: {e}")
        return []

def extract_archive(archive: Path, dest: Path) -> None:
    archive, dest = Path(archive), Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    try:
        print(f"[extract] {archive.name} -> {dest}")
        shutil.unpack_archive(str(archive), str(dest))
    except Exception as e:
        print(f"[extract] skipped/failed for {archive}: {e}")

def multicentre_asset_status(root: Path) -> Dict[str, Any]:
    """Fast-ish asset audit used to decide whether network is needed on rerun."""
    root = Path(root)
    head_csvs = find_head_annotation_csvs(root) if root.exists() else []
    image_count = 0
    sample_images = []
    if root.exists():
        for base in [root, root / "data"]:
            if not base.exists():
                continue
            for p in progress_iter(base.glob("**/*"), desc=f"index images under {base.name}", stage="DATA"):
                if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES:
                    image_count += 1
                    if len(sample_images) < 5:
                        sample_images.append(str(p))
                    if image_count >= 25:
                        break
            if image_count >= 25:
                break
    archives = []
    if root.exists():
        archives = [p for p in (root / "data_downloads").glob("*") if p.is_file()] if (root / "data_downloads").exists() else []
    return {
        "root": str(root),
        "root_exists": root.exists(),
        "looks_like_repo_or_data_root": looks_like_multicentre_root(root),
        "head_csv_count": len(head_csvs),
        "has_head_csv": len(head_csvs) > 0,
        "sample_head_csvs": [str(p) for p in head_csvs[:5]],
        "sample_image_count_capped_at_25": image_count,
        "has_images": image_count > 0,
        "sample_images": sample_images,
        "downloaded_archive_count": len(archives),
        "sample_archives": [str(p) for p in archives[:5]],
    }

def cache_manifest_path(cfg: ExtendedEvidenceConfig) -> Path:
    return Path(cfg.output_root) / "manifests" / "multicentre_dataset_cache_manifest.json"

def write_cache_manifest(cfg: ExtendedEvidenceConfig, status: Dict[str, Any], extra: Optional[Dict[str, Any]] = None) -> None:
    mp = cache_manifest_path(cfg)
    mp.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "created_utc": pd.Timestamp.utcnow().isoformat(),
        "multicentre_root": str(cfg.multicentre_root),
        "multicentre_annotations": str(cfg.multicentre_annotations),
        "status": status,
        "config": {
            "reuse_existing_multicentre_data": cfg.reuse_existing_multicentre_data,
            "reuse_prepared_external_ledger": cfg.reuse_prepared_external_ledger,
            "force_redownload_multicentre": cfg.force_redownload_multicentre,
            "force_rebuild_external_ledger": cfg.force_rebuild_external_ledger,
            "allow_network_downloads": cfg.allow_network_downloads,
        },
    }
    if extra:
        payload.update(extra)
    mp.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"[cache] wrote manifest: {mp}")

def cached_standardised_ledger_path(cfg: ExtendedEvidenceConfig) -> Path:
    return Path(cfg.output_root) / "tables" / "multicentre_head_landmarks_standardised_cache.csv"

def cached_raw_ledger_path(cfg: ExtendedEvidenceConfig) -> Path:
    return Path(cfg.output_root) / "tables" / "multicentre_head_landmarks_combined_raw.csv"

def validate_cached_ledger(df: pd.DataFrame, max_rows: int = 25) -> Tuple[bool, str]:
    if df is None or len(df) == 0:
        return False, "empty cache"
    required = {"image_path", "domain", "split", "subject_id", "true_source_domain", "declared_domain", "ref_bpd_px", "ref_ofd_px", "ref_bpd_mm", "ref_hc_mm", "ledger_schema_version"}
    missing = sorted(required - set(df.columns))
    if missing:
        return False, f"not a v11 true-source cache; missing required columns: {missing}"
    expected_schema = getattr(CFG, "ledger_schema_version", "v11_true_image_source_domain")
    schemas = set(df["ledger_schema_version"].dropna().astype(str).unique())
    if expected_schema not in schemas:
        return False, f"cache schema mismatch: found={sorted(schemas)} expected={expected_schema}"
    if "M-C" in set(df["domain"].astype(str)):
        return False, "cache still contains M-C as an independent domain"
    sample = df.head(max_rows)
    if "image_exists" in df.columns and bool(sample["image_exists"].all()):
        return True, "v11 cache has image_exists=True for validation sample"
    exists = sample["image_path"].apply(lambda p: Path(str(p)).exists())
    if bool(exists.all()):
        return True, "v11 cache image paths exist for validation sample"
    return False, f"cache image paths missing in validation sample: {int((~exists).sum())}/{len(sample)}"

def load_cached_standardised_ledger(cfg: ExtendedEvidenceConfig) -> Optional[pd.DataFrame]:
    if not cfg.reuse_prepared_external_ledger or cfg.force_rebuild_external_ledger:
        return None
    cache_path = cached_standardised_ledger_path(cfg)
    # Use only the raw standardised cache by default.  Post-split working ledgers can be
    # stale after domain-policy changes and should not silently become the source cache.
    candidates = [cache_path]
    if getattr(cfg, "reuse_postsplit_working_ledger", False):
        candidates.append(Path(cfg.output_root) / "tables" / "exp1_external_annotations_subject_exclusive.csv")
    for cp in candidates:
        if cp.exists() and cp.stat().st_size > 0:
            try:
                df = pd.read_csv(cp)
                ok, msg = validate_cached_ledger(df, cfg.max_cache_validation_rows)
                print(f"[cache] checked {cp}: {msg}")
                if ok:
                    print(f"[cache] using prepared standardised ledger: {cp} rows={len(df)}")
                    return df
            except Exception as e:
                print(f"[cache] failed reading {cp}: {e}")
    return None

def save_standardised_ledger_cache(df: pd.DataFrame, cfg: ExtendedEvidenceConfig) -> None:
    cp = cached_standardised_ledger_path(cfg)
    cp.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(cp, index=False)
    print(f"[cache] saved standardised ledger: {cp} rows={len(df)}")

def ensure_multicentre_dataset(cfg: ExtendedEvidenceConfig) -> Path:
    """Ensure multicentre assets are available, with idempotent rerun behaviour.

    The function first audits the current root.  If Head CSVs and images already
    exist, it skips all network calls.  This enables Run All after a successful
    first run without re-downloading the data.
    """
    root = Path(cfg.multicentre_root)
    if not looks_like_multicentre_root(root):
        # Install into DockerRoot data area when possible.
        base = Path(cfg.atlas_repro_root) / "data" if Path(cfg.atlas_repro_root).exists() else root.parent
        candidate = base / "Multicentre-Fetal-Biometry"
        # If the configured path contains data assets, keep it. Otherwise use candidate.
        st_current = multicentre_asset_status(root)
        st_candidate = multicentre_asset_status(candidate)
        if st_current["has_head_csv"] or st_current["has_images"]:
            pass
        elif st_candidate["has_head_csv"] or st_candidate["has_images"] or looks_like_multicentre_root(candidate):
            root = candidate
            cfg.multicentre_root = root
        else:
            root = candidate
            cfg.multicentre_root = root

    status0 = multicentre_asset_status(root)
    print("[data] multicentre asset status before network:")
    print(json.dumps(status0, indent=2)[:4000])

    if cfg.reuse_existing_multicentre_data and status0["has_head_csv"] and status0["has_images"] and not cfg.force_redownload_multicentre:
        print("[data] Existing Head CSVs and images found; clone/download/extract skipped.")
        write_cache_manifest(cfg, status0, {"network_used": False, "reason": "existing_head_csvs_and_images"})
        return root

    if cfg.auto_download_multicentre:
        clone_or_download_repo(cfg.multicentre_repo_url, cfg.multicentre_repo_zip_url, root)

    # Re-audit after repo clone before attempting large data downloads.
    status1 = multicentre_asset_status(root)
    if cfg.reuse_existing_multicentre_data and status1["has_head_csv"] and status1["has_images"] and not cfg.force_redownload_multicentre:
        print("[data] Assets found after repo audit; data archive download skipped.")
        write_cache_manifest(cfg, status1, {"network_used": False, "reason": "assets_found_after_repo_audit"})
        return root

    has_head_csv = status1["has_head_csv"]
    has_images = status1["has_images"]
    if cfg.auto_download_multicentre and cfg.download_data_archives and (cfg.force_redownload_multicentre or not has_head_csv or not has_images):
        if not cfg.allow_network_downloads:
            print("[data] network downloads disabled. Existing assets are incomplete; downstream cells will show the audit.")
            write_cache_manifest(cfg, status1, {"network_used": False, "reason": "network_disabled_assets_incomplete"})
            return root
        files = figshare_file_manifest_from_doi(cfg.multicentre_data_doi)
        if not files:
            print("[data] Could not auto-resolve data files from DOI. Manual download may be required:", cfg.multicentre_data_doi)
        data_dir = root / "data_downloads"
        data_dir.mkdir(parents=True, exist_ok=True)
        for f in files:
            name = f.get("name") or f.get("filename") or f"file_{f.get('id','unknown')}"
            url = f.get("download_url")
            if not url:
                continue
            dest = data_dir / name
            try:
                download_file(url, dest, timeout=cfg.download_timeout_sec, force=cfg.force_redownload_multicentre)
                if cfg.auto_extract_archives and any(str(dest).lower().endswith(suf) for suf in ARCHIVE_SUFFIXES):
                    # Extraction is idempotent enough for reruns because existing files are overwritten by shutil only if archive says so.
                    # To avoid repeated extraction, skip when root/data already contains Head CSVs and images.
                    st_pre_extract = multicentre_asset_status(root)
                    if cfg.reuse_existing_multicentre_data and st_pre_extract["has_head_csv"] and st_pre_extract["has_images"] and not cfg.force_redownload_multicentre:
                        print(f"[extract] assets already present; skipping extraction of {dest.name}")
                    else:
                        extract_archive(dest, root / "data")
            except Exception as e:
                print(f"[data] Failed to download/extract {name}: {e}")

    status2 = multicentre_asset_status(root)
    write_cache_manifest(cfg, status2, {"network_used": True, "reason": "download_path_checked"})
    return root

def infer_domain_from_path(csv_path: Path) -> str:
    parts = [p.upper().replace('_','-') for p in csv_path.parts]
    if "FP" in parts: return "FP"
    if "HC18" in parts: return "HC18"
    if "UCL" in parts: return "UCL"
    if "MULTI-CENTRE" in parts or "MULTICENTRE" in parts or "M-C" in parts: return "M-C"
    return "unknown"

def find_head_annotation_csvs(root: Path) -> List[Path]:
    root = Path(root)
    pats = ["Head.csv", "Head_Train.csv", "Head_Test.csv", "*Head*.csv"]
    hits = []
    for pat in pats:
        hits.extend(root.glob(f"**/{pat}"))
    # Keep only CSVs under annotations if possible; fallback to all Head CSVs.
    ann_hits = [p for p in hits if "annotation" in str(p).lower()]
    hits = ann_hits or hits
    return sorted(set(hits))

_IMAGE_INDEX_CACHE: Dict[str, Dict[str, List[Path]]] = {}
def build_image_index(root: Path) -> Dict[str, List[Path]]:
    root = Path(root)
    key = str(root.resolve()) if root.exists() else str(root)
    if key in _IMAGE_INDEX_CACHE:
        return _IMAGE_INDEX_CACHE[key]
    idx: Dict[str, List[Path]] = defaultdict(list)
    for base in [root, root / "data"]:
        if base.exists():
            for p in progress_iter(base.glob("**/*"), desc=f"index images under {base.name}", stage="DATA"):
                if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES:
                    idx[p.name].append(p)
    _IMAGE_INDEX_CACHE[key] = idx
    return idx

def resolve_dataset_image(root: Path, rel: Any, domain: str = "unknown", anatomy: str = "Head") -> Path:
    root = Path(root)
    rels = str(rel)
    p = Path(rels)
    if p.is_absolute() and p.exists():
        return p
    domain_dir = {"M-C": "MULTI-CENTRE", "MC": "MULTI-CENTRE", "MULTICENTRE": "MULTI-CENTRE"}.get(domain, domain)
    candidates = [
        root / rels,
        root / "data" / rels,
        root / domain_dir / "data" / anatomy / p.name,
        root / "data" / domain_dir / "data" / anatomy / p.name,
        root / "data" / domain_dir / anatomy / p.name,
        root / "data" / "images" / domain_dir / anatomy / p.name,
        root / "data" / "images" / domain_dir / "Head" / p.name,
    ]
    for c in candidates:
        if c.exists():
            return c
    idx = build_image_index(root)
    matches = idx.get(p.name, [])
    if matches:
        # prefer domain/anatomy paths when filename is ambiguous
        dom = str(domain_dir).lower(); anat = str(anatomy).lower()
        matches = sorted(matches, key=lambda q: (0 if dom in str(q).lower() and anat in str(q).lower() else 1, len(str(q))))
        return matches[0]
    return candidates[0]

def combine_head_csvs_to_prepared_raw(root: Path, output_csv: Path, force: bool = False) -> Optional[Path]:
    output_csv = Path(output_csv)
    if output_csv.exists() and output_csv.stat().st_size > 0 and not force:
        print(f"[annotations] combined raw cache exists, skipping rebuild: {output_csv}")
        return output_csv
    csvs = find_head_annotation_csvs(root)
    if not csvs:
        print("[annotations] No Head*.csv files found under", root)
        return None
    frames = []
    for csv in progress_iter(csvs, desc="read Head CSVs", total=len(csvs), stage="ANN"):
        try:
            df = pd.read_csv(csv)
            if len(df) == 0:
                continue
            df["_source_csv"] = str(csv)
            df["_domain_from_path"] = infer_domain_from_path(csv)
            # If the CSV itself is a Train/Test file and Split absent/empty, infer split.
            if "Split" not in df.columns or df.get("Split", pd.Series(dtype=str)).astype(str).str.lower().isin(["nan", "none", "", "unknown"]).all():
                if "train" in csv.name.lower(): df["Split"] = "Train"
                elif "test" in csv.name.lower(): df["Split"] = "Test"
            frames.append(df)
        except Exception as e:
            print(f"[annotations] failed reading {csv}: {e}")
    if not frames:
        return None
    raw = pd.concat(frames, ignore_index=True, sort=False)
    # Drop exact duplicates caused by reading Head.csv plus Head_Train/Test.csv.
    subset = [c for c in ["_domain_from_path", "image_name", "SubjectID", "bpd_1_x", "bpd_1_y", "bpd_2_x", "bpd_2_y", "ofd_1_x", "ofd_1_y", "ofd_2_x", "ofd_2_y"] if c in raw.columns]
    if subset:
        raw = raw.drop_duplicates(subset=subset).reset_index(drop=True)
    output_csv = Path(output_csv); output_csv.parent.mkdir(parents=True, exist_ok=True)
    raw.to_csv(output_csv, index=False)
    print(f"[annotations] combined {len(csvs)} Head CSVs -> {output_csv} rows={len(raw)}")
    return output_csv


# ------------------------------------------------------------
# v11 domain policy: true image-source domain, M-C aggregate exclusion, HC18 source anchor only
# ------------------------------------------------------------
def normalise_domain_label(x: Any) -> str:
    return canonical_domain_token(x)

def _domain_source_priority(row: pd.Series) -> int:
    """Lower is better. Prefer true-source/dedicated rows over aggregate M-C rows."""
    declared = canonical_domain_token(row.get("declared_domain", row.get("domain", "unknown")))
    true_source = canonical_domain_token(row.get("true_source_domain", row.get("domain", "unknown")))
    source_csv = str(row.get("source_csv", row.get("_source_csv", ""))).upper().replace("_", "-")
    if declared == true_source and true_source in {"FP", "HC18", "UCL"}:
        return 0
    if true_source in {"FP", "HC18", "UCL"} and "MULTI" not in source_csv and "M-C" not in source_csv:
        return 1
    if true_source in {"FP", "HC18", "UCL"}:
        return 2
    if true_source == "M-C_AGGREGATE":
        return 9
    return 8

def enforce_v11_true_source_domain_ledger(df: pd.DataFrame, cfg: ExtendedEvidenceConfig = CFG) -> pd.DataFrame:
    """Repair a standardised/all-domain ledger into the v11 evidence ledger.

    - `domain` is overwritten with the true image-source domain.
    - M-C/multicentre rows are treated as aggregate containers only.
    - Duplicate aggregate rows are removed by resolved image path.
    - UCL missing subjects are derived from filename prefixes.
    """
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.copy()
    if "declared_domain" not in out.columns:
        out["declared_domain"] = out.get("domain", "unknown").map(canonical_domain_token)
    else:
        out["declared_domain"] = out["declared_domain"].map(canonical_domain_token)
    if "source_csv" not in out.columns:
        out["source_csv"] = out.get("_source_csv", "")
    out["true_source_domain"] = [infer_true_image_source_domain(p, source_csv=sc, declared_domain=dd)
                                  for p, sc, dd in zip(out["image_path"], out.get("source_csv", pd.Series([""]*len(out))), out["declared_domain"])]
    out["true_source_domain"] = out["true_source_domain"].map(canonical_domain_token)
    repaired_subjects = []
    subject_vals = out["subject_id"] if "subject_id" in out.columns else pd.Series([np.nan] * len(out), index=out.index)
    for idx, (raw_subj, path, dom) in enumerate(zip(subject_vals, out["image_path"], out["true_source_domain"])):
        sid, src = derive_subject_id(raw_subj, path, dom)
        repaired_subjects.append((sid, src))
    out["subject_id"] = [x[0] for x in repaired_subjects]
    if "subject_id_source" not in out.columns:
        out["subject_id_source"] = [x[1] for x in repaired_subjects]
    else:
        # Preserve annotation provenance unless subject was missing/unknown.
        out["subject_id_source"] = [old if not _is_missing_subject(old) and src == "annotation_subject" else src
                                    for old, (_, src) in zip(out["subject_id_source"], repaired_subjects)]
    out["domain"] = out["true_source_domain"]
    out["ledger_schema_version"] = getattr(cfg, "ledger_schema_version", "v11_true_image_source_domain")
    out["domain_assignment_policy"] = "domain=true_image_source; M-C=aggregate_container_not_domain"
    out["_domain_source_priority"] = out.apply(_domain_source_priority, axis=1)

    # Audit all duplicate resolved images before dropping aggregate duplicates.
    dup_mask = out.duplicated(subset=["image_path"], keep=False)
    dup_audit = out.loc[dup_mask].sort_values(["image_path", "_domain_source_priority", "declared_domain"]).copy()
    dup_path = Path(cfg.output_root) / "tables" / "exp1_v11_duplicate_image_resolution_audit.csv"
    dup_path.parent.mkdir(parents=True, exist_ok=True)
    if len(dup_audit):
        dup_audit.to_csv(dup_path, index=False)
        log_event("DOMAIN", "duplicate image rows audited before v11 dedupe", duplicate_rows=len(dup_audit), audit=dup_path)
    else:
        pd.DataFrame(columns=list(out.columns)).to_csv(dup_path, index=False)

    if getattr(cfg, "deduplicate_true_source_image_rows", True):
        before = len(out)
        out = out.sort_values(["image_path", "_domain_source_priority", "declared_domain", "source_csv"], kind="stable")
        out = out.drop_duplicates(subset=["image_path"], keep="first").reset_index(drop=True)
        log_event("DOMAIN", "v11 true-source dedupe complete", before=before, after=len(out), dropped=before-len(out))
    out = out.drop(columns=["_domain_source_priority"], errors="ignore")
    # Keep M-C aggregate rows in all-domain audit if any unresolved rows remain, but they will be excluded by policy.
    return out

def apply_primary_external_domain_policy(df: pd.DataFrame, cfg: ExtendedEvidenceConfig = CFG) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Return (primary_df, all_domain_df, domain_policy_audit) under v11 policy.

    Primary external evidence is limited to true-source FP and UCL by default. HC18
    is retained only as source-domain/base-model anchor evidence. M-C is never an
    independent evidence domain because it is an aggregate annotation container.
    """
    if df is None or len(df) == 0:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    all_df = enforce_v11_true_source_domain_ledger(df, cfg).copy()
    all_df["domain"] = all_df["domain"].map(normalise_domain_label)
    source = normalise_domain_label(getattr(cfg, "training_source_domain", "HC18"))
    primary = {normalise_domain_label(d) for d in getattr(cfg, "primary_external_domains", ("FP", "UCL"))}
    include_source_anchor = bool(getattr(cfg, "include_training_source_domain_as_anchor", False))
    allowed = set(primary)
    if include_source_anchor:
        allowed.add(source)

    all_df["domain_role"] = "nonprimary_excluded"
    all_df.loc[all_df["domain"].isin(primary), "domain_role"] = "primary_external"
    all_df.loc[all_df["domain"].eq(source), "domain_role"] = "source_domain_anchor" if include_source_anchor else "source_domain_excluded"
    all_df.loc[all_df["domain"].eq("M-C_AGGREGATE"), "domain_role"] = "aggregate_container_excluded"
    all_df["included_in_primary_external_evidence"] = all_df["domain"].isin(allowed)

    # Hard safety: M-C aggregate must never be included as primary evidence.
    if bool(all_df.loc[all_df["domain"].eq("M-C_AGGREGATE"), "included_in_primary_external_evidence"].any()):
        raise RuntimeError("M-C aggregate rows were included in primary evidence. This violates v11 policy.")
    if not include_source_anchor and bool(all_df.loc[all_df["domain"].eq(source), "included_in_primary_external_evidence"].any()):
        raise RuntimeError(f"Source-domain rows ({source}) were included despite include_training_source_domain_as_anchor=False")

    audit = all_df.groupby(["domain", "declared_domain", "domain_role", "included_in_primary_external_evidence"], dropna=False).agg(
        rows=("image_path", "size"),
        subjects=("subject_id", "nunique"),
        images_available=("image_exists", "sum") if "image_exists" in all_df.columns else ("image_path", "size"),
    ).reset_index()
    audit_path = Path(cfg.output_root) / "tables" / "exp1_domain_policy_audit_all_domains.csv"
    excluded_path = Path(cfg.output_root) / "tables" / "exp1_excluded_source_or_nonprimary_domains.csv"
    all_path = Path(cfg.output_root) / "tables" / "exp1_external_annotations_all_domains_unfiltered.csv"
    audit_path.parent.mkdir(parents=True, exist_ok=True)
    audit.to_csv(audit_path, index=False)
    all_df.to_csv(all_path, index=False)
    excluded = all_df.loc[~all_df["included_in_primary_external_evidence"]].copy()
    excluded.to_csv(excluded_path, index=False)
    primary_df = all_df.loc[all_df["included_in_primary_external_evidence"]].copy().reset_index(drop=True)
    log_event("DOMAIN", "applied v11 primary external domain policy",
              raw_rows=len(all_df), primary_rows=len(primary_df),
              primary_domains=sorted(primary), source_domain=source,
              include_source_anchor=include_source_anchor)
    if len(excluded):
        excl_summary = excluded.groupby(["domain", "declared_domain", "domain_role"]).size().reset_index(name="rows")
        show_df(excl_summary, "excluded/non-primary domain rows", max_rows=30)
    show_df(audit, "v11 true-source domain policy audit", max_rows=30)
    return primary_df, all_df, audit

# Ensure repo/data and prepare or load Head annotations.
# v6 rerun/progress policy:
#   1) Use cached standardised ledger when present and valid.
#   2) Otherwise reuse existing data assets and rebuild the ledger locally.
#   3) Download/clone only when assets are missing and CFG.allow_network_downloads=True.
ext_df = load_cached_standardised_ledger(CFG)
if ext_df is not None:
    ext_df = enforce_v11_true_source_domain_ledger(ext_df, CFG)
if ext_df is None:
    if CFG.auto_download_multicentre:
        CFG.multicentre_root = ensure_multicentre_dataset(CFG)
    prepared_raw_csv = cached_raw_ledger_path(CFG)
    if (not CFG.multicentre_annotations.exists()) or CFG.force_rebuild_external_ledger:
        maybe = combine_head_csvs_to_prepared_raw(CFG.multicentre_root, prepared_raw_csv, force=CFG.force_rebuild_external_ledger)
        if maybe is not None:
            CFG.multicentre_annotations = maybe
    # Load if available; otherwise create an empty placeholder so downstream cells do not crash.
    if CFG.multicentre_annotations.exists():
        ext_df = standardise_multicentre_annotations(CFG.multicentre_annotations, CFG.multicentre_root)
        ext_df = enforce_v11_true_source_domain_ledger(ext_df, CFG)
        print("Loaded/rebuilt v11 true-source external annotations:", ext_df.shape)
        save_standardised_ledger_cache(ext_df, CFG)
    else:
        ext_df = pd.DataFrame()
        print("External annotation file not found. Set CFG.multicentre_annotations and re-run this cell.")
else:
    print("Loaded v11 true-source external annotations from cache:", ext_df.shape)

# Preserve the all-domain ledger for audit, then apply the primary external-domain policy.
ext_df_all_domains = ext_df.copy() if len(ext_df) else pd.DataFrame()
if len(ext_df_all_domains):
    ext_df, ext_df_all_domains, domain_policy_audit = apply_primary_external_domain_policy(ext_df_all_domains, CFG)
    print("Primary external cohort after domain policy:")
    print(ext_df.groupby(["domain", "split"]).size().unstack(fill_value=0) if len(ext_df) else "No rows after domain policy")
    show_df(ext_df, "primary external annotation ledger preview", max_rows=5)
else:
    domain_policy_audit = pd.DataFrame()


[data] multicentre asset status before network:
{
  "root": "/content/atlas_fn_workspace/data/Multicentre-Fetal-Biometry",
  "root_exists": false,
  "looks_like_repo_or_data_root": false,
  "head_csv_count": 0,
  "has_head_csv": false,
  "sample_head_csvs": [],
  "sample_image_count_capped_at_25": 0,
  "has_images": false,
  "sample_images": [],
  "downloaded_archive_count": 0,
  "sample_archives": []
}
[12:11:37 +    8.9s] [CMD] running | cmd=git clone --depth 1 https://github.com/surgical-vision/Multicentre-Fetal-Biometry.git /content/atlas_fn_workspace/data/Multicentre-Fetal-Biometry, cwd=/content
[12:11:38 +   10.1s] [CMD] finished | returncode=0, elapsed_sec=1.21


index images under Multicentre-Fetal-Biometry: 0it [00:00, ?it/s]

index images under data: 0it [00:00, ?it/s]

[12:11:39 +   11.0s] [DOWNLOAD] start | url=https://ndownloader.figshare.com/files/66050144, dest=/content/atlas_fn_workspace/data/Multicentre-Fetal-Biometry/data_downloads/Multicentre-Fetal-Biometry-2026.zip


download Multicentre-Fetal-Biometry-2026.zip:   0%|          | 0.00/1.59G [00:00<?, ?B/s]

...

,domain,declared_domain,domain_role,rows
0,HC18,HC18,source_domain_excluded,999


[12:13:32 +  123.8s] [DATA] v11 true-source domain policy audit | rows=3, columns=7


,domain,declared_domain,domain_role,included_in_primary_external_evidence,rows,subjects,images_available
0,FP,FP,primary_external,True,1637,909,1637
1,HC18,HC18,source_domain_excluded,False,999,806,999
2,UCL,UCL,primary_external,True,159,47,159


Primary external cohort after domain policy:
split   test  train
domain             
FP       880    757
UCL       49    110
[12:13:32 +  123.8s] [DATA] primary external annotation ledger preview | rows=1796, columns=36


,row_id,image_path,image_exists,img_w,img_h,subject_id,subject_id_source,declared_domain,true_source_domain,domain,...,ref_x1,ref_y1,ref_x2,ref_y2,source_csv,column_map_json,ledger_schema_version,domain_assignment_policy,domain_role,included_in_primary_external_evidence
0,217,/content/atlas_fn_workspace/data/Multicentre-F...,True,640,392,168,annotation_subject,FP,FP,FP,...,115.0,15.0,524.0,353.0,/content/atlas_fn_workspace/data/Multicentre-F...,"{""bpd_x1"": ""bpd_1_x"", ""bpd_x2"": ""bpd_2_x"", ""bp...",v12_true_source_shm_safe_complete_ties,domain=true_image_source; M-C=aggregate_contai...,primary_external,True
1,549,/content/atlas_fn_workspace/data/Multicentre-F...,True,640,392,168,annotation_subject,FP,FP,FP,...,153.0,20.0,488.0,278.0,/content/atlas_fn_workspace/data/Multicentre-F...,"{""bpd_x1"": ""bpd_1_x"", ""bpd_x2"": ""bpd_2_x"", ""bp...",v12_true_source_shm_safe_complete_ties,domain=true_image_source; M-C=aggregate_contai...,primary_external,True
2,361,/content/atlas_fn_workspace/data/Multicentre-F...,True,640,392,188,annotation_subject,FP,FP,FP,...,138.0,73.0,515.0,362.0,/content/atlas_fn_workspace/data/Multicentre-F...,"{""bpd_x1"": ""bpd_1_x"", ""bpd_x2"": ""bpd_2_x"", ""bp...",v12_true_source_shm_safe_complete_ties,domain=true_image_source; M-C=aggregate_contai...,primary_external,True
3,969,/content/atlas_fn_workspace/data/Multicentre-F...,True,961,663,216,annotation_subject,FP,FP,FP,...,194.0,92.0,752.0,564.0,/content/atlas_fn_workspace/data/Multicentre-F...,"{""bpd_x1"": ""bpd_1_x"", ""bpd_x2"": ""bpd_2_x"", ""bp...",v12_true_source_shm_safe_complete_ties,domain=true_image_source; M-C=aggregate_contai...,primary_external,True
4,1156,/content/atlas_fn_workspace/data/Multicentre-F...,True,961,663,216,annotation_subject,FP,FP,FP,...,122.0,142.0,573.0,523.0,/content/atlas_fn_workspace/data/Multicentre-F...,"{""bpd_x1"": ""bpd_1_x"", ""bpd_x2"": ""bpd_2_x"", ""bp...",v12_true_source_shm_safe_complete_ties,domain=true_image_source; M-C=aggregate_contai...,primary_external,True


## 3. Patient exclusivity, denominator audit and split handling

This mirrors the Master Evidence logic: the notebook audits denominators before any metric is reported. If the external dataset does not provide a split, this cell creates a deterministic subject-exclusive split within each domain. If official splits are provided, it verifies subject disjointness across split labels.


In [8]:
# ============================================================
# 4. Split handling and denominator audit
# ============================================================
# v3 fix: the multicentre repository can expose official Train/Test CSVs that are
# not subject-exclusive within a domain, or can contain duplicated subjects after
# combining Head.csv + Head_Train.csv + Head_Test.csv.  We therefore audit the
# provided split first, save the leakage evidence, and then automatically repair
# the working split into a subject-exclusive split before any downstream metric,
# export or TIES experiment is run.

_SPLIT_MAP = {
    "train": "train", "training": "train", "tr": "train",
    "val": "val", "valid": "val", "validation": "val", "dev": "val",
    "test": "test", "testing": "test", "te": "test", "holdout": "test", "heldout": "test",
    "unknown": "unknown", "nan": "unknown", "none": "unknown", "": "unknown",
}

def normalise_split_series(s: pd.Series) -> pd.Series:
    return (s.fillna("unknown").astype(str).str.strip().str.lower().map(lambda x: _SPLIT_MAP.get(x, x)))

def normalise_subject_series(s: pd.Series) -> pd.Series:
    # CSV readers often turn integer subject ids into floats (e.g., 683.0).
    # The split audit should treat 683 and 683.0 as the same subject.
    out = s.fillna("unknown_subject").astype(str).str.strip()
    out = out.str.replace(r"\.0$", "", regex=True)
    return out

def has_usable_split(df: pd.DataFrame, split_col: str = "split") -> bool:
    if split_col not in df.columns:
        return False
    vals = set(normalise_split_series(df[split_col]).unique())
    return bool(vals - {"unknown"})

def deterministic_subject_split(df: pd.DataFrame, split_col: str = "split", seed: int = SEED,
                                train_frac: float = 0.70, val_frac: float = 0.10,
                                group_col: str = "domain") -> pd.DataFrame:
    """Create deterministic subject-exclusive train/val/test splits within each domain."""
    df = df.copy()
    df["subject_key"] = normalise_subject_series(df["subject_id"])
    rng = np.random.default_rng(seed)
    df[split_col] = "unknown"
    for domain, sub in df.groupby(group_col, dropna=False):
        subjects = np.array(sorted(sub["subject_key"].unique()))
        rng.shuffle(subjects)
        n = len(subjects)
        n_train = int(round(n * train_frac))
        n_val = int(round(n * val_frac))
        train_s = set(subjects[:n_train])
        val_s = set(subjects[n_train:n_train+n_val])
        test_s = set(subjects[n_train+n_val:])
        df.loc[(df[group_col] == domain) & df["subject_key"].isin(train_s), split_col] = "train"
        df.loc[(df[group_col] == domain) & df["subject_key"].isin(val_s), split_col] = "val"
        df.loc[(df[group_col] == domain) & df["subject_key"].isin(test_s), split_col] = "test"
    df["split_repair_action"] = "deterministic_subject_split"
    return df

def audit_subject_disjoint(df: pd.DataFrame, group_cols=("domain",), split_col: str = "split",
                           subject_col: str = "subject_key") -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Return a domain-level audit and row-level overlap summary without raising."""
    rows, overlap_rows = [], []
    if df.empty:
        return pd.DataFrame(rows), pd.DataFrame(overlap_rows)
    work = df.copy()
    if subject_col not in work.columns:
        work[subject_col] = normalise_subject_series(work["subject_id"])
    work[split_col] = normalise_split_series(work[split_col])
    groups = list(work.groupby(list(group_cols), dropna=False))
    for group_key, sub in progress_iter(groups, desc="assert subject disjoint", total=len(groups), stage="SPLIT"):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        group_dict = {col: val for col, val in zip(group_cols, group_key)}
        split_subjects = {sp: set(g[subject_col].astype(str)) for sp, g in sub.groupby(split_col)}
        splits = sorted(split_subjects)
        overlap_pairs = []
        overlap_subjects = set()
        for i, a in enumerate(splits):
            for b in splits[i+1:]:
                inter = split_subjects[a] & split_subjects[b]
                if inter:
                    overlap_pairs.append((a, b, len(inter)))
                    overlap_subjects |= inter
                    preview = ",".join(sorted(list(inter))[:12])
                    overlap_rows.append({**group_dict, "split_a": a, "split_b": b,
                                         "n_overlap_subjects": len(inter),
                                         "subject_preview": preview})
        rows.append({**group_dict,
                     "patient_exclusive": len(overlap_pairs) == 0,
                     "n_splits": len(splits),
                     "n_subjects": int(sub[subject_col].nunique()),
                     "n_overlap_subjects": int(len(overlap_subjects)),
                     "overlap_summary": overlap_pairs})
    return pd.DataFrame(rows), pd.DataFrame(overlap_rows)

def repair_leaky_subject_splits(df: pd.DataFrame, split_col: str = "split",
                                group_col: str = "domain", strategy: str = "majority") -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Repair official row-level splits into subject-exclusive splits.

    strategy='majority' assigns all rows for an overlapping subject to the split that
    contains most rows for that subject, with test > val > train priority for ties.
    This preserves the official split as much as possible while eliminating leakage.
    """
    df = df.copy()
    df["subject_key"] = normalise_subject_series(df["subject_id"])
    df["original_split"] = normalise_split_series(df[split_col]) if split_col in df.columns else "unknown"
    df[split_col] = df["original_split"]
    priority = {"test": 3, "val": 2, "train": 1, "unknown": 0}
    repairs = []
    for (domain, subject), sub in df.groupby([group_col, "subject_key"], dropna=False):
        splits = list(sub["original_split"].astype(str))
        known = [sp for sp in splits if sp != "unknown"]
        if not known:
            chosen = "unknown"
        elif strategy == "test_priority":
            chosen = sorted(set(known), key=lambda sp: priority.get(sp, 0), reverse=True)[0]
        else:
            counts = pd.Series(known).value_counts()
            best_count = counts.max()
            candidates = [sp for sp, n in counts.items() if n == best_count]
            chosen = sorted(candidates, key=lambda sp: priority.get(sp, 0), reverse=True)[0]
        if len(set(known)) > 1 or "unknown" in set(splits):
            repairs.append({"domain": domain, "subject_key": subject,
                            "original_splits": ",".join(sorted(set(splits))),
                            "assigned_split": chosen, "n_rows": int(len(sub))})
        df.loc[(df[group_col] == domain) & (df["subject_key"] == subject), split_col] = chosen
    repair_df = pd.DataFrame(repairs)
    # If unknown subjects remain, assign them with deterministic subject split only for those rows.
    if (df[split_col] == "unknown").any():
        unknown_mask = df[split_col] == "unknown"
        tmp = deterministic_subject_split(df.loc[unknown_mask].copy(), split_col=split_col, seed=SEED, group_col=group_col)
        df.loc[unknown_mask, split_col] = tmp[split_col].values
        df.loc[unknown_mask, "split_repair_action"] = "deterministic_unknown_subject_split"
    df["split_repair_action"] = df.get("split_repair_action", pd.Series(index=df.index, dtype=object)).fillna("official_preserved_or_majority_repaired")
    return df, repair_df

def make_subject_exclusive_split(df: pd.DataFrame, split_col: str = "split", seed: int = SEED,
                                 train_frac: float = 0.70, val_frac: float = 0.10,
                                 repair_strategy: str = "majority") -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Return working subject-exclusive dataframe plus audit tables.

    If source splits are present, they are audited. If they are already subject-exclusive,
    they are preserved. If leakage is detected, a repaired split is created and the
    overlap/repair evidence is written out for transparency.
    """
    df = df.copy()
    if "subject_id" not in df.columns:
        raise ValueError("Expected standardised external dataframe with subject_id column.")
    df["subject_key"] = normalise_subject_series(df["subject_id"])
    if has_usable_split(df, split_col):
        df[split_col] = normalise_split_series(df[split_col])
        official_audit, official_overlaps = audit_subject_disjoint(df, group_cols=("domain",), split_col=split_col)
        if bool(official_audit["patient_exclusive"].all()):
            df["original_split"] = df[split_col]
            df["split_repair_action"] = "official_subject_exclusive_preserved"
            return df, official_audit, pd.DataFrame()
        print("[split] Official/source split is not patient-exclusive in at least one domain.")
        print("[split] Repairing working split at subject level; original split is preserved in original_split.")
        repaired, repair_df = repair_leaky_subject_splits(df, split_col=split_col, strategy=repair_strategy)
        repaired_audit, repaired_overlaps = audit_subject_disjoint(repaired, group_cols=("domain",), split_col=split_col)
        if not bool(repaired_audit["patient_exclusive"].all()):
            raise RuntimeError(f"Split repair failed: {repaired_overlaps.to_dict('records')[:5]}")
        return repaired, official_audit, repair_df
    print("[split] No usable source split found; creating deterministic subject-exclusive split within each domain.")
    repaired = deterministic_subject_split(df, split_col=split_col, seed=seed, train_frac=train_frac, val_frac=val_frac)
    repaired_audit, repaired_overlaps = audit_subject_disjoint(repaired, group_cols=("domain",), split_col=split_col)
    if not bool(repaired_audit["patient_exclusive"].all()):
        raise RuntimeError(f"Deterministic split failed: {repaired_overlaps.to_dict('records')[:5]}")
    return repaired, repaired_audit, pd.DataFrame()

def assert_subject_disjoint(df: pd.DataFrame, group_cols=("domain",), split_col: str = "split",
                            raise_on_fail: bool = True) -> pd.DataFrame:
    """Compatibility wrapper used by later cells."""
    audit, overlaps = audit_subject_disjoint(df, group_cols=group_cols, split_col=split_col)
    if raise_on_fail and len(audit) and not bool(audit["patient_exclusive"].all()):
        raise RuntimeError(f"Patient leakage detected: {overlaps.to_dict('records')[:10]}")
    return audit


def ensure_validation_split(df: pd.DataFrame, split_col: str = "split", group_col: str = "domain",
                            seed: int = SEED, val_frac_from_train: float = 0.12,
                            min_train_subjects: int = 8) -> pd.DataFrame:
    """Create a small subject-exclusive validation split from train when a domain has no val rows.

    Many public Train/Test annotation packages expose only train and test. YOLO training
    expects a validation folder, so this function moves a deterministic subset of train
    subjects to val without touching test subjects and without creating leakage.
    """
    df = df.copy()
    if "subject_key" not in df.columns:
        df["subject_key"] = normalise_subject_series(df["subject_id"])
    rng = np.random.default_rng(seed)
    actions = []
    for domain, sub in df.groupby(group_col, dropna=False):
        splits = set(sub[split_col].astype(str))
        if "val" in splits:
            continue
        train_subjects = np.array(sorted(sub.loc[sub[split_col] == "train", "subject_key"].unique()))
        if len(train_subjects) < min_train_subjects:
            continue
        rng.shuffle(train_subjects)
        n_val = max(1, int(round(len(train_subjects) * val_frac_from_train)))
        val_subjects = set(train_subjects[:n_val])
        mask = (df[group_col] == domain) & (df["subject_key"].isin(val_subjects))
        df.loc[mask, split_col] = "val"
        df.loc[mask, "split_repair_action"] = df.loc[mask, "split_repair_action"].astype(str) + "+val_from_train"
        actions.append({"domain": domain, "n_val_subjects_created": len(val_subjects),
                        "n_val_rows_created": int(mask.sum())})
    if actions:
        action_df = pd.DataFrame(actions)
        action_df.to_csv(CFG.output_root / "tables" / "exp1_validation_split_creation_log.csv", index=False)
        print("[split] Created validation split from train subjects for YOLO training:")
        show_df(action_df, "validation split creation actions")
    return df

def ensure_test_split_for_primary_domains(df: pd.DataFrame, split_col: str = "split", group_col: str = "domain",
                                          seed: int = SEED, train_frac: float = 0.70, val_frac: float = 0.10) -> pd.DataFrame:
    """Create a deterministic subject-exclusive test split for primary external domains lacking one.

    UCL in the public benchmark may appear as train-only after aggregate CSV repair. For
    external validation, a domain with no test subjects cannot support an Experiment 1
    test-set claim. This function preserves `original_split` and creates train/val/test
    subject splits only for affected primary domains.
    """
    df = df.copy()
    if "subject_key" not in df.columns:
        df["subject_key"] = normalise_subject_series(df["subject_id"])
    actions = []
    primary = {normalise_domain_label(d) for d in getattr(CFG, "primary_external_domains", ("FP", "UCL"))}
    for domain, sub in df.groupby(group_col, dropna=False):
        nd = normalise_domain_label(domain)
        if nd not in primary:
            continue
        split_counts = normalise_split_series(sub[split_col]).value_counts().to_dict()
        has_test = split_counts.get("test", 0) > 0
        if has_test or not getattr(CFG, "force_subject_split_when_no_test", True):
            continue
        tmp = deterministic_subject_split(sub.copy(), split_col=split_col, seed=seed, train_frac=train_frac, val_frac=val_frac, group_col=group_col)
        df.loc[sub.index, split_col] = tmp[split_col].values
        df.loc[sub.index, "split_repair_action"] = df.loc[sub.index, "split_repair_action"].astype(str) + "+v11_test_created_no_official_test"
        actions.append({"domain": nd, "reason": "no_official_test_split", "rows": int(len(sub)),
                        "subjects": int(sub["subject_key"].nunique()),
                        "new_split_counts": tmp[split_col].value_counts().to_dict()})
    if actions:
        action_df = pd.DataFrame(actions)
        action_df.to_csv(CFG.output_root / "tables" / "exp1_v11_test_split_creation_log.csv", index=False)
        print("[split] v11 created deterministic test split for primary domains lacking official test rows:")
        show_df(action_df, "v11 test split creation actions", max_rows=20)
    return df

if len(ext_df):
    # ext_df has already been filtered by v9 primary external-domain policy. HC18 rows
    # remain available in exp1_external_annotations_all_domains_unfiltered.csv but are
    # excluded from primary cross-dataset validation unless explicitly enabled.
    if getattr(CFG, "exclude_training_source_from_primary", True) and not getattr(CFG, "include_training_source_domain_as_anchor", False):
        source_domain = normalise_domain_label(getattr(CFG, "training_source_domain", "HC18"))
        n_source = int((ext_df["domain"].map(normalise_domain_label) == source_domain).sum())
        if n_source:
            raise RuntimeError(f"Primary external cohort still contains source domain {source_domain}: {n_source} rows")
    ext_df, split_audit, split_repair = make_subject_exclusive_split(ext_df, repair_strategy="majority")
    ext_df = ensure_test_split_for_primary_domains(ext_df, split_col="split", group_col="domain")
    ext_df = ensure_validation_split(ext_df, split_col="split", group_col="domain")
    final_audit = assert_subject_disjoint(ext_df, group_cols=("domain",), split_col="split", raise_on_fail=True)
    denom = ext_df.groupby(["domain", "split"]).agg(
        frames=("image_path", "size"),
        subjects=("subject_key", "nunique"),
        images_available=("image_exists", "sum"),
        spacing_metadata=("spacing_source", lambda s: int((s == "metadata").sum())),
        spacing_fallback=("spacing_source", lambda s: int((s != "metadata").sum())),
    ).reset_index()
    denom.to_csv(CFG.output_root / "tables" / "exp1_external_denominator_audit.csv", index=False)
    split_audit.to_csv(CFG.output_root / "tables" / "exp1_source_split_audit_before_repair.csv", index=False)
    final_audit.to_csv(CFG.output_root / "tables" / "exp1_patient_exclusivity_audit.csv", index=False)
    ext_df.to_csv(CFG.output_root / "tables" / "exp1_external_annotations_subject_exclusive.csv", index=False)
    # Save the working primary-external ledger separately; do not overwrite the raw standardised cache.
    working_cache_path = CFG.output_root / "tables" / "exp1_external_annotations_subject_exclusive.csv"
    ext_df.to_csv(working_cache_path, index=False)
    write_cache_manifest(CFG, multicentre_asset_status(CFG.multicentre_root), {
        "network_used": False,
        "reason": "post_split_primary_external_working_ledger_saved",
        "rows_primary_external": int(len(ext_df)),
        "domains_primary_external": sorted(map(str, ext_df["domain"].dropna().unique())) if "domain" in ext_df.columns else [],
        "all_domain_rows_available": int(len(ext_df_all_domains)) if "ext_df_all_domains" in globals() else None,
    })
    if len(split_repair):
        split_repair.to_csv(CFG.output_root / "tables" / "exp1_split_repair_log.csv", index=False)
        print(f"[split] Repaired {len(split_repair)} overlapping/unknown subject assignments. See exp1_split_repair_log.csv")
    show_df(denom, "repaired denominator audit")
    show_df(final_audit, "final post-repair patient-exclusivity audit")
    if len(split_repair):
        show_df(split_repair, "split repair log", max_rows=20)
else:
    denom = pd.DataFrame()
    split_audit = pd.DataFrame()
    final_audit = pd.DataFrame()
    split_repair = pd.DataFrame()
    print("No external dataframe loaded yet.")


assert subject disjoint:   0%|          | 0/2 [00:00<?, ?it/s]

[split] Official/source split is not patient-exclusive in at least one domain.
[split] Repairing working split at subject level; original split is preserved in original_split.


assert subject disjoint:   0%|          | 0/2 [00:00<?, ?it/s]

[split] Created validation split from train subjects for YOLO training:
[12:13:33 +  125.1s] [DATA] validation split creation actions | rows=2, columns=3


,domain,n_val_subjects_created,n_val_rows_created
0,FP,54,95
1,UCL,4,31


...

,domain,split,frames,subjects,images_available,spacing_metadata,spacing_fallback
0,FP,test,880,460,880,880,0
1,FP,train,662,395,662,660,2
2,FP,val,95,54,95,95,0
3,UCL,test,55,15,55,35,20
4,UCL,train,73,28,73,60,13
5,UCL,val,31,4,31,31,0


[12:13:33 +  125.5s] [DATA] final post-repair patient-exclusivity audit | rows=2, columns=6


,domain,patient_exclusive,n_splits,n_subjects,n_overlap_subjects,overlap_summary
0,FP,True,3,909,0,[]
1,UCL,True,3,47,0,[]


[12:13:33 +  125.5s] [DATA] split repair log | rows=7, columns=5


,domain,subject_key,original_splits,assigned_split,n_rows
0,UCL,UCL_031,"test,train",test,3
1,UCL,UCL_032,"test,train",train,3
2,UCL,UCL_034,"test,train",test,4
3,UCL,UCL_050,"test,train",test,2
4,UCL,UCL_052,"test,train",test,3
5,UCL,UCL_053,"test,train",test,2
6,UCL,UCL_059,"test,train",test,2


## 4. YOLO export and Find fine-tuning for v12

This export creates one-class YOLO detection labels from BPD/OFD landmarks. v12 is configured to fine-tune the **Find** model on FP and UCL true-source training splits by default so that Experiment 2 can run actual real-domain TIES rather than a placeholder.

**Important ATLAS-FN boundary.** We fine-tune **Find** only. **Measure is not a trainable model**: BPD/OFD/HC are deterministic functions of the Find-stage box and pixel spacing. Measure can be externally validated and audited, but it must not be fine-tuned in the main ATLAS-FN evidence chain.

Default v12 strategy:

- `CFG.run_yolo_training=True`
- `CFG.yolo_training_strategy="both"`
- domain-specific FP and UCL specialists are trained for TIES;
- a pooled external Find model is trained for Experiment 1 sensitivity;
- HC18 remains the source-domain/base anchor and is not included in primary external validation.


In [9]:
# ============================================================
# 5. YOLO export and SHM-safe Find fine-tuning (v12)
# ============================================================
# Section 5 is intentionally defensive. In Docker/Colab environments, YOLO/PyTorch
# dataloaders can fail with bus errors or OSError [Errno 28] when /dev/shm or
# multiprocessing semaphore quotas are small. v12 uses workers=0 by default, deletes
# stale YOLO cache files before training, reuses completed fine-tuned checkpoints, and
# retries once with an even safer configuration if a dataloader/SHM error is detected.

import gc


def export_yolo_find_dataset(df: pd.DataFrame, domain: str, out_dir: Path, split_col: str = "split") -> Optional[Path]:
    # domain="ALL" creates a pooled external dataset across all primary external domains.
    if str(domain).upper() in {"ALL", "POOLED", "EXTERNAL_ALL"}:
        sub = df.copy()
        export_domain_name = "ALL_external"
    else:
        sub = df[df["domain"].astype(str) == str(domain)].copy()
        export_domain_name = str(domain)
    if sub.empty:
        print(f"[YOLO EXPORT] No rows for domain {domain}")
        return None
    out_dir = Path(out_dir) / export_domain_name.replace("/", "_").replace(" ", "_")
    images_dir = out_dir / "images"
    labels_dir = out_dir / "labels"
    for sp in ["train", "val", "test"]:
        (images_dir / sp).mkdir(parents=True, exist_ok=True)
        (labels_dir / sp).mkdir(parents=True, exist_ok=True)

    manifest = []
    log_event("YOLO", "export start", domain=export_domain_name, rows=len(sub), out_dir=out_dir)
    for _, r in progress_iter(sub.iterrows(), desc=f"YOLO export {export_domain_name}", total=len(sub), stage="YOLO"):
        sp = str(r.get(split_col, "train")).lower()
        if sp not in {"train", "val", "test"}:
            sp = "train"
        src = Path(r["image_path"])
        if not src.exists() or not finite(r["img_w"]) or not finite(r["img_h"]):
            continue
        stem = f"{r['domain']}_{r['subject_id']}_{src.stem}".replace("/", "_").replace(" ", "_")
        dst_img = images_dir / sp / (stem + src.suffix.lower())
        if not dst_img.exists():
            try:
                os.symlink(src, dst_img)
            except Exception:
                shutil.copy2(src, dst_img)
        xyxy = [r["ref_x1"], r["ref_y1"], r["ref_x2"], r["ref_y2"]]
        cx, cy, bw, bh = normalise_xywh_from_xyxy(xyxy, int(r["img_w"]), int(r["img_h"]))
        label = f"0 {cx:.8f} {cy:.8f} {bw:.8f} {bh:.8f}\n"
        (labels_dir / sp / (stem + ".txt")).write_text(label)
        manifest.append({"src": str(src), "dst": str(dst_img), "split": sp, "label": label.strip(), "row_id": r["row_id"]})

    yaml_text = {
        "path": str(out_dir),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "fetal_head"},
    }
    yaml_path = out_dir / f"find_{export_domain_name}.yaml".replace("/", "_").replace(" ", "_")
    if yaml is None:
        yaml_path.write_text(json.dumps(yaml_text, indent=2))
    else:
        yaml_path.write_text(yaml.safe_dump(yaml_text, sort_keys=False))
    pd.DataFrame(manifest).to_csv(out_dir / "export_manifest.csv", index=False)
    split_counts = pd.DataFrame(manifest)["split"].value_counts().to_dict() if manifest else {}
    log_event("YOLO", "export done", domain=export_domain_name, labelled=len(manifest), yaml=yaml_path, splits=split_counts)
    return yaml_path


def _read_yolo_yaml(data_yaml: Path) -> Dict[str, Any]:
    if yaml is not None:
        return yaml.safe_load(Path(data_yaml).read_text())
    return json.loads(Path(data_yaml).read_text())


def _count_yolo_export_images(data_yaml: Path) -> Dict[str, int]:
    meta = _read_yolo_yaml(Path(data_yaml))
    root = Path(meta["path"])
    return {sp: len(list((root / meta.get(sp, f"images/{sp}")).glob("*"))) for sp in ["train", "val", "test"]}


def _delete_yolo_cache_files(data_yaml: Path, cfg: ExtendedEvidenceConfig = CFG) -> int:
    """Delete stale Ultralytics .cache files inside the export root. These can retain old paths after reruns."""
    if not getattr(cfg, "yolo_delete_dataset_caches_before_train", True):
        return 0
    meta = _read_yolo_yaml(Path(data_yaml))
    root = Path(meta["path"])
    n = 0
    for p in root.rglob("*.cache"):
        try:
            p.unlink()
            n += 1
        except Exception as e:
            log_event("TRAIN", "cache unlink warning", path=p, error=repr(e))
    if n:
        log_event("TRAIN", "deleted stale YOLO dataset cache files", data_yaml=data_yaml, count=n)
    return n


def _runtime_storage_audit(path: Path) -> Dict[str, Any]:
    """Audit disk and /dev/shm availability; useful for diagnosing DataLoader crashes."""
    out = {}
    for label, p in {"output_root": Path(path), "tmp": Path(os.environ.get("TMPDIR", "/tmp")), "dev_shm": Path("/dev/shm")}.items():
        try:
            target = p if p.exists() else p.parent
            du = shutil.disk_usage(target)
            out[f"{label}_free_mb"] = round(du.free / (1024 * 1024), 1)
            out[f"{label}_total_mb"] = round(du.total / (1024 * 1024), 1)
        except Exception as e:
            out[f"{label}_audit_error"] = repr(e)
    return out


def _cleanup_training_runtime(data_yaml: Optional[Path] = None, run_dir: Optional[Path] = None, cfg: ExtendedEvidenceConfig = CFG):
    if data_yaml is not None:
        _delete_yolo_cache_files(Path(data_yaml), cfg)
    if run_dir is not None and getattr(cfg, "yolo_clean_incomplete_runs", True):
        best = run_dir / "weights" / "best.pt"
        last = run_dir / "weights" / "last.pt"
        if run_dir.exists() and not best.exists() and not last.exists():
            try:
                shutil.rmtree(run_dir)
                log_event("TRAIN", "removed incomplete YOLO run directory", run_dir=run_dir)
            except Exception as e:
                log_event("TRAIN", "could not remove incomplete YOLO run directory", run_dir=run_dir, error=repr(e))
    try:
        if torch is not None and hasattr(torch, "cuda") and torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    gc.collect()


def _is_shm_or_dataloader_error(exc: BaseException) -> bool:
    txt = (repr(exc) + " " + str(exc)).lower()
    needles = [
        "bus error", "shared memory", "shm", "semlock", "boundedsemaphore",
        "no space left on device", "errno 28", "multiprocessing", "dataloader", "worker",
    ]
    return any(k in txt for k in needles)


def _existing_finetuned_checkpoint(domain: str, cfg: ExtendedEvidenceConfig = CFG) -> Path:
    return cfg.output_root / "checkpoints" / f"find_{domain}.pt".replace("/", "_").replace(" ", "_")


def _build_train_kwargs(data_yaml: Path, run_name: str, cfg: ExtendedEvidenceConfig = CFG, *, safe_retry: bool = False) -> Dict[str, Any]:
    batch = int(getattr(cfg, "yolo_safe_retry_batch", 4) if safe_retry else cfg.yolo_batch)
    workers = int(getattr(cfg, "yolo_safe_retry_workers", 0) if safe_retry else cfg.yolo_workers)
    # Enforce non-negative safe workers. workers=0 is the intended default for Docker/Colab SHM limits.
    workers = max(0, workers)
    train_kwargs = dict(
        data=str(data_yaml),
        imgsz=cfg.yolo_imgsz,
        epochs=cfg.yolo_epochs,
        patience=cfg.yolo_patience,
        batch=batch,
        seed=SEED,
        project=str(cfg.output_root / "yolo_runs"),
        name=run_name,
        exist_ok=True,
        pretrained=True,
        verbose=True,
        lr0=cfg.yolo_lr0,
        lrf=cfg.yolo_lrf,
        weight_decay=cfg.yolo_weight_decay,
        workers=workers,
        cache=bool(cfg.yolo_cache),
        amp=bool(getattr(cfg, "yolo_amp", False)),
        plots=bool(getattr(cfg, "yolo_plots", False)),
        save_period=int(getattr(cfg, "yolo_save_period", -1)),
        single_cls=True,
    )
    if cfg.yolo_device is not None:
        train_kwargs["device"] = cfg.yolo_device
    if cfg.yolo_freeze is not None:
        train_kwargs["freeze"] = cfg.yolo_freeze
    return train_kwargs


def train_yolo_find_model(domain: str, data_yaml: Path, base_checkpoint: Path, cfg: ExtendedEvidenceConfig = CFG) -> Optional[Path]:
    """Fine-tune the Find detector only, with SHM-safe defaults and safe retry.

    Measure remains deterministic and is not fine-tuned. This function returns a copied
    checkpoint under CFG.output_root/checkpoints/find_<domain>.pt for TIES and Exp1 sensitivity.
    """
    if not cfg.run_yolo_training:
        log_event("TRAIN", "skip: CFG.run_yolo_training=False", domain=domain)
        return None
    base_checkpoint = Path(base_checkpoint)
    if not base_checkpoint.exists():
        # Late repair: the checkpoint may have been restored after Cell 0C or copied under package layout.
        try:
            repaired = discover_base_find_checkpoint(WORKSPACE, restore_checkpoint_package_if_missing=True)
            if repaired is not None and Path(repaired).exists():
                CFG.base_find_checkpoint = Path(repaired)
                base_checkpoint = Path(repaired)
        except Exception as e:
            log_event("TRAIN", "late base checkpoint discovery failed", domain=domain, error=repr(e))
    if not base_checkpoint.exists():
        log_event("TRAIN", "skip: base Find checkpoint missing; cannot fine-tune Find", domain=domain, expected=base_checkpoint)
        skip_path = cfg.output_root / "tables" / f"exp2_find_finetune_skip_{domain}.csv".replace("/", "_").replace(" ", "_")
        pd.DataFrame([{
            "domain": domain,
            "reason": "base_find_checkpoint_missing",
            "expected_base_checkpoint": str(base_checkpoint),
            "data_yaml": str(data_yaml),
        }]).to_csv(skip_path, index=False)
        return None
    counts = _count_yolo_export_images(Path(data_yaml))
    if counts.get("train", 0) < cfg.min_train_images_for_finetune:
        log_event("TRAIN", "skip: too few training images", domain=domain, train_images=counts.get("train",0), minimum=cfg.min_train_images_for_finetune)
        return None
    if counts.get("val", 0) == 0:
        raise RuntimeError(f"YOLO export for {domain} has no validation images. Re-run Section 3 split repair.")

    dst = _existing_finetuned_checkpoint(domain, cfg)
    if getattr(cfg, "reuse_existing_finetuned_checkpoints", True) and dst.exists() and dst.stat().st_size > 0:
        log_event("TRAIN", "reuse existing fine-tuned checkpoint", domain=domain, checkpoint=dst, sha256=sha256_file(dst))
        return dst

    (cfg.output_root / "checkpoints").mkdir(parents=True, exist_ok=True)
    register_yolo26_compatibility_shims(verbose=True)
    if "ensure_ultralytics_available" in globals():
        if not ensure_ultralytics_available(required=False):
            # Do not hard-stop the whole notebook; record a skip and return None.
            rows = [{
                "domain": domain,
                "stage": "find_finetune",
                "status": "skipped",
                "reason": "ultralytics_unavailable_after_install_attempt",
                "data_yaml": str(data_yaml),
                "base_checkpoint": str(base_checkpoint),
            }]
            out = cfg.output_root / "tables" / f"exp2_find_finetune_skip_{domain}.csv"
            pd.DataFrame(rows).to_csv(out, index=False)
            log_event("TRAIN", "skip fine-tuning because ultralytics is unavailable", domain=domain, audit=out)
            return None
    try:
        from ultralytics import YOLO
    except Exception as e:
        rows = [{
            "domain": domain,
            "stage": "find_finetune",
            "status": "skipped",
            "reason": "ultralytics_import_failed",
            "error": repr(e),
            "data_yaml": str(data_yaml),
            "base_checkpoint": str(base_checkpoint),
        }]
        out = cfg.output_root / "tables" / f"exp2_find_finetune_skip_{domain}.csv"
        pd.DataFrame(rows).to_csv(out, index=False)
        log_event("TRAIN", "skip fine-tuning because ultralytics import failed", domain=domain, audit=out, error=repr(e))
        return None

    storage = _runtime_storage_audit(cfg.output_root)
    log_event("TRAIN", "runtime storage audit", domain=domain, **storage)
    _cleanup_training_runtime(data_yaml=data_yaml, cfg=cfg)

    def _load_model():
        try:
            return YOLO(str(base_checkpoint))
        except AttributeError as e:
            if "SPPF_YOLO26_Compatible" in str(e):
                register_yolo26_compatibility_shims(verbose=True)
                return YOLO(str(base_checkpoint))
            raise

    attempts = [False]
    # If someone overrides workers>0 or AMP=True and the first attempt fails, retry with workers=0/batch<=4/AMP off.
    if getattr(cfg, "yolo_retry_on_shm_error", True):
        attempts.append(True)

    last_error = None
    used_checkpoint_kind = None
    for safe_retry in attempts:
        run_suffix = "_safe" if safe_retry else ""
        run_name = f"ATLAS_FN_ext_find_{domain}{run_suffix}".replace("/", "_").replace(" ", "_")
        run_dir = cfg.output_root / "yolo_runs" / run_name
        if getattr(cfg, "yolo_cleanup_before_train", True):
            _cleanup_training_runtime(data_yaml=data_yaml, run_dir=run_dir, cfg=cfg)
        train_kwargs = _build_train_kwargs(Path(data_yaml), run_name, cfg, safe_retry=safe_retry)
        if safe_retry:
            train_kwargs["workers"] = 0
            train_kwargs["batch"] = min(int(train_kwargs.get("batch", 4)), int(getattr(cfg, "yolo_safe_retry_batch", 4)))
            train_kwargs["amp"] = False
            train_kwargs["cache"] = False
            train_kwargs["plots"] = False
        log_event("TRAIN", "YOLO fine-tune start", domain=domain, counts=counts, base=base_checkpoint,
                  attempt=("safe_retry" if safe_retry else "primary"), epochs=cfg.yolo_epochs,
                  batch=train_kwargs.get("batch"), workers=train_kwargs.get("workers"), amp=train_kwargs.get("amp"), imgsz=cfg.yolo_imgsz)
        try:
            model = _load_model()
            with stage_timer("TRAIN", f"YOLO fine-tune {domain} ({'safe_retry' if safe_retry else 'primary'})"):
                _ = model.train(**train_kwargs)
            best = run_dir / "weights" / "best.pt"
            last = run_dir / "weights" / "last.pt"
            src_ckpt = None
            if best.exists():
                src_ckpt = best
                used_checkpoint_kind = "best"
            elif last.exists():
                src_ckpt = last
                used_checkpoint_kind = "last_fallback"
                log_event("TRAIN", "best.pt not found; using last.pt fallback", domain=domain, checkpoint=last)
            if src_ckpt is None:
                raise FileNotFoundError(f"Neither best.pt nor last.pt was found after YOLO training for {domain}: {run_dir / 'weights'}")
            shutil.copy2(src_ckpt, dst)
            audit = {
                "domain": domain, "data_yaml": str(data_yaml), "base_checkpoint": str(base_checkpoint),
                "base_sha256": sha256_file(Path(base_checkpoint)), "selected_checkpoint": str(dst),
                "selected_checkpoint_kind": used_checkpoint_kind, "selected_sha256": sha256_file(dst),
                "source_run_checkpoint": str(src_ckpt), **{f"n_{k}": v for k, v in counts.items()},
                "epochs": cfg.yolo_epochs, "patience": cfg.yolo_patience, "imgsz": cfg.yolo_imgsz,
                "batch": train_kwargs.get("batch"), "workers": train_kwargs.get("workers"), "amp": train_kwargs.get("amp"),
                "cache": train_kwargs.get("cache"), "plots": train_kwargs.get("plots"),
                "lr0": cfg.yolo_lr0, "lrf": cfg.yolo_lrf,
                "weight_decay": cfg.yolo_weight_decay, "seed": SEED,
                "attempt": ("safe_retry" if safe_retry else "primary"), **storage,
            }
            audit_path = cfg.output_root / "tables" / f"exp2_find_finetune_audit_{domain}.csv".replace("/", "_").replace(" ", "_")
            pd.DataFrame([audit]).to_csv(audit_path, index=False)
            print("[TRAIN DONE]", domain, "->", dst, f"({used_checkpoint_kind})")
            try:
                del model
            except Exception:
                pass
            _cleanup_training_runtime(data_yaml=data_yaml, cfg=cfg)
            return dst
        except Exception as e:
            last_error = e
            try:
                del model
            except Exception:
                pass
            _cleanup_training_runtime(data_yaml=data_yaml, run_dir=run_dir, cfg=cfg)
            if safe_retry or not _is_shm_or_dataloader_error(e):
                log_event("TRAIN", "YOLO fine-tune failed without further retry", domain=domain, attempt=("safe_retry" if safe_retry else "primary"), error=repr(e))
                raise
            log_event("TRAIN", "YOLO fine-tune failed with SHM/DataLoader-like error; retrying safely", domain=domain, error=repr(e))
            time.sleep(2)
            continue
    if last_error is not None:
        raise last_error
    return None


if len(ext_df):
    domain_yamls = {}
    for domain in sorted(ext_df["domain"].astype(str).unique()):
        yp = export_yolo_find_dataset(ext_df, domain, CFG.output_root / "yolo_exports")
        if yp is not None:
            domain_yamls[domain] = yp
    pooled_yaml = export_yolo_find_dataset(ext_df, "ALL", CFG.output_root / "yolo_exports")
    if pooled_yaml is not None:
        domain_yamls["ALL_external"] = pooled_yaml
    pd.Series({k: str(v) for k, v in domain_yamls.items()}).to_csv(CFG.output_root / "tables" / "exp2_yolo_yaml_paths.csv")
else:
    domain_yamls = {}
    pooled_yaml = None
    print("No external dataframe loaded; skipping YOLO export.")

# Optional Find fine-tuning. Measure is deterministic and is not fine-tuned.
trained_domain_checkpoints = {}
# Refresh base checkpoint path after YOLO export and any late restore/repair.
try:
    repaired_base = discover_base_find_checkpoint(WORKSPACE, restore_checkpoint_package_if_missing=True)
    if repaired_base is not None and Path(repaired_base).exists():
        CFG.base_find_checkpoint = Path(repaired_base)
        log_event("TRAIN", "base Find checkpoint ready for fine-tuning", checkpoint=CFG.base_find_checkpoint, sha256=sha256_file(CFG.base_find_checkpoint))
    else:
        log_event("TRAIN", "base Find checkpoint still missing before fine-tuning; domain-specific training will be skipped", expected=CFG.base_find_checkpoint)
except Exception as e:
    log_event("TRAIN", "base Find checkpoint refresh failed before fine-tuning", error=repr(e))
if CFG.run_yolo_training and domain_yamls:
    strategy = str(CFG.yolo_training_strategy).lower().strip()
    allowed = {"domain_specific", "pooled", "both"}
    if strategy not in allowed:
        raise ValueError(f"CFG.yolo_training_strategy must be one of {allowed}, got {CFG.yolo_training_strategy!r}")
    train_domains = [d for d in CFG.yolo_train_domains if d in domain_yamls]
    if strategy in {"domain_specific", "both"}:
        for domain in train_domains:
            ck = train_yolo_find_model(domain, domain_yamls[domain], CFG.base_find_checkpoint)
            if ck is not None:
                trained_domain_checkpoints[domain] = ck
    if strategy in {"pooled", "both"} or CFG.train_pooled_external_find:
        if "ALL_external" in domain_yamls:
            ck = train_yolo_find_model("ALL_external", domain_yamls["ALL_external"], CFG.base_find_checkpoint)
            if ck is not None:
                trained_domain_checkpoints["ALL_external"] = ck
    if trained_domain_checkpoints:
        pd.Series({k: str(v) for k, v in trained_domain_checkpoints.items()}).to_csv(CFG.output_root / "tables" / "exp2_trained_find_checkpoints.csv")
        show_df(pd.DataFrame([{"domain": k, "checkpoint": str(v), "sha256": sha256_file(Path(v))} for k, v in trained_domain_checkpoints.items()]), "trained Find checkpoints")
else:
    print("Training disabled. This is safe for zero-shot evidence. To fine-tune Find, set:")
    print("  CFG.run_yolo_training=True")
    print("  CFG.yolo_training_strategy='domain_specific'   # for TIES")
    print("  # or CFG.yolo_training_strategy='both'          # domain + pooled external")
    print("Measure is not trainable in ATLAS-FN; it remains deterministic geometry + pixel spacing.")


[12:13:33 +  125.6s] [YOLO] export start | domain=FP, rows=1637, out_dir=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/yolo_exports/FP


YOLO export FP:   0%|          | 0/1637 [00:00<?, ?it/s]

[12:13:34 +  126.1s] [YOLO] export done | domain=FP, labelled=1637, yaml=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/yolo_exports/FP/find_FP.yaml, splits={'test': 880, 'train': 662, 'val': 95}
[12:13:34 +  126.1s] [YOLO] export start | domain=UCL, rows=159, out_dir=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/yolo_exports/UCL


YOLO export UCL:   0%|          | 0/159 [00:00<?, ?it/s]

[12:13:34 +  126.2s] [YOLO] export done | domain=UCL, labelled=159, yaml=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/yolo_exports/UCL/find_UCL.yaml, splits={'train': 73, 'test': 55, 'val': 31}
[12:13:34 +  126.2s] [YOLO] export start | domain=ALL_external, rows=1796, out_dir=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/yolo_exports/ALL_external


YOLO export ALL_external:   0%|          | 0/1796 [00:00<?, ?it/s]

[12:13:35 +  126.8s] [YOLO] export done | domain=ALL_external, labelled=1796, yaml=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/yolo_exports/ALL_external/find_ALL_external.yaml, splits={'test': 935, 'train': 735, 'val': 126}
[12:13:35 +  126.9s] [TRAIN] base Find checkpoint ready for fine-tuning | checkpoint=/content/atlas_fn_workspace/Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt, sha256=bcc55db31feaf617d1643185c89c743b1450ef5de4b9dff2e422702a66ff14cd
[12:13:35 +  126.9s] [COMPAT] YOLO26 compatibility shim registered | class_name=SPPF_YOLO26_Compatible
[12:13:35 +  126.9s] [TRAIN] runtime storage audit | domain=FP, output_root_free_mb=162008.5, output_root_total_mb=241332.1, tmp_free_mb=162008.5, tmp_total_mb=241332.1, dev_shm_free_mb=84147.0, dev_shm_total_mb=84147.0
[12:13:35 +  127.2s] [TRAIN] YOLO fine-tune start | domain=FP, counts={'train': 662, 'val': 95, 'test': 880}, base=/content/atlas_fn_workspace/Training_Runs/BrainLocator_YOLO26n_MuSGD/weigh

,domain,checkpoint,sha256
0,FP,/content/atlas_fn_workspace/outputs/extended_e...,cc6e8b92e3fe27b2469cfb5b94b1637e9f3e79d086ebac...
1,UCL,/content/atlas_fn_workspace/outputs/extended_e...,32a1a9e3c96899a7a2b4cfc5822aad703a8f520f703891...
2,ALL_external,/content/atlas_fn_workspace/outputs/extended_e...,b5298fd49fc6ef4d52c490ca7e637d6a6d2e8ca197f121...


## 5. Experiment 1 — Cross-dataset external validation of Find/Measure

This section runs the existing Find model on the multicentre external dataset and computes:

- detection availability
- reference-box IoU
- BPD MAE, OFD MAE and HC MAE
- HC absolute percentage error
- broad paired precision using BPD ≤ 2.0 mm and HC ≤ 5.0%
- domain-wise bootstrap confidence intervals

**Important geometry note.** ATLAS-FN does not output a separate OFD variable. OFD is estimated from the major dimension of the Find-stage box for secondary geometry analysis. The primary ATLAS-FN-compatible endpoints remain BPD and HC.


In [10]:
# ============================================================
# 6. Find inference and Experiment 1 metrics
# ============================================================
def _load_yolo_model_once(checkpoint: Path):
    checkpoint = Path(checkpoint)
    register_yolo26_compatibility_shims(verbose=True)
    if "ensure_ultralytics_available" in globals():
        ensure_ultralytics_available(required=False)
    try:
        from ultralytics import YOLO
    except Exception as e:
        raise RuntimeError("Ultralytics is required for external Find/Measure inference and could not be imported after install attempt.") from e
    log_event("INFER", "loading YOLO checkpoint", checkpoint=checkpoint)
    try:
        model = YOLO(str(checkpoint))
    except AttributeError as e:
        if "SPPF_YOLO26_Compatible" in str(e):
            log_event("INFER", "retrying YOLO load after compatibility shim", checkpoint=checkpoint, error=repr(e))
            register_yolo26_compatibility_shims(verbose=True)
            model = YOLO(str(checkpoint))
        else:
            raise
    except Exception as e:
        log_event("INFER", "YOLO checkpoint load failed", checkpoint=checkpoint, error=repr(e))
        raise
    log_event("INFER", "YOLO checkpoint loaded", checkpoint=checkpoint)
    return model

def _predict_yolo_xyxy_loaded(model, image_path: Path, conf: float = 0.25) -> Tuple[Optional[List[float]], float, Dict[str, Any]]:
    """Return best xyxy and confidence from an already-loaded Ultralytics YOLO model."""
    res = model.predict(source=str(image_path), conf=conf, verbose=False)
    if not res:
        return None, 0.0, {"n": 0}
    r0 = res[0]
    if r0.boxes is None or len(r0.boxes) == 0:
        return None, 0.0, {"n": 0}
    boxes = r0.boxes
    confs = boxes.conf.detach().cpu().numpy()
    idx = int(np.argmax(confs))
    xyxy = boxes.xyxy[idx].detach().cpu().numpy().astype(float).tolist()
    return xyxy, float(confs[idx]), {"n": int(len(boxes))}

def run_find_measure_external(df: pd.DataFrame, checkpoint: Path, model_name: str = "base_find", cfg: ExtendedEvidenceConfig = CFG) -> pd.DataFrame:
    rows = []
    if df.empty:
        log_event("INFER", "empty dataframe; nothing to evaluate", model=model_name)
        return pd.DataFrame(rows)
    if not Path(checkpoint).exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint}")
    model = _load_yolo_model_once(Path(checkpoint))
    ck_sha = sha256_file(Path(checkpoint)) if Path(checkpoint).exists() else None
    log_event("INFER", "evaluation start", model=model_name, rows=len(df), checkpoint=checkpoint)
    t_eval = time.time()
    iterator = progress_iter(df.iterrows(), desc=f"infer {model_name}", total=len(df), stage="INFER")
    for j, (i, r) in enumerate(iterator, start=1):
        img = Path(r["image_path"])
        row = r.to_dict()
        row["model_name"] = model_name
        row["checkpoint"] = str(checkpoint)
        row["checkpoint_sha256"] = ck_sha
        t0 = time.perf_counter()
        if not img.exists():
            row.update({"detected": 0, "pred_conf": 0.0, "pred_x1": np.nan, "pred_y1": np.nan, "pred_x2": np.nan, "pred_y2": np.nan, "latency_ms": np.nan, "inference_note": "image_missing"})
            rows.append(row)
            continue
        try:
            xyxy, conf, meta = _predict_yolo_xyxy_loaded(model, img, conf=cfg.yolo_conf)
            latency_ms = (time.perf_counter() - t0) * 1000
            if xyxy is None:
                row.update({"detected": 0, "pred_conf": conf, "pred_x1": np.nan, "pred_y1": np.nan, "pred_x2": np.nan, "pred_y2": np.nan, "latency_ms": latency_ms, "inference_note": "no_detection"})
            else:
                bpd_mm, hc_mm = bpd_hc_from_xyxy(xyxy, r["pixel_spacing_mm"])
                w_px = abs(xyxy[2] - xyxy[0])
                h_px = abs(xyxy[3] - xyxy[1])
                pred_ofd_mm = max(w_px, h_px) * float(r["pixel_spacing_mm"])
                ref_xyxy = [r["ref_x1"], r["ref_y1"], r["ref_x2"], r["ref_y2"]]
                bpd_err = bpd_error_mm(bpd_mm, r["ref_bpd_mm"])
                ofd_err = abs(pred_ofd_mm - r["ref_ofd_mm"]) if finite(pred_ofd_mm) and finite(r["ref_ofd_mm"]) else np.nan
                hc_err_mm = abs(hc_mm - r["ref_hc_mm"]) if finite(hc_mm) and finite(r["ref_hc_mm"]) else np.nan
                hc_err_pct = hc_error_pct(hc_mm, r["ref_hc_mm"])
                row.update({
                    "detected": 1,
                    "pred_conf": conf,
                    "pred_x1": xyxy[0], "pred_y1": xyxy[1], "pred_x2": xyxy[2], "pred_y2": xyxy[3],
                    "latency_ms": latency_ms,
                    "pred_bpd_mm": bpd_mm,
                    "pred_ofd_mm": pred_ofd_mm,
                    "pred_hc_mm": hc_mm,
                    "ref_box_iou": xyxy_iou(xyxy, ref_xyxy),
                    "bpd_abs_err_mm": bpd_err,
                    "ofd_abs_err_mm": ofd_err,
                    "hc_abs_err_mm": hc_err_mm,
                    "hc_abs_err_pct": hc_err_pct,
                    "broad_pair": broad_pair_flag(bpd_err, hc_err_pct),
                    "inference_note": "ok",
                })
        except Exception as e:
            row.update({"detected": 0, "pred_conf": 0.0, "latency_ms": np.nan, "inference_note": f"error:{type(e).__name__}:{e}"})
        rows.append(row)
        if j % cfg.progress_log_every_rows == 0:
            det = int(pd.DataFrame(rows)["detected"].sum()) if rows else 0
            log_event("INFER", "heartbeat", model=model_name, processed=j, total=len(df), detected=det)
    ledger = pd.DataFrame(rows)
    log_event("INFER", "evaluation done", model=model_name, rows=len(ledger), detected=int(ledger.get("detected", pd.Series(dtype=int)).sum()) if len(ledger) else 0, elapsed_sec=round(time.time()-t_eval, 2))
    return ledger

def summarize_external_metrics(ledger: pd.DataFrame, group_cols: Sequence[str] = ("domain", "split", "model_name")) -> pd.DataFrame:
    if ledger.empty:
        return pd.DataFrame()
    def p90(x): return float(np.nanpercentile(x, 90)) if len(pd.Series(x).dropna()) else np.nan
    out = ledger.groupby(list(group_cols), dropna=False).agg(
        n=("image_path", "size"),
        detected=("detected", "sum"),
        detection_rate=("detected", "mean"),
        mean_iou=("ref_box_iou", "mean"),
        median_iou=("ref_box_iou", "median"),
        bpd_mae_mm=("bpd_abs_err_mm", "mean"),
        bpd_p90_mm=("bpd_abs_err_mm", p90),
        ofd_mae_mm=("ofd_abs_err_mm", "mean"),
        hc_mae_mm=("hc_abs_err_mm", "mean"),
        hc_mae_pct=("hc_abs_err_pct", "mean"),
        broad_precision=("broad_pair", "mean"),
        latency_mean_ms=("latency_ms", "mean"),
        latency_p90_ms=("latency_ms", p90),
    ).reset_index()
    out["detected"] = out["detected"].astype(int)
    out["detection_rate"] = 100.0 * out["detection_rate"]
    out["broad_precision"] = 100.0 * out["broad_precision"]
    return out

def get_primary_exp1_evaluation_df(df: pd.DataFrame, cfg: ExtendedEvidenceConfig = CFG) -> pd.DataFrame:
    """Return the locked primary external evaluation cohort for Experiment 1.

    By default this is the subject-exclusive `test` split from true-source FP and UCL.
    Train/val rows are retained for optional fine-tuning, but not used for primary
    external-validation metrics.
    """
    if df is None or df.empty:
        return pd.DataFrame()
    primary = {normalise_domain_label(d) for d in getattr(cfg, "primary_external_domains", ("FP", "UCL"))}
    eval_split = str(getattr(cfg, "primary_eval_split", "test")).lower()
    work = df.copy()
    work["domain"] = work["domain"].map(normalise_domain_label)
    work["split"] = normalise_split_series(work["split"])
    eval_df = work[(work["domain"].isin(primary)) & (work["split"].eq(eval_split))].copy()
    audit = work.groupby(["domain", "split"], dropna=False).agg(rows=("image_path", "size"), subjects=("subject_key", "nunique") if "subject_key" in work.columns else ("subject_id", "nunique")).reset_index()
    audit.to_csv(cfg.output_root / "tables" / "exp1_v11_primary_eval_split_audit.csv", index=False)
    eval_df.to_csv(cfg.output_root / "tables" / "exp1_v11_primary_test_evaluation_cohort.csv", index=False)
    show_df(audit, "v11 Experiment 1 available split audit", max_rows=30)
    log_event("EVAL", "selected primary external evaluation cohort", split=eval_split, rows=len(eval_df), domains=sorted(eval_df["domain"].unique()) if len(eval_df) else [])
    return eval_df

def build_find_evaluation_jobs(df: pd.DataFrame, cfg: ExtendedEvidenceConfig = CFG) -> List[Dict[str, Any]]:
    jobs = []
    if cfg.base_find_checkpoint.exists():
        jobs.append({"model_name": "ATLAS_base_find_zero_shot", "checkpoint": cfg.base_find_checkpoint, "df": df})
    # Domain-specific trained checkpoints are evaluated on their matching domain.
    for domain, ck in trained_domain_checkpoints.items() if "trained_domain_checkpoints" in globals() else []:
        if not Path(ck).exists():
            continue
        if str(domain) == "ALL_external":
            jobs.append({"model_name": "ATLAS_find_pooled_external", "checkpoint": Path(ck), "df": df})
        else:
            sub = df[df["domain"].astype(str) == str(domain)].copy()
            if len(sub):
                jobs.append({"model_name": f"ATLAS_find_finetuned_{domain}", "checkpoint": Path(ck), "df": sub})
    return jobs

# Refresh base checkpoint after any late checkpoint-package restore before zero-shot evaluation.
try:
    repaired_base = discover_base_find_checkpoint(WORKSPACE, restore_checkpoint_package_if_missing=True)
    if repaired_base is not None and Path(repaired_base).exists():
        CFG.base_find_checkpoint = Path(repaired_base)
        log_event("INFER", "base Find checkpoint ready for zero-shot evaluation", checkpoint=CFG.base_find_checkpoint, sha256=sha256_file(CFG.base_find_checkpoint))
    else:
        log_event("INFER", "base Find checkpoint unavailable; Experiment 1 zero-shot will be skipped unless domain checkpoints exist", expected=CFG.base_find_checkpoint)
except Exception as e:
    log_event("INFER", "base Find checkpoint refresh failed before Experiment 1", error=repr(e))

if CFG.run_find_inference and len(ext_df):
    ledgers = []
    exp1_eval_df = get_primary_exp1_evaluation_df(ext_df, CFG)
    if exp1_eval_df.empty:
        raise RuntimeError("No primary external test rows available after v11 domain/split policy. Check exp1_v11_primary_eval_split_audit.csv")
    jobs = build_find_evaluation_jobs(exp1_eval_df, CFG)
    if not jobs:
        print("No Find evaluation jobs: base checkpoint and trained checkpoints are missing.")
    for job in progress_iter(jobs, desc="Experiment 1 evaluation jobs", total=len(jobs), stage="INFER"):
        log_event("EVAL", "job start", model=job["model_name"], frames=len(job["df"]), checkpoint=job["checkpoint"])
        ledgers.append(run_find_measure_external(job["df"], Path(job["checkpoint"]), job["model_name"]))
    exp1_ledger = pd.concat(ledgers, ignore_index=True) if ledgers else pd.DataFrame()
    exp1_summary = summarize_external_metrics(exp1_ledger) if len(exp1_ledger) else pd.DataFrame()
    exp1_ledger.to_csv(CFG.output_root / "tables" / "exp1_external_find_measure_ledger.csv", index=False)
    exp1_summary.to_csv(CFG.output_root / "tables" / "exp1_external_find_measure_summary.csv", index=False)
    show_df(exp1_summary, "Experiment 1 external Find/Measure summary")
else:
    exp1_ledger = pd.DataFrame()
    exp1_summary = pd.DataFrame()
    print("Experiment 1 inference not run. Check CFG.run_find_inference, data paths and checkpoint path.")


[12:27:12 +  944.7s] [INFER] base Find checkpoint ready for zero-shot evaluation | checkpoint=/content/atlas_fn_workspace/Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt, sha256=bcc55db31feaf617d1643185c89c743b1450ef5de4b9dff2e422702a66ff14cd
[12:27:13 +  944.8s] [DATA] v11 Experiment 1 available split audit | rows=6, columns=4


,domain,split,rows,subjects
0,FP,test,880,460
1,FP,train,662,395
2,FP,val,95,54
3,UCL,test,55,15
4,UCL,train,73,28
5,UCL,val,31,4


[12:27:13 +  944.8s] [EVAL] selected primary external evaluation cohort | split=test, rows=935, domains=['FP', 'UCL']


Experiment 1 evaluation jobs:   0%|          | 0/4 [00:00<?, ?it/s]

[12:27:13 +  944.8s] [EVAL] job start | model=ATLAS_base_find_zero_shot, frames=935, checkpoint=/content/atlas_fn_workspace/Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt
[12:27:13 +  944.8s] [COMPAT] YOLO26 compatibility shim registered | class_name=SPPF_YOLO26_Compatible
[12:27:13 +  944.8s] [INFER] loading YOLO checkpoint | checkpoint=/content/atlas_fn_workspace/Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt
[12:27:13 +  944.8s] [INFER] YOLO checkpoint loaded | checkpoint=/content/atlas_fn_workspace/Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt
[12:27:13 +  944.9s] [INFER] evaluation start | model=ATLAS_base_find_zero_shot, rows=935, checkpoint=/content/atlas_fn_workspace/Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt


...

infer ATLAS_find_finetuned_UCL:   0%|          | 0/55 [00:00<?, ?it/s]

[12:27:55 +  986.7s] [INFER] evaluation done | model=ATLAS_find_finetuned_UCL, rows=55, detected=55, elapsed_sec=1.74
[12:27:55 +  986.7s] [EVAL] job start | model=ATLAS_find_pooled_external, frames=935, checkpoint=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/find_ALL_external.pt
[12:27:55 +  986.7s] [COMPAT] YOLO26 compatibility shim registered | class_name=SPPF_YOLO26_Compatible
[12:27:55 +  986.7s] [INFER] loading YOLO checkpoint | checkpoint=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/find_ALL_external.pt
[12:27:55 +  986.8s] [INFER] YOLO checkpoint loaded | checkpoint=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/find_ALL_external.pt
[12:27:55 +  986.8s] [INFER] evaluation start | model=ATLAS_find_pooled_external, rows=935, checkpoint=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/find_ALL_external.pt


infer ATLAS_find_pooled_external:   0%|          | 0/935 [00:00<?, ?it/s]

[12:28:00 +  992.3s] [INFER] heartbeat | model=ATLAS_find_pooled_external, processed=250, total=935, detected=248
[12:28:05 +  997.6s] [INFER] heartbeat | model=ATLAS_find_pooled_external, processed=500, total=935, detected=498
[12:28:11 + 1003.1s] [INFER] heartbeat | model=ATLAS_find_pooled_external, processed=750, total=935, detected=747
[12:28:15 + 1007.5s] [INFER] evaluation done | model=ATLAS_find_pooled_external, rows=935, detected=931, elapsed_sec=20.69
[12:28:15 + 1007.7s] [DATA] Experiment 1 external Find/Measure summary | rows=6, columns=16


,domain,split,model_name,n,detected,detection_rate,mean_iou,median_iou,bpd_mae_mm,bpd_p90_mm,ofd_mae_mm,hc_mae_mm,hc_mae_pct,broad_precision,latency_mean_ms,latency_p90_ms
0,FP,test,ATLAS_base_find_zero_shot,880,879,99.886364,0.912621,0.920856,0.927250,1.737465,1.791539,3.129525,1.243802,91.240046,21.531561,23.256507
1,FP,test,ATLAS_find_finetuned_FP,880,878,99.772727,0.944965,0.961909,2.627878,4.320593,3.069473,8.463212,3.546064,39.407745,21.179194,22.907767
2,FP,test,ATLAS_find_pooled_external,880,877,99.659091,0.942182,0.959289,2.535195,4.143991,2.595300,7.308958,3.017119,38.198404,21.287068,23.379864
3,UCL,test,ATLAS_base_find_zero_shot,55,55,100.000000,0.884449,0.896612,1.936474,3.649432,3.715841,7.930714,2.843074,61.818182,32.098379,47.529990
4,UCL,test,ATLAS_find_finetuned_UCL,55,55,100.000000,0.923151,0.943774,3.383688,5.536753,2.942949,8.770826,3.167935,20.000000,31.090550,39.645415
5,UCL,test,ATLAS_find_pooled_external,55,54,98.181818,0.931548,0.952142,3.133822,5.452607,2.492153,7.372718,2.787514,25.925926,29.687763,38.192198


In [11]:
# ============================================================
# 7. Enhanced Experiment 1 evidence: bootstrap CIs, thresholds, quantiles, failures and paired deltas
# ============================================================
def bootstrap_mean_ci(values: Sequence[float], n_boot: int = 10_000, seed: int = 42, ci: float = 0.95) -> Tuple[float, float, float]:
    x = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(n_boot, len(x)))
    means = x[idx].mean(axis=1)
    alpha = (1 - ci) / 2
    return float(x.mean()), float(np.quantile(means, alpha)), float(np.quantile(means, 1 - alpha))

def bootstrap_domain_table(ledger: pd.DataFrame, metrics=("bpd_abs_err_mm", "ofd_abs_err_mm", "hc_abs_err_mm", "hc_abs_err_pct", "ref_box_iou")) -> pd.DataFrame:
    rows = []
    if ledger.empty:
        return pd.DataFrame(rows)
    groups = list(ledger.groupby(["domain", "split", "model_name"], dropna=False))
    for keys, sub in progress_iter(groups, desc="bootstrap CI groups", total=len(groups), stage="BOOTCI"):
        domain, split, model = keys
        for m in metrics:
            if m not in sub.columns:
                continue
            mean, lo, hi = bootstrap_mean_ci(sub.loc[sub.get("detected", 1).eq(1), m] if "detected" in sub.columns else sub[m], n_boot=CFG.bootstrap_n, seed=CFG.bootstrap_seed)
            rows.append({"domain": domain, "split": split, "model_name": model, "metric": m, "mean": mean, "ci95_low": lo, "ci95_high": hi, "n_nonmissing": int(sub[m].notna().sum())})
    return pd.DataFrame(rows)

def exp1_threshold_grid(ledger: pd.DataFrame,
                        bpd_thresholds=(1.0, 1.5, 2.0, 3.0),
                        hc_thresholds=(2.0, 3.0, 5.0, 10.0)) -> pd.DataFrame:
    rows = []
    if ledger.empty:
        return pd.DataFrame(rows)
    groups = list(ledger.groupby(["domain", "split", "model_name"], dropna=False))
    for (domain, split, model), sub in progress_iter(groups, desc="Exp1 threshold grids", total=len(groups), stage="EXP1"):
        det = sub[sub["detected"].eq(1)].copy() if "detected" in sub.columns else sub.copy()
        for bt in bpd_thresholds:
            for ht in hc_thresholds:
                eligible = det["bpd_abs_err_mm"].notna() & det["hc_abs_err_pct"].notna()
                n = int(eligible.sum())
                ok = ((det.loc[eligible, "bpd_abs_err_mm"] <= bt) & (det.loc[eligible, "hc_abs_err_pct"] <= ht)).sum() if n else 0
                rows.append({"domain": domain, "split": split, "model_name": model, "bpd_threshold_mm": bt, "hc_threshold_pct": ht, "n_detected_with_metrics": n, "n_success": int(ok), "success_rate_pct": 100.0 * ok / n if n else np.nan})
    return pd.DataFrame(rows)

def exp1_error_quantiles(ledger: pd.DataFrame) -> pd.DataFrame:
    rows = []
    metrics = ["ref_box_iou", "bpd_abs_err_mm", "ofd_abs_err_mm", "hc_abs_err_mm", "hc_abs_err_pct", "latency_ms"]
    if ledger.empty:
        return pd.DataFrame(rows)
    for (domain, split, model), sub in ledger.groupby(["domain", "split", "model_name"], dropna=False):
        det = sub[sub["detected"].eq(1)] if "detected" in sub.columns else sub
        for m in metrics:
            if m not in det.columns:
                continue
            vals = pd.Series(det[m]).dropna()
            if len(vals) == 0:
                continue
            qs = vals.quantile([0.05, 0.25, 0.50, 0.75, 0.90, 0.95]).to_dict()
            rows.append({"domain": domain, "split": split, "model_name": model, "metric": m, "n": int(len(vals)), "mean": float(vals.mean()), "sd": float(vals.std(ddof=1)) if len(vals)>1 else np.nan, "q05": qs.get(0.05), "q25": qs.get(0.25), "q50": qs.get(0.50), "q75": qs.get(0.75), "q90": qs.get(0.90), "q95": qs.get(0.95)})
    return pd.DataFrame(rows)

def exp1_detection_failure_audit(ledger: pd.DataFrame) -> pd.DataFrame:
    if ledger.empty or "detected" not in ledger.columns:
        return pd.DataFrame()
    cols = [c for c in ["domain", "split", "model_name", "row_id", "image_path", "subject_id", "subject_key", "pred_conf", "inference_note", "img_w", "img_h", "ref_bpd_mm", "ref_ofd_mm", "ref_hc_mm"] if c in ledger.columns]
    return ledger.loc[~ledger["detected"].eq(1), cols].copy()

def exp1_paired_model_delta(ledger: pd.DataFrame, baseline_model: str = "ATLAS_base_find_zero_shot") -> pd.DataFrame:
    """Within-row paired error differences versus the zero-shot base model."""
    if ledger.empty or baseline_model not in set(ledger["model_name"].astype(str)):
        return pd.DataFrame()
    keys = ["domain", "split", "row_id"]
    metrics = ["bpd_abs_err_mm", "ofd_abs_err_mm", "hc_abs_err_mm", "hc_abs_err_pct", "ref_box_iou"]
    base = ledger[ledger["model_name"].eq(baseline_model)][keys + metrics].copy()
    base = base.rename(columns={m: f"base_{m}" for m in metrics})
    rows = []
    for model, sub in ledger[~ledger["model_name"].eq(baseline_model)].groupby("model_name"):
        merged = sub[keys + metrics].merge(base, on=keys, how="inner")
        for m in metrics:
            if m not in merged.columns:
                continue
            delta = merged[m] - merged[f"base_{m}"]
            mean, lo, hi = bootstrap_mean_ci(delta, n_boot=CFG.bootstrap_n, seed=CFG.bootstrap_seed)
            rows.append({"comparison_model": model, "baseline_model": baseline_model, "metric": m, "n_paired": int(delta.notna().sum()), "mean_delta_vs_base": mean, "ci95_low": lo, "ci95_high": hi, "interpretation": "negative favours comparison for error metrics; positive favours comparison for IoU"})
    return pd.DataFrame(rows)

if len(exp1_ledger):
    exp1_boot = bootstrap_domain_table(exp1_ledger)
    exp1_thresholds = exp1_threshold_grid(exp1_ledger)
    exp1_quantiles = exp1_error_quantiles(exp1_ledger)
    exp1_failures = exp1_detection_failure_audit(exp1_ledger)
    exp1_deltas = exp1_paired_model_delta(exp1_ledger)
    exp1_boot.to_csv(CFG.output_root / "tables" / "exp1_bootstrap_ci_by_domain.csv", index=False)
    exp1_thresholds.to_csv(CFG.output_root / "tables" / "exp1_threshold_precision_grid.csv", index=False)
    exp1_quantiles.to_csv(CFG.output_root / "tables" / "exp1_error_quantiles.csv", index=False)
    exp1_failures.to_csv(CFG.output_root / "tables" / "exp1_detection_failure_audit.csv", index=False)
    exp1_deltas.to_csv(CFG.output_root / "tables" / "exp1_paired_model_deltas_vs_base.csv", index=False)
    show_df(exp1_boot, "Experiment 1 bootstrap confidence intervals", max_rows=20)
    show_df(exp1_thresholds, "Experiment 1 paired precision threshold grid", max_rows=20)
    show_df(exp1_quantiles, "Experiment 1 error quantiles", max_rows=20)
    log_event("EXP1", "enhanced evidence tables written", boot=len(exp1_boot), thresholds=len(exp1_thresholds), quantiles=len(exp1_quantiles), failures=len(exp1_failures), deltas=len(exp1_deltas))
else:
    exp1_boot = exp1_thresholds = exp1_quantiles = exp1_failures = exp1_deltas = pd.DataFrame()
    print("No Experiment 1 ledger available for enhanced evidence tables.")


bootstrap CI groups:   0%|          | 0/6 [00:00<?, ?it/s]

Exp1 threshold grids:   0%|          | 0/6 [00:00<?, ?it/s]

[12:28:18 + 1010.2s] [DATA] Experiment 1 bootstrap confidence intervals | rows=30, columns=8


,domain,split,model_name,metric,mean,ci95_low,ci95_high,n_nonmissing
0,FP,test,ATLAS_base_find_zero_shot,bpd_abs_err_mm,0.927250,0.840563,1.023014,879
1,FP,test,ATLAS_base_find_zero_shot,ofd_abs_err_mm,1.791539,1.665679,1.923297,879
2,FP,test,ATLAS_base_find_zero_shot,hc_abs_err_mm,3.129525,2.903091,3.371703,879
3,FP,test,ATLAS_base_find_zero_shot,hc_abs_err_pct,1.243802,1.142248,1.360294,879
4,FP,test,ATLAS_base_find_zero_shot,ref_box_iou,0.912621,0.908865,0.916052,879
5,FP,test,ATLAS_find_finetuned_FP,bpd_abs_err_mm,2.627878,2.536845,2.722857,878
6,FP,test,ATLAS_find_finetuned_FP,ofd_abs_err_mm,3.069473,2.940282,3.201425,878
7,FP,test,ATLAS_find_finetuned_FP,hc_abs_err_mm,8.463212,8.177358,8.744645,878
8,FP,test,ATLAS_find_finetuned_FP,hc_abs_err_pct,3.546064,3.446604,3.642429,878
9,FP,test,ATLAS_find_finetuned_FP,ref_box_iou,0.944965,0.940882,0.948724,878


[12:28:18 + 1010.2s] [DATA] Experiment 1 paired precision threshold grid | rows=96, columns=8


,domain,split,model_name,bpd_threshold_mm,hc_threshold_pct,n_detected_with_metrics,n_success,success_rate_pct
0,FP,test,ATLAS_base_find_zero_shot,1.0,2.0,879,575,65.415245
1,FP,test,ATLAS_base_find_zero_shot,1.0,3.0,879,619,70.420933
2,FP,test,ATLAS_base_find_zero_shot,1.0,5.0,879,648,73.720137
3,FP,test,ATLAS_base_find_zero_shot,1.0,10.0,879,651,74.061433
4,FP,test,ATLAS_base_find_zero_shot,1.5,2.0,879,659,74.971559
5,FP,test,ATLAS_base_find_zero_shot,1.5,3.0,879,714,81.228669
6,FP,test,ATLAS_base_find_zero_shot,1.5,5.0,879,755,85.893060
7,FP,test,ATLAS_base_find_zero_shot,1.5,10.0,879,759,86.348123
8,FP,test,ATLAS_base_find_zero_shot,2.0,2.0,879,690,78.498294
9,FP,test,ATLAS_base_find_zero_shot,2.0,3.0,879,758,86.234357


[12:28:18 + 1010.3s] [DATA] Experiment 1 error quantiles | rows=36, columns=13


,domain,split,model_name,metric,n,mean,sd,q05,q25,q50,q75,q90,q95
0,FP,test,ATLAS_base_find_zero_shot,ref_box_iou,879,0.912621,0.053704,0.835870,0.902515,0.920856,0.939459,0.956771,0.967759
1,FP,test,ATLAS_base_find_zero_shot,bpd_abs_err_mm,879,0.927250,1.365882,0.060191,0.277933,0.604906,1.011884,1.737465,2.878600
2,FP,test,ATLAS_base_find_zero_shot,ofd_abs_err_mm,879,1.791539,1.968926,0.088657,0.436819,1.043068,2.439250,4.651542,5.897984
3,FP,test,ATLAS_base_find_zero_shot,hc_abs_err_mm,879,3.129525,3.623582,0.161089,0.800560,1.833294,4.109604,7.608190,10.596374
4,FP,test,ATLAS_base_find_zero_shot,hc_abs_err_pct,879,1.243802,1.683664,0.080686,0.402572,0.813296,1.527626,2.762990,3.596230
5,FP,test,ATLAS_base_find_zero_shot,latency_ms,879,21.532478,7.222037,19.204687,20.372617,21.071703,21.760280,23.265409,24.146918
6,FP,test,ATLAS_find_finetuned_FP,ref_box_iou,878,0.944965,0.059170,0.841806,0.944077,0.961909,0.973255,0.980644,0.984717
7,FP,test,ATLAS_find_finetuned_FP,bpd_abs_err_mm,878,2.627878,1.427446,0.926778,1.689810,2.302202,3.398536,4.320593,5.196633
8,FP,test,ATLAS_find_finetuned_FP,ofd_abs_err_mm,878,3.069473,1.965268,0.604026,1.756870,2.586691,4.122926,5.899259,6.672484
9,FP,test,ATLAS_find_finetuned_FP,hc_abs_err_mm,878,8.463212,4.247719,2.740640,5.612785,7.603814,10.849480,14.243681,16.033416


[12:28:18 + 1010.3s] [EXP1] enhanced evidence tables written | boot=30, thresholds=96, quantiles=36, failures=7, deltas=15


## 6. Experiment 2 — complete real-domain TIES-Merging

This section implements the required v12 Experiment 2:

1. resolve actual FP and UCL Find-specialist checkpoints trained from the source/base Find checkpoint;
2. compute task vectors relative to the source/base checkpoint;
3. apply Trim, Elect Sign and Disjoint Merge;
4. save both a generic merged state and an executable Ultralytics checkpoint;
5. evaluate the TIES-merged checkpoint against the same FP+UCL held-out test cohort used in Experiment 1.

The notebook runs with `CFG.require_complete_exp2=True`; therefore, it stops before export if real-domain checkpoints, TIES outputs, or the executable merged-checkpoint evaluation are missing.


In [12]:
# ============================================================
# 8. TIES state-dict extraction, executable checkpoint construction and algebra
# ============================================================
try:
    import torch
except Exception as e:
    torch = None
    print("PyTorch not available; TIES cells will not run until torch is installed.")

import copy

def load_checkpoint_any(path: Path):
    if torch is None:
        raise RuntimeError("PyTorch is required for TIES-Merging.")
    register_yolo26_compatibility_shims(verbose=False)
    path = Path(path)
    try:
        return torch.load(str(path), map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(str(path), map_location="cpu")
    except AttributeError as e:
        if "SPPF_YOLO26_Compatible" in str(e):
            log_event("TIES", "retrying torch.load after compatibility shim", checkpoint=path, error=repr(e))
            register_yolo26_compatibility_shims(verbose=True)
            try:
                return torch.load(str(path), map_location="cpu", weights_only=False)
            except TypeError:
                return torch.load(str(path), map_location="cpu")
        raise

def _candidate_state_sources(obj: Any) -> List[Any]:
    candidates = []
    if isinstance(obj, dict):
        # Ultralytics normally prefers ema if present. For task vectors we prefer the deployed model if available,
        # but include all options as fallbacks.
        for key in ["model", "ema", "state_dict"]:
            if key in obj and obj[key] is not None:
                candidates.append(obj[key])
        candidates.append(obj)
    else:
        candidates.append(obj)
    return candidates

def extract_float_state_dict(obj: Any) -> Dict[str, "torch.Tensor"]:
    """Extract a float tensor state dict from Ultralytics/PyTorch checkpoints."""
    if torch is None:
        raise RuntimeError("PyTorch not available")
    for cand in _candidate_state_sources(obj):
        try:
            sd = cand.state_dict() if hasattr(cand, "state_dict") else cand
            if isinstance(sd, dict):
                out = {str(k): v.detach().cpu().float().clone() for k, v in sd.items() if hasattr(v, "dtype") and torch.is_floating_point(v)}
                if out:
                    return out
        except Exception:
            continue
    raise ValueError("Could not extract a float tensor state_dict from checkpoint.")

def get_checkpoint_model_module(ckpt_obj: Any):
    if isinstance(ckpt_obj, dict):
        model = ckpt_obj.get("model", None)
        if model is not None and hasattr(model, "state_dict"):
            return model
        ema = ckpt_obj.get("ema", None)
        if ema is not None and hasattr(ema, "state_dict"):
            return ema
    if hasattr(ckpt_obj, "state_dict"):
        return ckpt_obj
    raise ValueError("Checkpoint does not contain an executable nn.Module under 'model' or 'ema'.")

def align_state_dicts(base_sd: Dict[str, Any], site_sds: Dict[str, Dict[str, Any]]) -> Tuple[List[str], Dict[str, Any], Dict[str, Dict[str, Any]]]:
    keys = set(base_sd.keys())
    for sd in site_sds.values():
        keys &= set(sd.keys())
    aligned = []
    for k in sorted(keys):
        if all(tuple(site_sds[d][k].shape) == tuple(base_sd[k].shape) for d in site_sds):
            aligned.append(k)
    base_aligned = {k: base_sd[k] for k in aligned}
    sites_aligned = {d: {k: sd[k] for k in aligned} for d, sd in site_sds.items()}
    return aligned, base_aligned, sites_aligned

def flatten_state_dict(sd: Dict[str, "torch.Tensor"], keys: Sequence[str]) -> "torch.Tensor":
    chunks = []
    for k in progress_iter(keys, desc="flatten state_dict", total=len(keys), stage="TIES", leave=False):
        chunks.append(sd[k].reshape(-1))
    return torch.cat(chunks)

def unflatten_to_state_dict(vec: "torch.Tensor", template_sd: Dict[str, "torch.Tensor"], keys: Sequence[str]) -> Dict[str, "torch.Tensor"]:
    out = {}
    pos = 0
    for k in progress_iter(keys, desc="unflatten merged state", total=len(keys), stage="TIES", leave=False):
        n = template_sd[k].numel()
        out[k] = vec[pos:pos+n].reshape_as(template_sd[k]).clone()
        pos += n
    return out

def ties_merge_vectors(task_vectors: Dict[str, "torch.Tensor"], density: float = 0.20) -> Dict[str, Any]:
    domains = list(task_vectors.keys())
    M = torch.stack([task_vectors[d] for d in domains], dim=0)  # [S, D]
    D = M.shape[1]
    trimmed = torch.zeros_like(M)
    thresholds = {}
    retained_counts = {}
    log_event("TIES", "merge start", domains=domains, D=int(D), density=density)
    for i, d in enumerate(progress_iter(domains, desc="TIES Trim", total=len(domains), stage="TIES")):
        vals = torch.abs(M[i])
        thr = torch.quantile(vals, 1.0 - density)
        keep = vals >= thr
        trimmed[i, keep] = M[i, keep]
        thresholds[d] = float(thr.item())
        retained_counts[d] = int(keep.sum().item())
    raw_consensus = torch.sign(torch.sum(torch.sign(M), dim=0))
    any_raw = torch.any(M != 0, dim=0)
    raw_disagree = (torch.sign(M) != raw_consensus.unsqueeze(0)) & (M != 0) & any_raw.unsqueeze(0)
    raw_interference = float(raw_disagree.sum().item() / max(1, ((M != 0) & any_raw.unsqueeze(0)).sum().item()))
    consensus = torch.sign(torch.sum(torch.sign(trimmed), dim=0))
    active = torch.any(trimmed != 0, dim=0)
    post_trim_disagree = (torch.sign(trimmed) != consensus.unsqueeze(0)) & (trimmed != 0) & active.unsqueeze(0)
    post_trim_interference = float(post_trim_disagree.sum().item() / max(1, ((trimmed != 0) & active.unsqueeze(0)).sum().item()))
    elected = trimmed.clone()
    discard_counts = {}
    for i, d in enumerate(progress_iter(domains, desc="TIES Elect Sign", total=len(domains), stage="TIES")):
        discard = (torch.sign(elected[i]) != consensus) & (elected[i] != 0) & active
        discard_counts[d] = int(discard.sum().item())
        elected[i, discard] = 0.0
    post_elect_disagree = (torch.sign(elected) != consensus.unsqueeze(0)) & (elected != 0) & active.unsqueeze(0)
    post_elect_interference = float(post_elect_disagree.sum().item() / max(1, ((elected != 0) & active.unsqueeze(0)).sum().item()))
    nonzero = elected != 0
    denom = nonzero.sum(dim=0).clamp_min(1)
    merged = elected.sum(dim=0) / denom
    merged[~torch.any(nonzero, dim=0)] = 0.0
    naive = M.mean(dim=0)
    cos_ties = {d: float(torch.nn.functional.cosine_similarity(merged, task_vectors[d], dim=0).item()) for d in domains}
    cos_naive = {d: float(torch.nn.functional.cosine_similarity(naive, task_vectors[d], dim=0).item()) for d in domains}
    log_event("TIES", "merge done", raw_interference=round(raw_interference, 6), post_trim=round(post_trim_interference, 6), post_elect=round(post_elect_interference, 6), nonzero_ties=int((merged != 0).sum().item()))
    return {
        "domains": domains,
        "D": int(D),
        "density": float(density),
        "thresholds": thresholds,
        "retained_counts": retained_counts,
        "discard_counts": discard_counts,
        "raw_interference": raw_interference,
        "post_trim_interference": post_trim_interference,
        "post_elect_interference": post_elect_interference,
        "merged_vector": merged,
        "naive_vector": naive,
        "cos_ties": cos_ties,
        "cos_naive": cos_naive,
        "ties_norm": float(torch.linalg.norm(merged).item()),
        "naive_norm": float(torch.linalg.norm(naive).item()),
        "nonzero_ties": int((merged != 0).sum().item()),
    }

def create_executable_ultralytics_checkpoint(base_checkpoint: Path,
                                             merged_state_float: Dict[str, "torch.Tensor"],
                                             out_path: Path,
                                             metadata: Optional[Dict[str, Any]] = None) -> Path:
    """Insert a merged float state into a copy of the base Ultralytics checkpoint and save it as loadable .pt."""
    if torch is None:
        raise RuntimeError("PyTorch is required to create executable TIES checkpoint.")
    base_checkpoint = Path(base_checkpoint)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    ckpt = load_checkpoint_any(base_checkpoint)
    if not isinstance(ckpt, dict):
        raise ValueError("Expected Ultralytics checkpoint dictionary for executable checkpoint construction.")
    ckpt = copy.deepcopy(ckpt)
    model = get_checkpoint_model_module(ckpt)
    full_sd = model.state_dict()
    replaced = 0
    for k, v in merged_state_float.items():
        if k in full_sd and tuple(full_sd[k].shape) == tuple(v.shape):
            full_sd[k] = v.detach().cpu().to(dtype=full_sd[k].dtype)
            replaced += 1
    missing, unexpected = model.load_state_dict(full_sd, strict=False)
    ckpt["model"] = model.float().cpu()
    # Ultralytics uses ema preferentially if present; set to None so the merged model is used.
    ckpt["ema"] = None
    ckpt["updates"] = ckpt.get("updates", 0)
    ckpt["atlas_fn_ties_metadata"] = _json_safe(metadata or {})
    ckpt["train_args"] = ckpt.get("train_args", {})
    if isinstance(ckpt["train_args"], dict):
        ckpt["train_args"]["atlas_fn_ties_merged"] = True
    torch.save(ckpt, out_path)
    log_event("TIES", "saved executable merged checkpoint", path=out_path, replaced_float_tensors=replaced, missing=len(missing), unexpected=len(unexpected))
    return out_path

print("TIES utilities loaded, including executable checkpoint construction.")


TIES utilities loaded, including executable checkpoint construction.


In [13]:
# ============================================================
# HOTFIX: pickle-safe YOLO26 shim for executable TIES checkpoint
# Run once, then rerun Section 9 onward.
# ============================================================

import __main__
import copy
import torch
import torch.nn as nn
from pathlib import Path
from typing import Any, Dict, Optional

class SPPF_YOLO26_Compatible(nn.Module):
    """Top-level YOLO26-compatible SPPF block for pickle-safe checkpoint saving."""
    def __init__(self, *args, **kwargs):
        super().__init__()
        self.cv1 = kwargs.get("cv1", nn.Identity())
        self.cv2 = kwargs.get("cv2", nn.Identity())
        self.m = kwargs.get("m", nn.MaxPool2d(kernel_size=5, stride=1, padding=2))

    def forward(self, x):
        y0 = self.cv1(x)
        y1 = self.m(y0)
        y2 = self.m(y1)
        y3 = self.m(y2)
        return self.cv2(torch.cat((y0, y1, y2, y3), dim=1))

SPPF_YOLO26_Compatible.__module__ = "__main__"
SPPF_YOLO26_Compatible.__qualname__ = "SPPF_YOLO26_Compatible"
setattr(__main__, "SPPF_YOLO26_Compatible", SPPF_YOLO26_Compatible)
globals()["SPPF_YOLO26_Compatible"] = SPPF_YOLO26_Compatible

def _patch_sppf_class_for_pickle(obj) -> int:
    if obj is None:
        return 0
    patched = 0
    modules = []
    if hasattr(obj, "modules"):
        try:
            modules = list(obj.modules())
        except Exception:
            modules = []
    for m in modules:
        cls = m.__class__
        if "SPPF_YOLO26_Compatible" in getattr(cls, "__name__", "") or "SPPF_YOLO26_Compatible" in getattr(cls, "__qualname__", ""):
            cls.__module__ = "__main__"
            cls.__qualname__ = "SPPF_YOLO26_Compatible"
            cls.__name__ = "SPPF_YOLO26_Compatible"
            setattr(__main__, "SPPF_YOLO26_Compatible", cls)
            globals()["SPPF_YOLO26_Compatible"] = cls
            patched += 1
    return patched

def _patch_checkpoint_for_pickle(ckpt: Any) -> int:
    patched = 0
    if isinstance(ckpt, dict):
        for key in ("model", "ema"):
            patched += _patch_sppf_class_for_pickle(ckpt.get(key))
    else:
        patched += _patch_sppf_class_for_pickle(ckpt)
    return patched

def register_yolo26_compatibility_shims(verbose: bool = True) -> bool:
    setattr(__main__, "SPPF_YOLO26_Compatible", globals()["SPPF_YOLO26_Compatible"])
    for mod_name in ["ultralytics.nn.modules.block", "ultralytics.nn.tasks", "ultralytics.nn.modules"]:
        try:
            mod = __import__(mod_name, fromlist=["dummy"])
            setattr(mod, "SPPF_YOLO26_Compatible", getattr(__main__, "SPPF_YOLO26_Compatible"))
        except Exception:
            pass
    try:
        import torch.serialization as serialization
        if hasattr(serialization, "add_safe_globals"):
            serialization.add_safe_globals([getattr(__main__, "SPPF_YOLO26_Compatible")])
    except Exception:
        pass
    if verbose:
        try:
            log_event("COMPAT", "pickle-safe YOLO26 shim registered", class_name="SPPF_YOLO26_Compatible")
        except Exception:
            print("pickle-safe YOLO26 shim registered")
    return True

def create_executable_ultralytics_checkpoint(
    base_checkpoint: Path,
    merged_state_float: Dict[str, "torch.Tensor"],
    out_path: Path,
    metadata: Optional[Dict[str, Any]] = None,
) -> Optional[Path]:
    base_checkpoint = Path(base_checkpoint)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    register_yolo26_compatibility_shims(verbose=False)

    ckpt = load_checkpoint_any(base_checkpoint)
    if not isinstance(ckpt, dict):
        raise ValueError("Expected Ultralytics checkpoint dictionary for executable checkpoint construction.")

    ckpt = copy.deepcopy(ckpt)
    _patch_checkpoint_for_pickle(ckpt)

    model = get_checkpoint_model_module(ckpt)
    full_sd = model.state_dict()

    replaced = 0
    for k, v in merged_state_float.items():
        if k in full_sd and tuple(full_sd[k].shape) == tuple(v.shape):
            full_sd[k] = v.detach().cpu().to(dtype=full_sd[k].dtype)
            replaced += 1

    missing, unexpected = model.load_state_dict(full_sd, strict=False)
    _patch_sppf_class_for_pickle(model)

    ckpt["model"] = model.float().cpu()
    ckpt["ema"] = None
    ckpt["updates"] = ckpt.get("updates", 0)
    ckpt["atlas_fn_ties_metadata"] = _json_safe(metadata or {})
    ckpt["train_args"] = ckpt.get("train_args", {})
    if isinstance(ckpt["train_args"], dict):
        ckpt["train_args"]["atlas_fn_ties_merged"] = True
        ckpt["train_args"]["atlas_fn_checkpoint_type"] = "ties_executable_full_checkpoint"

    _patch_checkpoint_for_pickle(ckpt)
    torch.save(ckpt, out_path)

    log_event(
        "TIES",
        "saved executable merged checkpoint",
        path=out_path,
        replaced_float_tensors=replaced,
        missing=len(missing),
        unexpected=len(unexpected),
    )
    return out_path

print("✅ Pickle-safe executable TIES checkpoint hotfix installed. Rerun Section 9 onward.")


✅ Pickle-safe executable TIES checkpoint hotfix installed. Rerun Section 9 onward.


In [14]:

# Runtime safety guard: make sure progress_iter is the final safe definition even
# when this notebook is partially rerun from Section 9 onward.
try:
    from tqdm.auto import tqdm as _atlas_tqdm_runtime
except Exception:
    _atlas_tqdm_runtime = None

def progress_iter(iterable, desc: str = "", total=None, stage: str = "progress", **tqdm_kwargs):
    desc = tqdm_kwargs.pop("desc", desc)
    total = tqdm_kwargs.pop("total", total)
    if total is None:
        try:
            total = len(iterable)
        except Exception:
            total = None
    tqdm_kwargs.setdefault("leave", True)
    tqdm_kwargs.setdefault("mininterval", getattr(CFG, "progress_mininterval_sec", 1.0))
    if getattr(CFG, "progress_enabled", True) and _atlas_tqdm_runtime is not None:
        return _atlas_tqdm_runtime(iterable, total=total, desc=desc, **tqdm_kwargs)
    try:
        log_event(stage, f"{desc}: progress bar unavailable", total=total)
    except Exception:
        pass
    return iterable

# ============================================================
# 9. Run complete real-domain TIES-Merging across FP and UCL Find checkpoints
# ============================================================
def resolve_domain_checkpoints(cfg: ExtendedEvidenceConfig = CFG) -> Dict[str, Path]:
    out = {}
    allowed = {normalise_domain_label(d) for d in getattr(cfg, "ties_domains", getattr(cfg, "primary_external_domains", ("FP", "UCL")))}
    source = normalise_domain_label(getattr(cfg, "training_source_domain", "HC18"))
    include_source = bool(getattr(cfg, "include_training_source_domain_as_anchor", False))
    if include_source:
        allowed.add(source)
    # Prefer trained checkpoints from this run, then explicit CFG.domain_checkpoints.
    for d, ck in trained_domain_checkpoints.items() if "trained_domain_checkpoints" in globals() else []:
        nd = normalise_domain_label(d)
        if nd in allowed and Path(ck).exists():
            out[nd] = Path(ck)
    for d, ck in cfg.domain_checkpoints.items():
        nd = normalise_domain_label(d)
        if nd in allowed and nd not in out and Path(ck).exists():
            out[nd] = Path(ck)
    return out

def run_ties_from_checkpoints(base_checkpoint: Path, domain_ckpts: Dict[str, Path], density: float = 0.20) -> Tuple[pd.DataFrame, Optional[Dict[str, Any]], Optional[Dict[str, Any]]]:
    if torch is None:
        raise RuntimeError("PyTorch not available; cannot run required TIES experiment.")
    if not Path(base_checkpoint).exists():
        raise FileNotFoundError(f"Base checkpoint not found: {base_checkpoint}")
    if len(domain_ckpts) < 2:
        raise RuntimeError(f"Complete Experiment 2 requires at least two real domain-specific checkpoints. Found: {domain_ckpts}")
    log_event("TIES", "loading base checkpoint", checkpoint=base_checkpoint)
    base_obj = load_checkpoint_any(base_checkpoint)
    base_sd_raw = extract_float_state_dict(base_obj)
    site_sd_raw = {}
    for d, p in progress_iter(list(domain_ckpts.items()), desc="load domain checkpoints", total=len(domain_ckpts), stage="TIES"):
        log_event("TIES", "loading domain checkpoint", domain=d, checkpoint=p)
        site_sd_raw[d] = extract_float_state_dict(load_checkpoint_any(p))
    keys, base_sd, site_sds = align_state_dicts(base_sd_raw, site_sd_raw)
    if not keys:
        raise RuntimeError("No common float tensors found across base and domain checkpoints for TIES.")
    tensor_count = int(len(keys))
    parameter_count = int(sum(base_sd[k].numel() for k in keys))
    log_event("TIES", "aligned state dicts", keys=tensor_count, parameters=parameter_count)
    base_vec = flatten_state_dict(base_sd, keys)
    task_vectors = {}
    for d in progress_iter(list(site_sds.keys()), desc="build task vectors", total=len(site_sds), stage="TIES"):
        task_vectors[d] = flatten_state_dict(site_sds[d], keys) - base_vec
    log_event("TIES", "task vectors built", domains=list(task_vectors.keys()), D=int(base_vec.numel()), density=float(density))
    res = ties_merge_vectors(task_vectors, density=density)
    rows = []
    for d in res["domains"]:
        rows.append({
            "domain": d,
            "checkpoint": str(domain_ckpts[d]),
            "checkpoint_sha256": sha256_file(domain_ckpts[d]),
            "D": res["D"],
            "density": density,
            "trim_threshold": res["thresholds"][d],
            "retained_after_trim": res["retained_counts"][d],
            "discarded_by_elect_sign": res["discard_counts"][d],
            "cosine_ties_vs_site": res["cos_ties"][d],
            "cosine_naive_vs_site": res["cos_naive"][d],
            "raw_interference": res["raw_interference"],
            "post_trim_interference": res["post_trim_interference"],
            "post_elect_interference": res["post_elect_interference"],
            "ties_norm": res["ties_norm"],
            "naive_norm": res["naive_norm"],
            "nonzero_ties": res["nonzero_ties"],
        })
    summary = pd.DataFrame(rows)
    merged_state = unflatten_to_state_dict(base_vec + res["merged_vector"], base_sd, keys)
    tag = f"{density:.2f}".replace(".", "")
    merged_state_path = CFG.output_root / "checkpoints" / f"ties_merged_state_density_{density:.2f}.pt"
    torch.save({"state_dict": merged_state, "keys": keys, "density": density, "summary": rows}, merged_state_path)
    log_event("TIES", "saved generic merged state", path=merged_state_path)
    executable_path = None
    if getattr(CFG, "create_executable_ties_checkpoint", True):
        executable_path = CFG.output_root / "checkpoints" / f"ties_merged_full_checkpoint_density_{density:.2f}.pt"
        executable_path = create_executable_ultralytics_checkpoint(
            base_checkpoint=base_checkpoint,
            merged_state_float=merged_state,
            out_path=executable_path,
            metadata={
                "density": density,
                "domains": list(domain_ckpts.keys()),
                "base_checkpoint": str(base_checkpoint),
                "domain_checkpoints": {k: str(v) for k, v in domain_ckpts.items()},
                "D": res["D"],
                "raw_interference": res["raw_interference"],
                "post_trim_interference": res["post_trim_interference"],
                "post_elect_interference": res["post_elect_interference"],
            },
        )
    artifacts = {"merged_state_path": merged_state_path, "executable_checkpoint_path": executable_path, "keys": keys}
    return summary, res, artifacts

domain_ckpts = resolve_domain_checkpoints(CFG)
print("Resolved domain checkpoints for real-domain TIES:", {k: str(v) for k, v in domain_ckpts.items()})

if CFG.run_ties_merge:
    # Late refresh: if base checkpoint was restored or copied after Section 0C, standardise it now.
    try:
        refreshed_base = discover_base_find_checkpoint(WORKSPACE, restore_checkpoint_package_if_missing=True)
        if refreshed_base is not None and Path(refreshed_base).exists():
            CFG.base_find_checkpoint = Path(refreshed_base)
    except Exception as e:
        log_event("TIES", "late base checkpoint refresh failed before TIES", error=repr(e))
    if not Path(CFG.base_find_checkpoint).exists():
        msg = f"Base Find checkpoint unavailable for TIES: {CFG.base_find_checkpoint}."
        if CFG.require_complete_exp2:
            raise RuntimeError(msg + " Restore the checkpoint-training package first.")
        print("TIES not run:", msg)
        pd.DataFrame([{
            "stage": "TIES",
            "status": "skipped",
            "reason": "base_find_checkpoint_missing",
            "base_find_checkpoint": str(CFG.base_find_checkpoint),
            "domain_checkpoints_found": json.dumps({k: str(v) for k, v in domain_ckpts.items()}),
        }]).to_csv(CFG.output_root / "tables" / "exp2_ties_skip_audit.csv", index=False)
        ties_summary = pd.DataFrame(); ties_res = None; ties_artifacts = None
    elif len(domain_ckpts) < 2:
        msg = f"Complete Experiment 2 requires at least two real domain checkpoints for {CFG.ties_domains}; found {domain_ckpts}."
        if CFG.require_complete_exp2:
            raise RuntimeError(msg + " Ensure CFG.run_yolo_training=True and yolo_training_strategy includes 'domain_specific'.")
        print("TIES not run:", msg)
        pd.DataFrame([{
            "stage": "TIES",
            "status": "skipped",
            "reason": "less_than_two_domain_checkpoints",
            "base_find_checkpoint": str(CFG.base_find_checkpoint),
            "domain_checkpoints_found": json.dumps({k: str(v) for k, v in domain_ckpts.items()}),
        }]).to_csv(CFG.output_root / "tables" / "exp2_ties_skip_audit.csv", index=False)
        ties_summary = pd.DataFrame(); ties_res = None; ties_artifacts = None
    else:
        ties_summary, ties_res, ties_artifacts = run_ties_from_checkpoints(CFG.base_find_checkpoint, domain_ckpts, density=float(CFG.ties_density_primary))
        ties_summary.to_csv(CFG.output_root / "tables" / "exp2_ties_summary_density_020.csv", index=False)
        show_df(ties_summary, "Experiment 2 TIES summary")
elif CFG.allow_demo_vectors_if_checkpoints_missing and torch is not None:
    raise RuntimeError("v11 submission-grade mode does not allow demo-vector TIES claims. Set CFG.run_ties_merge=True with real checkpoints.")
else:
    ties_summary = pd.DataFrame(); ties_res = None; ties_artifacts = None
    print("TIES disabled by configuration.")



Resolved domain checkpoints for real-domain TIES: {'FP': '/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/find_FP.pt', 'UCL': '/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/find_UCL.pt'}
[12:28:18 + 1010.4s] [TIES] loading base checkpoint | checkpoint=/content/atlas_fn_workspace/Training_Runs/BrainLocator_YOLO26n_MuSGD/weights/best.pt


load domain checkpoints:   0%|          | 0/2 [00:00<?, ?it/s]

[12:28:18 + 1010.4s] [TIES] loading domain checkpoint | domain=FP, checkpoint=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/find_FP.pt
[12:28:18 + 1010.5s] [TIES] loading domain checkpoint | domain=UCL, checkpoint=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/find_UCL.pt
[12:28:18 + 1010.5s] [TIES] aligned state dicts | keys=594, parameters=2523182


flatten state_dict:   0%|          | 0/594 [00:00<?, ?it/s]

build task vectors:   0%|          | 0/2 [00:00<?, ?it/s]

...

TIES Elect Sign:   0%|          | 0/2 [00:00<?, ?it/s]

[12:28:19 + 1011.2s] [TIES] merge done | raw_interference=0.462679, post_trim=0.09135, post_elect=0.0, nonzero_ties=839070


unflatten merged state:   0%|          | 0/594 [00:00<?, ?it/s]

[12:28:19 + 1011.3s] [TIES] saved generic merged state | path=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/ties_merged_state_density_0.20.pt
[12:28:19 + 1011.5s] [TIES] saved executable merged checkpoint | path=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/ties_merged_full_checkpoint_density_0.20.pt, replaced_float_tensors=594, missing=0, unexpected=0
[12:28:19 + 1011.5s] [DATA] Experiment 2 TIES summary | rows=2, columns=16


,domain,checkpoint,checkpoint_sha256,D,density,trim_threshold,retained_after_trim,discarded_by_elect_sign,cosine_ties_vs_site,cosine_naive_vs_site,raw_interference,post_trim_interference,post_elect_interference,ties_norm,naive_norm,nonzero_ties
0,FP,/content/atlas_fn_workspace/outputs/extended_e...,cc6e8b92e3fe27b2469cfb5b94b1637e9f3e79d086ebac...,2523182,0.2,0.008133,505275,46128,0.858811,0.962190,0.462679,0.09135,0.0,33.027504,33.853828,839070
1,UCL,/content/atlas_fn_workspace/outputs/extended_e...,32a1a9e3c96899a7a2b4cfc5822aad703a8f520f703891...,2523182,0.2,0.002100,504645,46128,0.757223,0.705093,0.462679,0.09135,0.0,33.027504,33.853828,839070


In [15]:
# ============================================================
# 10. TIES density sensitivity using the same real domain checkpoints
# ============================================================
def ties_density_sweep_from_resolved(base_checkpoint: Path, domain_ckpts: Dict[str, Path], densities=None) -> pd.DataFrame:
    if densities is None:
        densities = getattr(CFG, "ties_density_grid", (0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40))
    if torch is None or len(domain_ckpts) < 2 or not Path(base_checkpoint).exists():
        return pd.DataFrame()
    base_sd_raw = extract_float_state_dict(load_checkpoint_any(base_checkpoint))
    site_sd_raw = {d: extract_float_state_dict(load_checkpoint_any(p)) for d, p in progress_iter(list(domain_ckpts.items()), desc="load sweep checkpoints", total=len(domain_ckpts), stage="TIES")}
    keys, base_sd, site_sds = align_state_dicts(base_sd_raw, site_sd_raw)
    base_vec = flatten_state_dict(base_sd, keys)
    task_vectors = {d: flatten_state_dict(site_sds[d], keys) - base_vec for d in site_sds}
    rows = []
    for den in progress_iter(list(densities), desc="TIES density sweep", total=len(list(densities)), stage="TIES"):
        res = ties_merge_vectors(task_vectors, density=float(den))
        rows.append({
            "density": float(den),
            "D": res["D"],
            "raw_interference": res["raw_interference"],
            "post_trim_interference": res["post_trim_interference"],
            "post_elect_interference": res["post_elect_interference"],
            "nonzero_ties": res["nonzero_ties"],
            "ties_norm": res["ties_norm"],
            "mean_cos_ties": float(np.mean(list(res["cos_ties"].values()))),
            "mean_cos_naive": float(np.mean(list(res["cos_naive"].values()))),
            "mean_discarded": float(np.mean(list(res["discard_counts"].values()))),
        })
    return pd.DataFrame(rows)

if CFG.run_ties_merge and len(domain_ckpts) >= 2 and CFG.base_find_checkpoint.exists():
    ties_density = ties_density_sweep_from_resolved(CFG.base_find_checkpoint, domain_ckpts)
    ties_density.to_csv(CFG.output_root / "tables" / "exp2_ties_density_sweep.csv", index=False)
    show_df(ties_density, "Experiment 2 TIES density sweep")
else:
    ties_density = pd.DataFrame()
    print("Density sweep skipped.")



load sweep checkpoints:   0%|          | 0/2 [00:00<?, ?it/s]

flatten state_dict:   0%|          | 0/594 [00:00<?, ?it/s]

flatten state_dict:   0%|          | 0/594 [00:00<?, ?it/s]

flatten state_dict:   0%|          | 0/594 [00:00<?, ?it/s]

TIES density sweep:   0%|          | 0/7 [00:00<?, ?it/s]

...

[12:28:23 + 1015.1s] [TIES] merge done | raw_interference=0.462679, post_trim=0.131247, post_elect=0.0, nonzero_ties=1162963
[12:28:23 + 1015.2s] [TIES] merge start | domains=['FP', 'UCL'], D=2523182, density=0.4


TIES Trim:   0%|          | 0/2 [00:00<?, ?it/s]

TIES Elect Sign:   0%|          | 0/2 [00:00<?, ?it/s]

[12:28:23 + 1015.7s] [TIES] merge done | raw_interference=0.462679, post_trim=0.173192, post_elect=0.0, nonzero_ties=1416836
[12:28:23 + 1015.7s] [DATA] Experiment 2 TIES density sweep | rows=7, columns=10


,density,D,raw_interference,post_trim_interference,post_elect_interference,nonzero_ties,ties_norm,mean_cos_ties,mean_cos_naive,mean_discarded
0,0.05,2523182,0.462679,0.056657,0.0,224349,32.667084,0.807912,0.833642,7178.0
1,0.10,2523182,0.462679,0.059605,0.0,446585,32.877190,0.807567,0.833642,15073.0
2,0.15,2523182,0.462679,0.074079,0.0,651902,32.952168,0.807836,0.833642,28100.0
3,0.20,2523182,0.462679,0.091350,0.0,839070,33.027504,0.808017,0.833642,46128.0
4,0.25,2523182,0.462679,0.110965,0.0,1011174,32.982002,0.808979,0.833642,70211.0
5,0.30,2523182,0.462679,0.131247,0.0,1162963,32.989349,0.809476,0.833642,99522.0
6,0.40,2523182,0.462679,0.173192,0.0,1416836,32.943989,0.810658,0.833642,175032.0


## 7. Optional evaluation matrix of base, site-specialist and TIES-merged models

This cell evaluates any available checkpoints on all external test rows. It produces a cross-domain matrix of error and availability. If the TIES output has been converted to a full Ultralytics checkpoint, add its path to `EVAL_CHECKPOINTS` before running this cell.


In [16]:
# ============================================================
# 11. Complete Experiment 2 evaluation matrix: base, specialists, pooled, TIES-merged
# ============================================================
def _ledger_with_renamed_model(ledger: pd.DataFrame, old_name: str, new_name: str) -> pd.DataFrame:
    if ledger is None or ledger.empty or "model_name" not in ledger.columns:
        return pd.DataFrame()
    sub = ledger[ledger["model_name"].eq(old_name)].copy()
    if len(sub):
        sub["model_name"] = new_name
    return sub

EVAL_CHECKPOINTS: Dict[str, Path] = {}
if CFG.base_find_checkpoint.exists():
    EVAL_CHECKPOINTS["base"] = CFG.base_find_checkpoint
for d, p in domain_ckpts.items() if "domain_ckpts" in globals() else []:
    if Path(p).exists():
        EVAL_CHECKPOINTS[f"specialist_{d}"] = Path(p)
if "trained_domain_checkpoints" in globals() and "ALL_external" in trained_domain_checkpoints and Path(trained_domain_checkpoints["ALL_external"]).exists():
    EVAL_CHECKPOINTS["pooled_external"] = Path(trained_domain_checkpoints["ALL_external"])
if "ties_artifacts" in globals() and ties_artifacts and ties_artifacts.get("executable_checkpoint_path"):
    ties_ck = Path(ties_artifacts["executable_checkpoint_path"])
    if ties_ck.exists():
        EVAL_CHECKPOINTS["ties_merged_density_020"] = ties_ck

if len(ext_df) and EVAL_CHECKPOINTS and CFG.run_find_inference:
    eval_ledgers = []
    test_df = get_primary_exp1_evaluation_df(ext_df, CFG)
    if test_df.empty:
        raise RuntimeError("No v11 primary external test cohort for checkpoint evaluation matrix.")
    for name, ck in progress_iter(list(EVAL_CHECKPOINTS.items()), desc="cross-domain checkpoints", total=len(EVAL_CHECKPOINTS), stage="EVAL2"):
        # Reuse the base zero-shot ledger from Experiment 1 to avoid redundant inference.
        if name == "base" and "exp1_ledger" in globals() and len(exp1_ledger):
            reused = _ledger_with_renamed_model(exp1_ledger, "ATLAS_base_find_zero_shot", "base")
            if len(reused) == len(test_df):
                log_event("EVAL2", "reused Experiment 1 base ledger", name=name, frames=len(reused))
                eval_ledgers.append(reused)
                continue
        log_event("EVAL2", "evaluating checkpoint", name=name, checkpoint=ck, frames=len(test_df))
        led = run_find_measure_external(test_df, ck, name)
        eval_ledgers.append(led)
    exp2_eval_ledger = pd.concat(eval_ledgers, ignore_index=True) if eval_ledgers else pd.DataFrame()
    exp2_eval_summary = summarize_external_metrics(exp2_eval_ledger, group_cols=("model_name", "domain")) if len(exp2_eval_ledger) else pd.DataFrame()
    exp2_eval_ledger.to_csv(CFG.output_root / "tables" / "exp2_checkpoint_evaluation_ledger.csv", index=False)
    exp2_eval_summary.to_csv(CFG.output_root / "tables" / "exp2_checkpoint_evaluation_summary.csv", index=False)
    show_df(exp2_eval_summary, "Experiment 2 checkpoint evaluation summary")
else:
    exp2_eval_ledger = pd.DataFrame()
    exp2_eval_summary = pd.DataFrame()
    print("Checkpoint evaluation matrix skipped.")



[12:28:24 + 1015.8s] [DATA] v11 Experiment 1 available split audit | rows=6, columns=4


,domain,split,rows,subjects
0,FP,test,880,460
1,FP,train,662,395
2,FP,val,95,54
3,UCL,test,55,15
4,UCL,train,73,28
5,UCL,val,31,4


[12:28:24 + 1015.8s] [EVAL] selected primary external evaluation cohort | split=test, rows=935, domains=['FP', 'UCL']


cross-domain checkpoints:   0%|          | 0/5 [00:00<?, ?it/s]

[12:28:24 + 1015.8s] [EVAL2] reused Experiment 1 base ledger | name=base, frames=935
[12:28:24 + 1015.8s] [EVAL2] evaluating checkpoint | name=specialist_FP, checkpoint=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/find_FP.pt, frames=935
[12:28:24 + 1015.8s] [COMPAT] pickle-safe YOLO26 shim registered | class_name=SPPF_YOLO26_Compatible
[12:28:24 + 1015.8s] [INFER] loading YOLO checkpoint | checkpoint=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/find_FP.pt
[12:28:24 + 1015.9s] [INFER] YOLO checkpoint loaded | checkpoint=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/find_FP.pt
[12:28:24 + 1015.9s] [INFER] evaluation start | model=specialist_FP, rows=935, checkpoint=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/find_FP.pt


...

infer pooled_external:   0%|          | 0/935 [00:00<?, ?it/s]

[12:29:10 + 1062.6s] [INFER] heartbeat | model=pooled_external, processed=250, total=935, detected=248
[12:29:16 + 1068.0s] [INFER] heartbeat | model=pooled_external, processed=500, total=935, detected=498
[12:29:21 + 1073.3s] [INFER] heartbeat | model=pooled_external, processed=750, total=935, detected=747
[12:29:26 + 1077.8s] [INFER] evaluation done | model=pooled_external, rows=935, detected=931, elapsed_sec=20.63
[12:29:26 + 1077.8s] [EVAL2] evaluating checkpoint | name=ties_merged_density_020, checkpoint=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/ties_merged_full_checkpoint_density_0.20.pt, frames=935
[12:29:26 + 1077.8s] [COMPAT] pickle-safe YOLO26 shim registered | class_name=SPPF_YOLO26_Compatible
[12:29:26 + 1077.8s] [INFER] loading YOLO checkpoint | checkpoint=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/checkpoints/ties_merged_full_checkpoint_density_0.20.pt
[12:29:26 + 1077.9s] [INFER] YOLO checkpoint loaded | checkpoint

infer ties_merged_density_020:   0%|          | 0/935 [00:00<?, ?it/s]

[12:29:31 + 1083.3s] [INFER] heartbeat | model=ties_merged_density_020, processed=250, total=935, detected=249
[12:29:36 + 1088.5s] [INFER] heartbeat | model=ties_merged_density_020, processed=500, total=935, detected=498
[12:29:42 + 1093.9s] [INFER] heartbeat | model=ties_merged_density_020, processed=750, total=935, detected=748
[12:29:46 + 1098.3s] [INFER] evaluation done | model=ties_merged_density_020, rows=935, detected=931, elapsed_sec=20.39
[12:29:46 + 1098.7s] [DATA] Experiment 2 checkpoint evaluation summary | rows=10, columns=15


,model_name,domain,n,detected,detection_rate,mean_iou,median_iou,bpd_mae_mm,bpd_p90_mm,ofd_mae_mm,hc_mae_mm,hc_mae_pct,broad_precision,latency_mean_ms,latency_p90_ms
0,base,FP,880,879,99.886364,0.912621,0.920856,0.927250,1.737465,1.791539,3.129525,1.243802,91.240046,21.531561,23.256507
1,base,UCL,55,55,100.000000,0.884449,0.896612,1.936474,3.649432,3.715841,7.930714,2.843074,61.818182,32.098379,47.529990
2,pooled_external,FP,880,877,99.659091,0.942182,0.959289,2.535195,4.143991,2.595300,7.308958,3.017119,38.198404,21.189293,23.147946
3,pooled_external,UCL,55,54,98.181818,0.931548,0.952142,3.133822,5.452607,2.492153,7.372718,2.787514,25.925926,30.269588,38.484187
4,specialist_FP,FP,880,878,99.772727,0.944965,0.961909,2.627878,4.320593,3.069473,8.463212,3.546064,39.407745,21.171214,22.808577
5,specialist_FP,UCL,55,55,100.000000,0.927343,0.953859,2.831319,5.421856,3.116429,7.664243,2.849313,45.454545,29.709865,38.032001
6,specialist_UCL,FP,880,879,99.886364,0.928538,0.949015,3.756671,5.742895,4.429361,12.784269,5.387501,6.257110,21.176757,23.155340
7,specialist_UCL,UCL,55,55,100.000000,0.923151,0.943774,3.383688,5.536753,2.942949,8.770826,3.167935,20.000000,29.423001,37.669614
8,ties_merged_density_020,FP,880,878,99.772727,0.937023,0.950202,1.385925,2.644908,2.268000,4.897215,2.136845,80.296128,20.981183,22.748198
9,ties_merged_density_020,UCL,55,53,96.363636,0.912947,0.932921,1.166811,2.107297,2.501548,4.382185,1.651059,88.679245,29.385952,37.836534


## 8. Figures for ACML evidence package

The notebook generates publication-ready figures from the ledgers:

- domain-wise BPD MAE
- cross-domain model matrix when multiple checkpoints are evaluated
- TIES interference trajectory and density sensitivity
- Bland–Altman-ready table export


In [17]:
# ============================================================
# 12. Figures and table exports
# ============================================================
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 140, "savefig.dpi": 300})

def save_bar_metric(df: pd.DataFrame, x: str, y: str, hue: Optional[str], title: str, out_path: Path):
    if df.empty or y not in df.columns:
        print("No figure data for", title)
        return
    fig, ax = plt.subplots(figsize=(9, 4.8))
    if hue and hue in df.columns:
        piv = df.pivot_table(index=x, columns=hue, values=y, aggfunc="mean")
        piv.plot(kind="bar", ax=ax)
    else:
        ax.bar(df[x].astype(str), df[y])
    ax.set_title(title)
    ax.set_ylabel(y)
    ax.set_xlabel(x)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(out_path)
    plt.close(fig)
    log_event("FIG", "saved figure", path=out_path)

if len(exp1_summary):
    save_bar_metric(exp1_summary[exp1_summary["split"].isin(["test", "val", "validation"])], "domain", "bpd_mae_mm", "model_name", "Experiment 1: External BPD MAE by domain", CFG.output_root / "figures" / "exp1_bpd_mae_by_domain.png")
    save_bar_metric(exp1_summary[exp1_summary["split"].isin(["test", "val", "validation"])], "domain", "hc_mae_pct", "model_name", "Experiment 1: External HC MAE (%) by domain", CFG.output_root / "figures" / "exp1_hc_mae_pct_by_domain.png")

if len(exp2_eval_summary):
    save_bar_metric(exp2_eval_summary, "domain", "bpd_mae_mm", "model_name", "Experiment 2: Cross-domain checkpoint matrix (BPD MAE)", CFG.output_root / "figures" / "exp2_cross_domain_bpd_mae.png")

if 'ties_density' in globals() and len(ties_density):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(ties_density["density"], 100*ties_density["post_trim_interference"], marker="o", label="After Trim")
    ax.plot(ties_density["density"], 100*ties_density["post_elect_interference"], marker="o", label="After Elect Sign")
    ax.set_xlabel("TIES density")
    ax.set_ylabel("Interference (%)")
    ax.set_title("Experiment 2: TIES density sensitivity")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    out = CFG.output_root / "figures" / "exp2_ties_density_sensitivity.png"
    fig.savefig(out)
    plt.close(fig)
    log_event("FIG", "saved figure", path=out)

# Bland-Altman export table for downstream plotting in R/GraphPad/Python.
if len(exp1_ledger):
    ba = exp1_ledger.copy()
    ba["bpd_mean_mm"] = (ba["pred_bpd_mm"] + ba["ref_bpd_mm"]) / 2
    ba["bpd_diff_mm"] = ba["pred_bpd_mm"] - ba["ref_bpd_mm"]
    ba["hc_mean_mm"] = (ba["pred_hc_mm"] + ba["ref_hc_mm"]) / 2
    ba["hc_diff_pct_signed"] = 100.0 * (ba["pred_hc_mm"] - ba["ref_hc_mm"]) / ba["ref_hc_mm"]
    ba.to_csv(CFG.output_root / "tables" / "exp1_bland_altman_export.csv", index=False)
    log_event("EXPORT", "saved Bland-Altman export", path=CFG.output_root / "tables" / "exp1_bland_altman_export.csv")



[12:29:47 + 1099.1s] [FIG] saved figure | path=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/figures/exp1_bpd_mae_by_domain.png
[12:29:47 + 1099.4s] [FIG] saved figure | path=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/figures/exp1_hc_mae_pct_by_domain.png
[12:29:48 + 1099.8s] [FIG] saved figure | path=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/figures/exp2_cross_domain_bpd_mae.png
[12:29:48 + 1100.1s] [FIG] saved figure | path=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/figures/exp2_ties_density_sensitivity.png
[12:29:48 + 1100.3s] [EXPORT] saved Bland-Altman export | path=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/tables/exp1_bland_altman_export.csv


## 9. Claim-safety matrix and manuscript-ready evidence summaries

This cell writes a concise evidence card that can be pasted into the ACML manuscript or used as a reviewer-facing `Extended Evidence` appendix.


In [18]:
# ============================================================
# 13. Evidence cards, claim-safety matrix, v11 completeness audit and manifest
# ============================================================
def fmt_ci(mean, lo, hi, digits=3):
    if pd.isna(mean):
        return "NA"
    return f"{mean:.{digits}f} ({lo:.{digits}f}, {hi:.{digits}f})"

claim_rows = [
    {
        "claim": "External Find/Measure generalisation",
        "evidence_table": "exp1_external_find_measure_summary.csv; exp1_bootstrap_ci_by_domain.csv; exp1_threshold_precision_grid.csv",
        "supported_if": "Domain-wise BPD/OFD/HC errors are reported on subject-exclusive FP+UCL test partitions with no HC18 or M-C contamination.",
        "boundary": "External dataset uses BPD/OFD landmarks. ATLAS-FN Measure remains bounding-box geometry + pixel spacing, not a learned measurement head.",
    },
    {
        "claim": "Domain-specialist Find adaptation",
        "evidence_table": "exp2_find_finetune_audit_FP.csv; exp2_find_finetune_audit_UCL.csv; exp2_checkpoint_evaluation_summary.csv",
        "supported_if": "FP and UCL specialists are trained only on their training split and evaluated on held-out test subjects.",
        "boundary": "This adapts the Find detector only. Measure is never fine-tuned.",
    },
    {
        "claim": "Real-domain TIES reduces parameter sign interference",
        "evidence_table": "exp2_ties_summary_density_020.csv and exp2_ties_density_sweep.csv",
        "supported_if": "At least two real domain-specific checkpoints are loaded; post-Elect-Sign interference is reported for actual FP/UCL task vectors.",
        "boundary": "Valid for real trained checkpoints in this notebook, not synthetic/demo vectors.",
    },
    {
        "claim": "TIES-merged checkpoint can be evaluated as a model",
        "evidence_table": "ties_merged_full_checkpoint_density_0.20.pt; exp2_checkpoint_evaluation_summary.csv",
        "supported_if": "The executable TIES checkpoint is created and evaluated on the same FP+UCL held-out test cohort as base and specialist checkpoints.",
        "boundary": "The merged checkpoint is an engineering artefact for external benchmark evaluation; clinical deployment still requires prospective validation.",
    },
]
claim_safety = pd.DataFrame(claim_rows)
claim_safety.to_csv(CFG.output_root / "tables" / "extended_evidence_claim_safety_matrix.csv", index=False)
show_df(claim_safety, "claim-safety matrix")

def required_artifact_audit() -> pd.DataFrame:
    req = [
        ("Experiment 1 primary ledger", CFG.output_root / "tables" / "exp1_external_find_measure_ledger.csv", True),
        ("Experiment 1 summary", CFG.output_root / "tables" / "exp1_external_find_measure_summary.csv", True),
        ("Experiment 1 threshold grid", CFG.output_root / "tables" / "exp1_threshold_precision_grid.csv", True),
        ("FP specialist checkpoint", CFG.output_root / "checkpoints" / "find_FP.pt", bool(CFG.require_complete_exp2)),
        ("UCL specialist checkpoint", CFG.output_root / "checkpoints" / "find_UCL.pt", bool(CFG.require_complete_exp2)),
        ("TIES summary", CFG.output_root / "tables" / "exp2_ties_summary_density_020.csv", bool(CFG.require_complete_exp2)),
        ("TIES density sweep", CFG.output_root / "tables" / "exp2_ties_density_sweep.csv", bool(CFG.require_complete_exp2)),
        ("TIES generic merged state", CFG.output_root / "checkpoints" / "ties_merged_state_density_0.20.pt", bool(CFG.require_complete_exp2)),
        ("TIES executable checkpoint", CFG.output_root / "checkpoints" / "ties_merged_full_checkpoint_density_0.20.pt", bool(CFG.require_complete_exp2 and CFG.create_executable_ties_checkpoint)),
        ("Experiment 2 checkpoint evaluation summary", CFG.output_root / "tables" / "exp2_checkpoint_evaluation_summary.csv", bool(CFG.require_complete_exp2)),
    ]
    rows = []
    for name, path, required in req:
        path = Path(path)
        rows.append({"artifact": name, "path": str(path), "required": bool(required), "exists": path.exists(), "size_bytes": int(path.stat().st_size) if path.exists() else 0})
    audit = pd.DataFrame(rows)
    audit.to_csv(CFG.output_root / "tables" / "v11_required_artifact_audit.csv", index=False)
    return audit

artifact_audit = required_artifact_audit()
show_df(artifact_audit, "v11 required artifact audit")

# Additional semantic checks: domain policy, test-only evaluation, and TIES model presence.
semantic_rows = []
try:
    exp1_summary_path = CFG.output_root / "tables" / "exp1_external_find_measure_summary.csv"
    if exp1_summary_path.exists():
        s1 = pd.read_csv(exp1_summary_path)
        semantic_rows.append({"check": "Experiment 1 domains are FP/UCL only", "passed": set(s1["domain"].astype(str)).issubset({"FP", "UCL"}), "details": sorted(s1["domain"].astype(str).unique())})
        semantic_rows.append({"check": "Experiment 1 split is test only", "passed": set(s1["split"].astype(str).str.lower()).issubset({"test"}), "details": sorted(s1["split"].astype(str).unique())})
    if "exp2_eval_summary" in globals() and len(exp2_eval_summary):
        models = set(exp2_eval_summary["model_name"].astype(str))
        semantic_rows.append({"check": "Experiment 2 includes executable TIES model", "passed": "ties_merged_density_020" in models, "details": sorted(models)})
except Exception as e:
    semantic_rows.append({"check": "semantic audit exception", "passed": False, "details": repr(e)})
semantic_audit = pd.DataFrame(semantic_rows)
semantic_audit.to_csv(CFG.output_root / "tables" / "v11_semantic_evidence_audit.csv", index=False)
show_df(semantic_audit, "v11 semantic evidence audit")

if CFG.require_complete_exp2:
    missing_required = artifact_audit.query("required and not exists")
    failed_semantic = semantic_audit[semantic_audit["passed"].eq(False)] if len(semantic_audit) else pd.DataFrame()
    if len(missing_required) or len(failed_semantic):
        raise RuntimeError(
            "v11 completeness guard failed. Missing required artifacts or failed semantic checks. "
            f"Missing={missing_required[['artifact','path']].to_dict('records')}; "
            f"Failed={failed_semantic.to_dict('records') if len(failed_semantic) else []}"
        )

manifest = {
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "seed": SEED,
    "cfg": _json_safe(asdict(CFG)),
    "paths": {
        "atlas_repro_root": path_status(CFG.atlas_repro_root),
        "multicentre_annotations": path_status(CFG.multicentre_annotations),
        "multicentre_repo_url": CFG.multicentre_repo_url,
        "multicentre_data_doi": CFG.multicentre_data_doi,
        "auto_download_multicentre": CFG.auto_download_multicentre,
        "base_find_checkpoint": path_status(CFG.base_find_checkpoint),
        "primary_external_domains": list(CFG.primary_external_domains),
        "training_source_domain": CFG.training_source_domain,
        "include_training_source_domain_as_anchor": CFG.include_training_source_domain_as_anchor,
    },
    "artifact_audit": artifact_audit.to_dict("records"),
    "semantic_audit": semantic_audit.to_dict("records"),
    "outputs": [str(p.relative_to(CFG.output_root)) for p in CFG.output_root.rglob("*") if p.is_file()],
}
manifest_path = CFG.output_root / "extended_evidence_manifest.json"
manifest_path.write_text(json.dumps(_json_safe(manifest), indent=2, default=str))
log_event("MANIFEST", "wrote manifest", path=manifest_path, outputs=len(manifest.get("outputs", [])))



[12:29:48 + 1100.3s] [DATA] claim-safety matrix | rows=4, columns=4


,claim,evidence_table,supported_if,boundary
0,External Find/Measure generalisation,exp1_external_find_measure_summary.csv; exp1_b...,Domain-wise BPD/OFD/HC errors are reported on ...,External dataset uses BPD/OFD landmarks. ATLAS...
1,Domain-specialist Find adaptation,exp2_find_finetune_audit_FP.csv; exp2_find_fin...,FP and UCL specialists are trained only on the...,This adapts the Find detector only. Measure is...
2,Real-domain TIES reduces parameter sign interf...,exp2_ties_summary_density_020.csv and exp2_tie...,At least two real domain-specific checkpoints ...,Valid for real trained checkpoints in this not...
3,TIES-merged checkpoint can be evaluated as a m...,ties_merged_full_checkpoint_density_0.20.pt; e...,The executable TIES checkpoint is created and ...,The merged checkpoint is an engineering artefa...


[12:29:48 + 1100.4s] [DATA] v11 required artifact audit | rows=10, columns=5


,artifact,path,required,exists,size_bytes
0,Experiment 1 primary ledger,/content/atlas_fn_workspace/outputs/extended_e...,True,True,4203463
1,Experiment 1 summary,/content/atlas_fn_workspace/outputs/extended_e...,True,True,1600
2,Experiment 1 threshold grid,/content/atlas_fn_workspace/outputs/extended_e...,True,True,6555
3,FP specialist checkpoint,/content/atlas_fn_workspace/outputs/extended_e...,False,True,5366725
4,UCL specialist checkpoint,/content/atlas_fn_workspace/outputs/extended_e...,False,True,5369093
5,TIES summary,/content/atlas_fn_workspace/outputs/extended_e...,False,True,899
6,TIES density sweep,/content/atlas_fn_workspace/outputs/extended_e...,False,True,1037
7,TIES generic merged state,/content/atlas_fn_workspace/outputs/extended_e...,False,True,10291907
8,TIES executable checkpoint,/content/atlas_fn_workspace/outputs/extended_e...,False,True,10518077
9,Experiment 2 checkpoint evaluation summary,/content/atlas_fn_workspace/outputs/extended_e...,False,True,2402


[12:29:48 + 1100.4s] [DATA] v11 semantic evidence audit | rows=3, columns=3


,check,passed,details
0,Experiment 1 domains are FP/UCL only,True,"[FP, UCL]"
1,Experiment 1 split is test only,True,[test]
2,Experiment 2 includes executable TIES model,True,"[base, pooled_external, specialist_FP, special..."


[12:29:49 + 1100.9s] [MANIFEST] wrote manifest | path=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/extended_evidence_manifest.json, outputs=7254


In [19]:
# ============================================================
# 14. Runtime progress-log summary
# ============================================================
log_event("DONE", "Notebook execution reached final summary")
try:
    progress_log = pd.read_json(PROGRESS_LOG_PATH, lines=True)
    show_df(progress_log.tail(25), "last 25 progress-log events", max_rows=25)
    print("Progress log:", PROGRESS_LOG_PATH)
except Exception as e:
    print("Could not load progress log:", e)
print("Output root:", CFG.output_root)
print("Tables:", len(list((CFG.output_root / "tables").glob("*"))))
print("Figures:", len(list((CFG.output_root / "figures").glob("*"))))
print("Checkpoints:", len(list((CFG.output_root / "checkpoints").glob("*"))))



[12:29:49 + 1100.9s] [DONE] Notebook execution reached final summary
[12:29:49 + 1100.9s] [DATA] last 25 progress-log events | rows=25, columns=76


,time,elapsed_sec,stage,message,log,class_name,rows,columns,label,checkpoint,...,density,raw_interference,post_trim,post_elect,nonzero_ties,replaced_float_tensors,missing,unexpected,name,outputs
139,12:29:05,1057.202548,INFER,evaluation start,NaN,NaN,935.0,NaN,NaN,/content/atlas_fn_workspace/outputs/extended_e...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
140,12:29:10,1062.640665,INFER,heartbeat,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
141,12:29:16,1067.969557,INFER,heartbeat,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142,12:29:21,1073.317742,INFER,heartbeat,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
143,12:29:26,20.630000,INFER,evaluation done,NaN,NaN,935.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144,12:29:26,1077.836922,EVAL2,evaluating checkpoint,NaN,NaN,NaN,NaN,NaN,/content/atlas_fn_workspace/outputs/extended_e...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ties_merged_density_020,NaN
145,12:29:26,1077.838285,COMPAT,pickle-safe YOLO26 shim registered,NaN,SPPF_YOLO26_Compatible,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
146,12:29:26,1077.839176,INFER,loading YOLO checkpoint,NaN,NaN,NaN,NaN,NaN,/content/atlas_fn_workspace/outputs/extended_e...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
147,12:29:26,1077.894503,INFER,YOLO checkpoint loaded,NaN,NaN,NaN,NaN,NaN,/content/atlas_fn_workspace/outputs/extended_e...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
148,12:29:26,1077.926264,INFER,evaluation start,NaN,NaN,935.0,NaN,NaN,/content/atlas_fn_workspace/outputs/extended_e...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Progress log: /content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/logs/extended_evidence_progress_log.jsonl
Output root: /content/atlas_fn_workspace/outputs/extended_evidence_ties_colab
Tables: 39
Figures: 4
Checkpoints: 5


## 9B. Extended-evidence manuscript consistency audit

This section compares generated external-validation and TIES metrics against the manuscript register without forcing equality.

In [20]:
# ============================================================
# 14B. Extended-evidence manuscript consistency audit
# ============================================================
# This cell audits the external-validation/TIES results against the manuscript register.
# It never overwrites computed values. Differences are recorded as computed_differs_from_manuscript.

EXPECTED_EXTENDED = {
    "fp_detection_pct": {"expected": 98.5, "tol": 0.2, "source": "Table 7 / UOG Table 8"},
    "fp_bpd_mae_mm": {"expected": 0.987, "tol": 0.03, "source": "Table 7 / UOG Table 8"},
    "fp_hc_mae_pct": {"expected": 1.836, "tol": 0.05, "source": "Table 7 / UOG Table 8"},
    "ucl_detection_pct": {"expected": 92.7, "tol": 0.5, "source": "Table 7 / UOG Table 8"},
    "ucl_bpd_mae_mm": {"expected": 1.771, "tol": 0.08, "source": "Table 7 / UOG Table 8"},
    "ucl_hc_mae_pct": {"expected": 2.337, "tol": 0.08, "source": "Table 7 / UOG Table 8"},
    "fpucl_bpd_mae_mm": {"expected": 1.030, "tol": 0.04, "source": "§4.4 Results"},
    "fpucl_broad_pct": {"expected": 86.2, "tol": 0.5, "source": "§4.4 Results"},
    "ties_D": {"expected": 2523182, "tol": 0, "source": "§4.7 Results"},
    "ties_raw_interference_pct": {"expected": 44.95, "tol": 0.15, "source": "§4.7 Results"},
    "ties_post_elect_pct": {"expected": 0.00, "tol": 0.01, "source": "§4.7 Results"},
    "ties_nonzero_merged": {"expected": 814520, "tol": 5000, "source": "§4.7 Results"},
}

def _read_csv_safe(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if Path(path).exists() else pd.DataFrame()

def _first_float(df: pd.DataFrame, mask, col: str):
    try:
        s = df.loc[mask, col]
        if len(s) == 0:
            return float("nan")
        return float(s.iloc[0])
    except Exception:
        return float("nan")

summary_path = CFG.output_root / "tables" / "exp1_external_find_measure_summary.csv"
ledger_path = CFG.output_root / "tables" / "exp1_external_find_measure_ledger.csv"
ties_path = CFG.output_root / "tables" / "exp2_ties_summary_density_020.csv"
exp1_sum = _read_csv_safe(summary_path)
exp1_led = _read_csv_safe(ledger_path)
ties_sum = _read_csv_safe(ties_path)

observed = {}
if len(exp1_sum):
    model_col = "model_name" if "model_name" in exp1_sum.columns else None
    base_mask = pd.Series(True, index=exp1_sum.index)
    if model_col:
        base_mask = exp1_sum[model_col].astype(str).str.contains("base|zero", case=False, na=False)
        if not base_mask.any():
            base_mask = pd.Series(True, index=exp1_sum.index)
    for dom, pref in [("FP", "fp"), ("UCL", "ucl")]:
        m = base_mask & exp1_sum["domain"].astype(str).str.upper().eq(dom)
        observed[f"{pref}_detection_pct"] = _first_float(exp1_sum, m, "detection_rate")
        observed[f"{pref}_bpd_mae_mm"] = _first_float(exp1_sum, m, "bpd_mae_mm")
        observed[f"{pref}_hc_mae_pct"] = _first_float(exp1_sum, m, "hc_mae_pct")

if len(exp1_led):
    work = exp1_led.copy()
    if "domain" in work.columns:
        work["domain_norm"] = work["domain"].astype(str).str.upper()
        work = work[work["domain_norm"].isin(["FP", "UCL"])]
    if "model_name" in work.columns:
        m = work["model_name"].astype(str).str.contains("base|zero", case=False, na=False)
        if m.any():
            work = work[m]
    det = work[work.get("detected", False).astype(bool)] if "detected" in work.columns else work
    observed["fpucl_bpd_mae_mm"] = float(det["bpd_abs_err_mm"].mean()) if "bpd_abs_err_mm" in det.columns and len(det) else float("nan")
    observed["fpucl_broad_pct"] = float(100.0 * det["broad_pair"].mean()) if "broad_pair" in det.columns and len(det) else float("nan")

if len(ties_sum):
    row = ties_sum.iloc[0]
    observed["ties_D"] = float(row.get("D", float("nan")))
    observed["ties_raw_interference_pct"] = float(row.get("raw_interference", float("nan"))) * (100.0 if float(row.get("raw_interference", 0)) <= 1.0 else 1.0)
    observed["ties_post_elect_pct"] = float(row.get("post_elect_interference", float("nan"))) * (100.0 if float(row.get("post_elect_interference", 0)) <= 1.0 else 1.0)
    observed["ties_nonzero_merged"] = float(row.get("nonzero_ties", float("nan")))

rows = []
for key, meta in EXPECTED_EXTENDED.items():
    obs = observed.get(key, float("nan"))
    exp = float(meta["expected"])
    tol = float(meta["tol"])
    if pd.isna(obs):
        status = "not_computed"
        diff = float("nan")
    else:
        diff = obs - exp
        status = "matches_manuscript" if abs(diff) <= tol else "computed_differs_from_manuscript"
    rows.append({
        "key": key,
        "observed": obs,
        "expected_manuscript": exp,
        "difference": diff,
        "tolerance": tol,
        "status": status,
        "source": meta.get("source", ""),
    })

extended_result_register = pd.DataFrame(rows)
extended_result_register.to_csv(CFG.output_root / "tables" / "extended_evidence_manuscript_consistency_register.csv", index=False)
show_df(extended_result_register, "Extended evidence manuscript consistency register", max_rows=50)

# Compact method-and-claim audit card.
claim_rows = [
    {"claim": "FP/UCL are treated as true external domains; HC18 is source anchor only", "audit_file": "exp1_domain_policy_audit_all_domains.csv", "status": "audited"},
    {"claim": "External measurement uses Find geometry + pixel spacing; Measure is deterministic", "audit_file": "exp1_external_find_measure_ledger.csv", "status": "audited"},
    {"claim": "TIES uses real FP and UCL Find specialist checkpoints", "audit_file": "exp2_ties_summary_density_020.csv", "status": "audited" if len(ties_sum) else "not_computed"},
    {"claim": "Manuscript consistency is not assumed; values are compared against the register", "audit_file": "extended_evidence_manuscript_consistency_register.csv", "status": "audited"},
]
pd.DataFrame(claim_rows).to_csv(CFG.output_root / "tables" / "extended_evidence_method_claim_audit.csv", index=False)



[12:29:49 + 1101.0s] [DATA] Extended evidence manuscript consistency register | rows=12, columns=7


,key,observed,expected_manuscript,difference,tolerance,status,source
0,fp_detection_pct,9.988636e+01,98.500,1.386364,0.20,computed_differs_from_manuscript,Table 7 / UOG Table 8
1,fp_bpd_mae_mm,9.272497e-01,0.987,-0.059750,0.03,computed_differs_from_manuscript,Table 7 / UOG Table 8
2,fp_hc_mae_pct,1.243802e+00,1.836,-0.592198,0.05,computed_differs_from_manuscript,Table 7 / UOG Table 8
3,ucl_detection_pct,1.000000e+02,92.700,7.300000,0.50,computed_differs_from_manuscript,Table 7 / UOG Table 8
4,ucl_bpd_mae_mm,1.936474e+00,1.771,0.165474,0.08,computed_differs_from_manuscript,Table 7 / UOG Table 8
5,ucl_hc_mae_pct,2.843074e+00,2.337,0.506074,0.08,computed_differs_from_manuscript,Table 7 / UOG Table 8
6,fpucl_bpd_mae_mm,9.866794e-01,1.030,-0.043321,0.04,computed_differs_from_manuscript,§4.4 Results
7,fpucl_broad_pct,8.950749e+01,86.200,3.307495,0.50,computed_differs_from_manuscript,§4.4 Results
8,ties_D,2.523182e+06,2523182.000,0.000000,0.00,matches_manuscript,§4.7 Results
9,ties_raw_interference_pct,4.626790e+01,44.950,1.317904,0.15,computed_differs_from_manuscript,§4.7 Results


In [21]:

# ============================================================
# 14C. Output-intention audit — final completed-notebook sanity check
# ============================================================
# This section verifies that the outputs produced by this extended-evidence notebook
# match the intended downstream use: external evidence + adaptation/TIES audit.
# It intentionally checks the artefacts actually produced by the legacy/Colab
# pipeline rather than a stale filename from an earlier draft.

from pathlib import Path
import json
import pandas as pd
import numpy as np

def file_exists_nonempty(path: Path) -> bool:
    path = Path(path)
    return path.exists() and path.is_file() and path.stat().st_size > 0

def csv_rows(path: Path) -> int:
    try:
        if not file_exists_nonempty(path):
            return 0
        return int(len(pd.read_csv(path)))
    except Exception:
        return -1

def choose_first_existing(paths):
    rows = []
    selected = None
    for p in paths:
        p = Path(p)
        exists = file_exists_nonempty(p)
        rows.append({
            "candidate": str(p),
            "exists_nonempty": bool(exists),
            "rows_if_csv": csv_rows(p) if p.suffix.lower() == ".csv" else None,
            "bytes": int(p.stat().st_size) if p.exists() and p.is_file() else 0,
        })
        if selected is None and exists:
            selected = p
    return selected, rows

def csv_has_domains_and_splits(path: Path, required_domains=("FP", "UCL")) -> dict:
    out = {
        "has_required_domains": False,
        "domains": [],
        "splits": [],
        "rows": 0,
        "detail": "",
    }
    try:
        df = pd.read_csv(path)
        out["rows"] = int(len(df))
        if "domain" in df.columns:
            domains = sorted(df["domain"].astype(str).str.upper().unique().tolist())
            out["domains"] = domains
            out["has_required_domains"] = all(str(d).upper() in domains for d in required_domains)
        if "split" in df.columns:
            out["splits"] = sorted(df["split"].astype(str).str.lower().unique().tolist())
        out["detail"] = f"rows={out['rows']}; domains={out['domains']}; splits={out['splits']}"
    except Exception as e:
        out["detail"] = repr(e)
    return out

def output_intention_audit() -> pd.DataFrame:
    tables = CFG.output_root / "tables"

    prepared_candidates = [
        tables / "exp1_external_annotations_subject_exclusive.csv",
        tables / "exp1_v11_primary_test_evaluation_cohort.csv",
        tables / "exp1_external_annotations_all_domains_unfiltered.csv",
        tables / "multicentre_head_landmarks_standardised_cache.csv",
        tables / "multicentre_head_landmarks_combined_raw.csv",
        Path(getattr(CFG, "multicentre_annotations", "")),
    ]
    prepared_selected, prepared_candidate_rows = choose_first_existing(prepared_candidates)
    pd.DataFrame(prepared_candidate_rows).to_csv(tables / "prepared_multicentre_ledger_candidate_audit.csv", index=False)

    checks = [
        {
            "block": "restore",
            "artifact": "restore manifest",
            "path": CFG.output_root.parent / "extended_evidence_colab_bootstrap" / "restore_manifest.json",
            "required": True,
            "expected": "non-empty JSON restore manifest",
        },
        {
            "block": "external_adapter",
            "artifact": "prepared multicentre / working external ledger",
            "path": prepared_selected,
            "required": True,
            "expected": "non-empty FP/UCL ledger or evaluation cohort generated by the dataset adapter",
            "candidate_count": len(prepared_candidate_rows),
        },
        {
            "block": "external_denominator",
            "artifact": "external denominator audit",
            "path": tables / "exp1_external_denominator_audit.csv",
            "required": True,
            "expected": "FP=1637 rows and UCL=159 rows, with test denominators FP=880/UCL=55",
        },
        {
            "block": "yolo_export",
            "artifact": "FP YOLO data yaml",
            "path": CFG.output_root / "yolo_exports" / "FP" / "find_FP.yaml",
            "required": bool(getattr(CFG, "run_yolo_training", False)),
            "expected": "YOLO export for FP fine-tuning",
        },
        {
            "block": "yolo_export",
            "artifact": "UCL YOLO data yaml",
            "path": CFG.output_root / "yolo_exports" / "UCL" / "find_UCL.yaml",
            "required": bool(getattr(CFG, "run_yolo_training", False)),
            "expected": "YOLO export for UCL fine-tuning",
        },
        {
            "block": "exp1",
            "artifact": "external Find/Measure ledger",
            "path": tables / "exp1_external_find_measure_ledger.csv",
            "required": True,
            "expected": "zero-shot and/or model comparison external validation ledger",
        },
        {
            "block": "exp1",
            "artifact": "external Find/Measure summary",
            "path": tables / "exp1_external_find_measure_summary.csv",
            "required": True,
            "expected": "domain-wise FP/UCL metrics",
        },
        {
            "block": "exp1",
            "artifact": "bootstrap CI table",
            "path": tables / "exp1_bootstrap_ci_by_domain.csv",
            "required": True,
            "expected": "bootstrap confidence intervals",
        },
        {
            "block": "exp2",
            "artifact": "FP specialist checkpoint",
            "path": CFG.output_root / "checkpoints" / "find_FP.pt",
            "required": bool(getattr(CFG, "require_complete_exp2", False)),
            "expected": "real-domain FP specialist checkpoint",
        },
        {
            "block": "exp2",
            "artifact": "UCL specialist checkpoint",
            "path": CFG.output_root / "checkpoints" / "find_UCL.pt",
            "required": bool(getattr(CFG, "require_complete_exp2", False)),
            "expected": "real-domain UCL specialist checkpoint",
        },
        {
            "block": "ties",
            "artifact": "TIES summary density 0.20",
            "path": tables / "exp2_ties_summary_density_020.csv",
            "required": bool(getattr(CFG, "require_complete_exp2", False)),
            "expected": "raw/post-elect sign interference and merged density",
        },
        {
            "block": "ties",
            "artifact": "TIES density sweep",
            "path": tables / "exp2_ties_density_sweep.csv",
            "required": bool(getattr(CFG, "require_complete_exp2", False)),
            "expected": "density sensitivity table",
        },
        {
            "block": "ties",
            "artifact": "Experiment 2 checkpoint evaluation summary",
            "path": tables / "exp2_checkpoint_evaluation_summary.csv",
            "required": bool(getattr(CFG, "require_complete_exp2", False)),
            "expected": "base/specialist/TIES cross-domain evaluation matrix",
        },
        {
            "block": "manuscript_audit",
            "artifact": "extended manuscript consistency register",
            "path": tables / "extended_evidence_manuscript_consistency_register.csv",
            "required": True,
            "expected": "computed-vs-manuscript metric audit",
        },
        {
            "block": "runtime",
            "artifact": "runtime dependency audit",
            "path": tables / "runtime_dependency_audit.csv",
            "required": True,
            "expected": "Ultralytics/package install audit",
        },
    ]

    rows = []
    for c in checks:
        p = c.get("path")
        exists = bool(p is not None and file_exists_nonempty(Path(p)))
        rows_if_csv = csv_rows(Path(p)) if p is not None and Path(p).suffix.lower() == ".csv" else None
        detail = ""

        if exists and c["block"] == "external_adapter":
            q = csv_has_domains_and_splits(Path(p))
            detail = q["detail"]
            # PASS for either full working ledger or test-only evaluation cohort as long as rows/domains are present.
            exists = q["rows"] > 0 and q["has_required_domains"]

        rows.append({
            "block": c["block"],
            "artifact": c["artifact"],
            "path": str(p) if p is not None else "",
            "required": bool(c["required"]),
            "expected": c["expected"],
            "exists_nonempty": exists,
            "rows_if_csv": rows_if_csv,
            "detail": detail,
            "status": "PASS" if exists else ("MISSING_REQUIRED" if c["required"] else "missing_optional"),
        })

    df = pd.DataFrame(rows)
    df.to_csv(tables / "extended_evidence_output_intention_audit.csv", index=False)

    # Final readiness summary for reviewer/manuscript use.
    required = df[df["required"]]
    manuscript = pd.read_csv(tables / "extended_evidence_manuscript_consistency_register.csv") if file_exists_nonempty(tables / "extended_evidence_manuscript_consistency_register.csv") else pd.DataFrame()
    readiness = {
        "required_outputs_total": int(len(required)),
        "required_outputs_passed": int(required["status"].eq("PASS").sum()),
        "required_outputs_missing": int(required["status"].eq("MISSING_REQUIRED").sum()),
        "optional_outputs_present": int(df[(~df["required"]) & (df["status"].eq("PASS"))].shape[0]),
        "manuscript_register_rows": int(len(manuscript)),
        "manuscript_metrics_matching": int(manuscript["status"].eq("matches_manuscript").sum()) if "status" in manuscript.columns else 0,
        "manuscript_metrics_differing": int(manuscript["status"].eq("computed_differs_from_manuscript").sum()) if "status" in manuscript.columns else 0,
        "interpretation": (
            "PASS: required evidence artefacts exist. Manuscript numeric differences are expected if regenerated checkpoints, "
            "Ultralytics version, split-repair policy, or checkpoint exports differ from the locked manuscript run; the register records these differences."
        ),
    }
    (tables / "extended_evidence_final_readiness_summary.json").write_text(json.dumps(readiness, indent=2))
    pd.DataFrame([readiness]).to_csv(tables / "extended_evidence_final_readiness_summary.csv", index=False)
    return df

OUTPUT_INTENTION_AUDIT = output_intention_audit()
show_df(OUTPUT_INTENTION_AUDIT, "extended evidence output-intention audit", max_rows=50)

missing_required = OUTPUT_INTENTION_AUDIT[OUTPUT_INTENTION_AUDIT["status"].eq("MISSING_REQUIRED")]
if len(missing_required):
    print("\n[output-audit:warning] Required evidence blocks are missing. Review this table before manuscript use.")
else:
    print("\n[output-audit] PASS: all required extended-evidence artefacts are present.")

try:
    readiness = pd.read_csv(CFG.output_root / "tables" / "extended_evidence_final_readiness_summary.csv")
    show_df(readiness, "final extended-evidence readiness summary", max_rows=5)
except Exception as e:
    print("Could not show readiness summary:", repr(e))


[12:29:49 + 1101.3s] [DATA] extended evidence output-intention audit | rows=15, columns=9


,block,artifact,path,required,expected,exists_nonempty,rows_if_csv,detail,status
0,restore,restore manifest,/content/atlas_fn_workspace/outputs/extended_e...,True,non-empty JSON restore manifest,True,NaN,,PASS
1,external_adapter,prepared multicentre / working external ledger,/content/atlas_fn_workspace/outputs/extended_e...,True,non-empty FP/UCL ledger or evaluation cohort g...,True,1796.0,"rows=1796; domains=['FP', 'UCL']; splits=['tes...",PASS
2,external_denominator,external denominator audit,/content/atlas_fn_workspace/outputs/extended_e...,True,"FP=1637 rows and UCL=159 rows, with test denom...",True,6.0,,PASS
3,yolo_export,FP YOLO data yaml,/content/atlas_fn_workspace/outputs/extended_e...,True,YOLO export for FP fine-tuning,True,NaN,,PASS
4,yolo_export,UCL YOLO data yaml,/content/atlas_fn_workspace/outputs/extended_e...,True,YOLO export for UCL fine-tuning,True,NaN,,PASS
5,exp1,external Find/Measure ledger,/content/atlas_fn_workspace/outputs/extended_e...,True,zero-shot and/or model comparison external val...,True,2805.0,,PASS
6,exp1,external Find/Measure summary,/content/atlas_fn_workspace/outputs/extended_e...,True,domain-wise FP/UCL metrics,True,6.0,,PASS
7,exp1,bootstrap CI table,/content/atlas_fn_workspace/outputs/extended_e...,True,bootstrap confidence intervals,True,30.0,,PASS
8,exp2,FP specialist checkpoint,/content/atlas_fn_workspace/outputs/extended_e...,False,real-domain FP specialist checkpoint,True,NaN,,PASS
9,exp2,UCL specialist checkpoint,/content/atlas_fn_workspace/outputs/extended_e...,False,real-domain UCL specialist checkpoint,True,NaN,,PASS



[output-audit] PASS: all required extended-evidence artefacts are present.
[12:29:49 + 1101.3s] [DATA] final extended-evidence readiness summary | rows=1, columns=8


,required_outputs_total,required_outputs_passed,required_outputs_missing,optional_outputs_present,manuscript_register_rows,manuscript_metrics_matching,manuscript_metrics_differing,interpretation
0,10,10,0,5,12,2,10,PASS: required evidence artefacts exist. Manus...


## 10. Export extended evidence outputs as a ZIP package

This final evidence-export section packages the runtime outputs from `CFG.output_root` into a single reviewer-facing ZIP archive. It is rerun-safe: it avoids recursively zipping previous ZIP exports, writes a file-level manifest, prints included/excluded counts, and can optionally include trained checkpoints. By default, model checkpoints are excluded to avoid very large archives unless `EXPORT_INCLUDE_CHECKPOINTS=True`.


In [22]:

# ============================================================
# 15. Export extended evidence outputs as a reviewer-ready ZIP
# ============================================================
import zipfile
from datetime import datetime

# Conservative defaults: output tables/figures/logs/manifests are included; heavy checkpoints are excluded.
EXPORT_INCLUDE_CHECKPOINTS = False       # set True only when you intentionally want .pt/.pth checkpoint files in the ZIP
EXPORT_MAX_FILE_MB = 750                 # safety guard per file; set None to disable
EXPORT_HASH_FILES = True                 # SHA-256 audit manifest for included files
EXPORT_COLAB_DOWNLOAD = False            # set True in Colab if you want an automatic browser download prompt
EXPORT_NAME_PREFIX = "ATLAS_FN_04_ExtendedEvidence_TIES_TRUE_FINAL_outputs"

EXPORT_INCLUDE_SUFFIXES = {
    ".csv", ".tsv", ".json", ".jsonl", ".txt", ".md", ".yaml", ".yml",
    ".png", ".jpg", ".jpeg", ".pdf", ".svg", ".html", ".parquet",
    ".npy", ".npz", ".pkl"
}
EXPORT_CHECKPOINT_SUFFIXES = {".pt", ".pth", ".ckpt", ".safetensors"}
EXPORT_ALWAYS_EXCLUDE_DIRS = {"packages", "__pycache__", ".ipynb_checkpoints", "wandb", "runs", "ultralytics_runs"}

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def is_relative_to_safe(path: Path, parent: Path) -> bool:
    try:
        path.resolve().relative_to(parent.resolve())
        return True
    except Exception:
        return False

def collect_export_files(output_root: Path = CFG.output_root,
                         include_checkpoints: bool = EXPORT_INCLUDE_CHECKPOINTS,
                         max_file_mb: Optional[float] = EXPORT_MAX_FILE_MB) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """Return included and excluded file records for ZIP export."""
    output_root = Path(output_root)
    package_dir = output_root / "packages"
    included, excluded = [], []
    if not output_root.exists():
        log_event("EXPORT", "output root does not exist; nothing to export", output_root=output_root)
        return included, excluded

    all_files = [p for p in output_root.rglob("*") if p.is_file()]
    for p in progress_iter(all_files, desc="scan export candidates", total=len(all_files), stage="EXPORT"):
        rel = p.relative_to(output_root)
        rel_parts = set(rel.parts[:-1])
        suffix = p.suffix.lower()
        size_bytes = p.stat().st_size
        reason = None

        if p.is_symlink() and not is_relative_to_safe(p, output_root):
            reason = "external_symlink_excluded_to_keep_zip_manifest_consistent"
        elif EXPORT_ALWAYS_EXCLUDE_DIRS.intersection(rel_parts):
            reason = "excluded_dir"
        elif suffix == ".zip":
            reason = "avoid_recursive_zip"
        elif suffix in EXPORT_CHECKPOINT_SUFFIXES and not include_checkpoints:
            reason = "checkpoint_excluded_by_default"
        elif suffix not in EXPORT_INCLUDE_SUFFIXES and suffix not in EXPORT_CHECKPOINT_SUFFIXES:
            reason = "suffix_not_in_export_allowlist"
        elif max_file_mb is not None and size_bytes > max_file_mb * 1024 * 1024:
            reason = f"file_larger_than_{max_file_mb}_MB"

        rec = {
            "path": str(p),
            "relative_path": str(rel),
            "suffix": suffix,
            "size_bytes": int(size_bytes),
            "size_mb": round(size_bytes / (1024 * 1024), 4),
        }
        if reason:
            rec["excluded_reason"] = reason
            excluded.append(rec)
        else:
            included.append(rec)
    return included, excluded

def export_extended_evidence_zip(output_root: Path = CFG.output_root,
                                 include_checkpoints: bool = EXPORT_INCLUDE_CHECKPOINTS,
                                 hash_files: bool = EXPORT_HASH_FILES) -> Path:
    """Create a reviewer-facing ZIP of Extended Evidence outputs with manifest and screen summary."""
    output_root = Path(output_root)
    package_dir = output_root / "packages"
    package_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    archive_path = package_dir / f"{EXPORT_NAME_PREFIX}_{stamp}.zip"

    with stage_timer("EXPORT", "collect output files for ZIP export"):
        included, excluded = collect_export_files(output_root, include_checkpoints=include_checkpoints)
        inc_df = pd.DataFrame(included)
        exc_df = pd.DataFrame(excluded)
        if len(inc_df) and hash_files:
            hashes = []
            for path_str in progress_iter(inc_df["path"].tolist(), desc="hash export files", total=len(inc_df), stage="EXPORT"):
                try:
                    hashes.append(sha256_file(Path(path_str)))
                except Exception as e:
                    hashes.append(f"HASH_ERROR: {e}")
            inc_df["sha256"] = hashes

        manifest_path = package_dir / f"{EXPORT_NAME_PREFIX}_{stamp}_manifest.csv"
        excluded_path = package_dir / f"{EXPORT_NAME_PREFIX}_{stamp}_excluded.csv"
        summary_path = package_dir / f"{EXPORT_NAME_PREFIX}_{stamp}_summary.json"
        inc_df.to_csv(manifest_path, index=False)
        exc_df.to_csv(excluded_path, index=False)
        summary = {
            "created_at": stamp,
            "output_root": str(output_root),
            "archive_path": str(archive_path),
            "include_checkpoints": include_checkpoints,
            "hash_files": hash_files,
            "included_files": int(len(inc_df)),
            "excluded_files": int(len(exc_df)),
            "included_size_mb": float(round(inc_df["size_mb"].sum(), 3)) if len(inc_df) else 0.0,
            "excluded_size_mb": float(round(exc_df["size_mb"].sum(), 3)) if len(exc_df) else 0.0,
        }
        summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    with stage_timer("EXPORT", "write ZIP archive"):
        # Include runtime outputs first.
        with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
            for rec in progress_iter(included, desc="zip included outputs", total=len(included), stage="EXPORT"):
                p = Path(rec["path"])
                if p.exists() and is_relative_to_safe(p, output_root):
                    zf.write(p, arcname=str(Path("extended_evidence_outputs") / rec["relative_path"]))
            # Include the export audit files themselves.
            for p in [manifest_path, excluded_path, summary_path]:
                zf.write(p, arcname=str(Path("export_audit") / p.name))
            # Include progress log if it exists and was not already included.
            if PROGRESS_LOG_PATH.exists() and is_relative_to_safe(PROGRESS_LOG_PATH, output_root):
                zf.write(PROGRESS_LOG_PATH, arcname=str(Path("export_audit") / PROGRESS_LOG_PATH.name))

    archive_size_mb = archive_path.stat().st_size / (1024 * 1024)
    log_event("EXPORT", "ZIP export complete", archive=archive_path, archive_size_mb=round(archive_size_mb, 3), included_files=len(included), excluded_files=len(excluded))
    print("\n[EXPORT COMPLETE]")
    print("Archive:", archive_path)
    print("Archive size MB:", round(archive_size_mb, 3))
    print("Manifest:", manifest_path)
    print("Excluded-file audit:", excluded_path)
    print("Summary:", summary_path)
    if len(included):
        show_df(pd.DataFrame(included).sort_values("size_bytes", ascending=False).head(15), "largest included export files", max_rows=15)
    if len(excluded):
        show_df(pd.DataFrame(excluded).sort_values("size_bytes", ascending=False).head(15), "largest excluded export files", max_rows=15)
    return archive_path

extended_evidence_zip_path = export_extended_evidence_zip(
    output_root=CFG.output_root,
    include_checkpoints=EXPORT_INCLUDE_CHECKPOINTS,
    hash_files=EXPORT_HASH_FILES,
)

if EXPORT_COLAB_DOWNLOAD:
    try:
        from google.colab import files
        files.download(str(extended_evidence_zip_path))
    except Exception as e:
        print("Colab download not available:", e)

print("Reviewer/output ZIP ready:", extended_evidence_zip_path)



[12:29:49 + 1101.3s] [EXPORT] START: collect output files for ZIP export


scan export candidates:   0%|          | 0/7261 [00:00<?, ?it/s]

hash export files:   0%|          | 0/3658 [00:00<?, ?it/s]

[12:29:51 + 1102.9s] [EXPORT] DONE: collect output files for ZIP export | duration_sec=1.55
[12:29:51 + 1102.9s] [EXPORT] START: write ZIP archive


zip included outputs:   0%|          | 0/3658 [00:00<?, ?it/s]

[12:29:52 + 1104.4s] [EXPORT] DONE: write ZIP archive | duration_sec=1.56
[12:29:52 + 1104.4s] [EXPORT] ZIP export complete | archive=/content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/packages/ATLAS_FN_04_ExtendedEvidence_TIES_TRUE_FINAL_outputs_20260709_122949.zip, archive_size_mb=5.488, included_files=3658, excluded_files=3603

[EXPORT COMPLETE]
Archive: /content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/packages/ATLAS_FN_04_ExtendedEvidence_TIES_TRUE_FINAL_outputs_20260709_122949.zip
Archive size MB: 5.488
Manifest: /content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/packages/ATLAS_FN_04_ExtendedEvidence_TIES_TRUE_FINAL_outputs_20260709_122949_manifest.csv
Excluded-file audit: /content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/packages/ATLAS_FN_04_ExtendedEvidence_TIES_TRUE_FINAL_outputs_20260709_122949_excluded.csv
Summary: /content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/packages/ATLAS_FN_04_ExtendedEvidence_T

,path,relative_path,suffix,size_bytes,size_mb
18,/content/atlas_fn_workspace/outputs/extended_e...,tables/exp2_checkpoint_evaluation_ledger.csv,.csv,6980595,6.6572
10,/content/atlas_fn_workspace/outputs/extended_e...,tables/exp1_bland_altman_export.csv,.csv,4411446,4.2071
44,/content/atlas_fn_workspace/outputs/extended_e...,tables/exp1_external_find_measure_ledger.csv,.csv,4203463,4.0087
46,/content/atlas_fn_workspace/outputs/extended_e...,tables/exp1_external_annotations_all_domains_u...,.csv,2798833,2.6692
32,/content/atlas_fn_workspace/outputs/extended_e...,tables/multicentre_head_landmarks_standardised...,.csv,2730300,2.6038
30,/content/atlas_fn_workspace/outputs/extended_e...,tables/exp1_external_annotations_subject_exclu...,.csv,1891852,1.8042
17,/content/atlas_fn_workspace/outputs/extended_e...,tables/multicentre_head_landmarks_combined_raw...,.csv,1490835,1.4218
45,/content/atlas_fn_workspace/outputs/extended_e...,tables/exp1_excluded_source_or_nonprimary_doma...,.csv,999688,0.9534
12,/content/atlas_fn_workspace/outputs/extended_e...,tables/exp1_v11_primary_test_evaluation_cohort...,.csv,983886,0.9383
60,/content/atlas_fn_workspace/outputs/extended_e...,yolo_exports/ALL_external/export_manifest.csv,.csv,623234,0.5944


[12:29:52 + 1104.5s] [DATA] largest excluded export files | rows=15, columns=6


,path,relative_path,suffix,size_bytes,size_mb,excluded_reason
2,/content/atlas_fn_workspace/outputs/extended_e...,checkpoints/ties_merged_full_checkpoint_densit...,.pt,10518077,10.0308,checkpoint_excluded_by_default
3,/content/atlas_fn_workspace/outputs/extended_e...,checkpoints/ties_merged_state_density_0.20.pt,.pt,10291907,9.8151,checkpoint_excluded_by_default
1,/content/atlas_fn_workspace/outputs/extended_e...,checkpoints/find_ALL_external.pt,.pt,5369349,5.1206,checkpoint_excluded_by_default
10,/content/atlas_fn_workspace/outputs/extended_e...,yolo_runs/ATLAS_FN_ext_find_ALL_external/weigh...,.pt,5369349,5.1206,checkpoint_excluded_by_default
9,/content/atlas_fn_workspace/outputs/extended_e...,yolo_runs/ATLAS_FN_ext_find_ALL_external/weigh...,.pt,5369349,5.1206,checkpoint_excluded_by_default
8,/content/atlas_fn_workspace/outputs/extended_e...,yolo_runs/ATLAS_FN_ext_find_UCL/weights/last.pt,.pt,5369093,5.1204,checkpoint_excluded_by_default
7,/content/atlas_fn_workspace/outputs/extended_e...,yolo_runs/ATLAS_FN_ext_find_UCL/weights/best.pt,.pt,5369093,5.1204,checkpoint_excluded_by_default
4,/content/atlas_fn_workspace/outputs/extended_e...,checkpoints/find_UCL.pt,.pt,5369093,5.1204,checkpoint_excluded_by_default
0,/content/atlas_fn_workspace/outputs/extended_e...,checkpoints/find_FP.pt,.pt,5366725,5.1181,checkpoint_excluded_by_default
6,/content/atlas_fn_workspace/outputs/extended_e...,yolo_runs/ATLAS_FN_ext_find_FP/weights/last.pt,.pt,5366725,5.1181,checkpoint_excluded_by_default


Reviewer/output ZIP ready: /content/atlas_fn_workspace/outputs/extended_evidence_ties_colab/packages/ATLAS_FN_04_ExtendedEvidence_TIES_TRUE_FINAL_outputs_20260709_122949.zip


## 11. Final execution checklist

Before using this notebook for manuscript evidence, confirm:

- [ ] `cross_notebook_dependency_audit_01_02_03_to_04.csv` reports `available` for Notebook 01 and 02; Notebook 03 may be `optional_missing_continue` unless explicitly required dependencies.
- [ ] `model_label_registry_bmc_vs_optimised_04.csv` keeps BMC and optimised Find checkpoints separate.
- [ ] `runtime_dependency_audit.csv` records Ultralytics/Torch/Python versions.
- [ ] `exp1_external_denominator_audit.csv` confirms FP/UCL denominators.
- [ ] `exp1_external_find_measure_summary.csv` contains external Find/Measure results.
- [ ] `exp2_ties_summary_density_020.csv` and `exp2_ties_density_sweep.csv` exist when TIES was requested.
- [ ] `extended_evidence_manuscript_consistency_register.csv` is used as the manuscript numeric-agreement audit.
- [ ] `extended_evidence_output_intention_audit.csv` reports the intended artefact contract.
- [ ] Numeric differences from the manuscript are interpreted as regenerated-run differences unless the exact manuscript-locked runtime is restored.
